# Clasificación Multi-etiqueta Jerárquica de Diagnósticos Médicos con CIE-10

Este notebook implementa una metodología completa para la clasificación automática de textos médicos utilizando códigos CIE-10 (Clasificación Internacional de Enfermedades, 10ª revisión) mediante un enfoque jerárquico y multi-etiqueta.

El flujo de trabajo incluye:
1. Configuración del entorno y parámetros
2. Carga y exploración de datos
3. Preprocesamiento de textos médicos
4. Vectorización y representación de textos
5. Codificación jerárquica de etiquetas CIE-10
6. Implementación y entrenamiento de modelos de clasificación
7. Evaluación y comparación de resultados

## 1. Configuración del Entorno y Parámetros

En esta sección definimos todas las rutas de archivos, parámetros de entrenamiento y configuraciones globales del notebook para facilitar su ajuste y ejecución tanto en CPU como en GPU.

In [ ]:
# Configuración de rutas y parámetros globales

import os
import torch
import numpy as np
import random
import pandas as pd
from datetime import datetime

# Rutas de datos
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'bert-classifier/codiesp_csvs')
TRAIN_DATA_PATH = os.path.join(DATA_DIR, 'codiesp_D_source_train.csv')
VAL_DATA_PATH = os.path.join(DATA_DIR, 'codiesp_D_source_validation.csv')
TEST_DATA_PATH = os.path.join(DATA_DIR, 'codiesp_D_source_test.csv')

# Rutas para guardar modelos y resultados
MODELS_DIR = os.path.join(os.getcwd(), 'models')
RESULTS_DIR = os.path.join(os.getcwd(), 'results')

# Crear directorios si no existen
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Parámetros generales para todo el notebook
RANDOM_SEED = 42
USE_EXPANSION = True  # Usar expansión de abreviaturas médicas
USE_NEGATION = True   # Usar detección de negaciones

# Configuración de dispositivo (CPU/GPU)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Fijar semillas para reproducibilidad
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(RANDOM_SEED)

print(f"Configuración completada:")
print(f"- Utilizando dispositivo: {DEVICE}")
print(f"- Datos de entrenamiento: {os.path.basename(TRAIN_DATA_PATH)}")
print(f"- Modelos guardados en: {MODELS_DIR}")

In [ ]:
# Importación de todas las librerías necesarias

# Librerías estándar
import os
import re
import json
import time
import pickle
from collections import Counter, defaultdict
from tqdm.notebook import tqdm

# Librerías para análisis de datos y visualización
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Librerías de procesamiento de texto y NLP
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize

# Descargar recursos de NLTK necesarios
nltk_resources = ['punkt']
for resource in nltk_resources:
    try:
        nltk.data.find(f'tokenizers/{resource}')
    except LookupError:
        nltk.download(resource)

# Librerías de machine learning
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, hamming_loss, f1_score,
                            precision_score, recall_score, classification_report,
                            multilabel_confusion_matrix)

# Librerías de deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# Transformers (cuando se necesiten)
try:
    from transformers import AutoTokenizer, AutoModel
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("Transformers no está instalado. Las funcionalidades de BERT no estarán disponibles.")

# Gensim para Word2Vec (cuando se necesite)
try:
    from gensim.models import Word2Vec, KeyedVectors
    GENSIM_AVAILABLE = True
except ImportError:
    GENSIM_AVAILABLE = False
    print("Gensim no está instalado. Las funcionalidades de Word2Vec no estarán disponibles.")

# Configurar visualización
plt.style.use('seaborn')
sns.set(style='whitegrid')

# Deshabilitar advertencias
import warnings
warnings.filterwarnings('ignore')

print("Todas las librerías importadas correctamente")

## 2. Carga y Exploración de Datos

En esta sección cargaremos los datos del conjunto CODIESP, que contiene textos médicos en español etiquetados con códigos CIE-10. Realizaremos un análisis exploratorio para entender la distribución de diagnósticos, longitud de textos y otros aspectos relevantes.

In [ ]:
# Función para cargar datos con manejo de errores
def load_dataset(data_path, fallback_paths=None):
    """
    Carga un conjunto de datos desde un archivo CSV con manejo de errores.

    Args:
        data_path: Ruta al archivo CSV principal
        fallback_paths: Lista de rutas alternativas si la principal falla

    Returns:
        DataFrame con los datos cargados
    """
    try:
        # Intentar cargar el archivo principal
        df = pd.read_csv(data_path, encoding='utf-8')
        print(f"✅ Datos cargados correctamente desde {data_path}")
        return df
    except FileNotFoundError:
        print(f"⚠️ No se encontró el archivo {data_path}")

        # Si hay rutas alternativas, intentar cargarlas
        if fallback_paths:
            for path in fallback_paths:
                try:
                    df = pd.read_csv(path, encoding='utf-8')
                    print(f"✅ Datos cargados desde ruta alternativa: {path}")
                    return df
                except FileNotFoundError:
                    print(f"⚠️ No se encontró el archivo alternativo {path}")

        # Si llegamos aquí, intentar buscar archivos CSV en la carpeta bert-classifier
        try:
            bert_classifier_dir = os.path.join(os.path.dirname(os.getcwd()), 'bert-classifier')
            codiesp_dir = os.path.join(bert_classifier_dir, 'codiesp_csvs')

            if os.path.exists(codiesp_dir):
                # Buscar archivos CSV en la carpeta
                csv_files = [f for f in os.listdir(codiesp_dir) if f.endswith('.csv')]

                if csv_files:
                    fallback_file = os.path.join(codiesp_dir, csv_files[0])
                    print(f"🔍 Intentando cargar archivo alternativo: {fallback_file}")
                    df = pd.read_csv(fallback_file, encoding='utf-8')
                    print(f"✅ Datos cargados desde archivo encontrado: {fallback_file}")
                    return df
        except Exception as e:
            print(f"Error al buscar archivos alternativos: {e}")

        # Si todo falla, crear un DataFrame de ejemplo
        print("⚠️ No se pudieron cargar datos reales. Creando conjunto de datos de ejemplo para demostración.")

        # Crear datos de ejemplo
        example_texts = [
            "Paciente con hipertensión arterial y diabetes mellitus tipo 2. Presenta dolor torácico.",
            "Varón de 65 años con antecedentes de EPOC. Ingresa por neumonía.",
            "Mujer de 45 años con fibrilación auricular paroxística. Tratamiento con anticoagulantes.",
            "Paciente con insuficiencia cardíaca congestiva descompensada. Edemas en MMII.",
            "Varón con IAM hace 2 años. Actualmente con angina de esfuerzo."
        ]

        example_codes = [
            "I10,E11",
            "J44.9,J18",
            "I48,Z79.01",
            "I50.9,R60.0",
            "I25.2,I20.8"
        ]

        return pd.DataFrame({'text': example_texts, 'codes': example_codes})

# Cargar datos de entrenamiento
print("Cargando datos de entrenamiento...")
fallback_train_paths = [
    os.path.join(DATA_DIR, 'codiesp_D_truncated_source_train.csv'),
    os.path.join(DATA_DIR, 'codiesp_extra_cie_source_train.csv'),
    os.path.join(os.path.dirname(os.getcwd()), 'bert-classifier/codiesp_csvs/codiesp_D_source_train.csv')
]

train_df = load_dataset(TRAIN_DATA_PATH, fallback_train_paths)

# Cargar datos de validación (opcional)
print("\nCargando datos de validación...")
fallback_val_paths = [
    os.path.join(DATA_DIR, 'codiesp_D_truncated_source_validation.csv'),
    os.path.join(os.path.dirname(os.getcwd()), 'bert-classifier/codiesp_csvs/codiesp_D_source_validation.csv')
]
try:
    val_df = load_dataset(VAL_DATA_PATH, fallback_val_paths)
except:
    print("No se pudieron cargar datos de validación. Se usará una parte del conjunto de entrenamiento.")
    val_df = None

# Cargar datos de prueba
print("\nCargando datos de prueba...")
fallback_test_paths = [
    os.path.join(DATA_DIR, 'codiesp_D_truncated_source_test.csv'),
    os.path.join(os.path.dirname(os.getcwd()), 'bert-classifier/codiesp_csvs/codiesp_D_source_test.csv')
]
test_df = load_dataset(TEST_DATA_PATH, fallback_test_paths)

# Verificar que las columnas necesarias están presentes
required_columns = ['text', 'codes']
for col in required_columns:
    if col not in train_df.columns:
        if col == 'codes' and 'labels' in train_df.columns:
            train_df['codes'] = train_df['labels']
        elif col == 'text' and 'content' in train_df.columns:
            train_df['text'] = train_df['content']
        else:
            print(f"⚠️ Columna '{col}' no encontrada. Por favor, verifica el formato de los datos.")

# Mostrar información básica de los datos
print("\nInformación básica del conjunto de entrenamiento:")
print(f"- Número de documentos: {len(train_df)}")
print(f"- Columnas disponibles: {train_df.columns.tolist()}")
print(f"- Primeras filas:")
display(train_df.head())

In [ ]:
# Análisis exploratorio de los datos

def analyze_dataset(df, title="Análisis del conjunto de datos"):
    """
    Realiza un análisis exploratorio del conjunto de datos.

    Args:
        df: DataFrame con los datos
        title: Título para los gráficos
    """
    print(f"\n{title}")
    print("-" * 80)

    # 1. Estadísticas básicas
    print("1. Estadísticas básicas:")
    print(f"- Número de documentos: {len(df)}")

    # Estadísticas de longitud de texto
    df['text_length'] = df['text'].apply(lambda x: len(str(x)))
    df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

    print(f"- Longitud promedio de texto (caracteres): {df['text_length'].mean():.1f}")
    print(f"- Longitud promedio de texto (palabras): {df['word_count'].mean():.1f}")
    print(f"- Texto más corto: {df['word_count'].min()} palabras")
    print(f"- Texto más largo: {df['word_count'].max()} palabras")

    # 2. Estadísticas de códigos CIE-10
    print("\n2. Estadísticas de códigos CIE-10:")

    # Convertir códigos de string a lista
    if df['codes'].dtype == object and isinstance(df['codes'].iloc[0], str):
        df['codes_list'] = df['codes'].apply(lambda x: str(x).split(',') if isinstance(x, str) else [])
    else:
        df['codes_list'] = df['codes']

    df['code_count'] = df['codes_list'].apply(len)

    print(f"- Promedio de códigos por documento: {df['code_count'].mean():.2f}")
    print(f"- Máximo de códigos por documento: {df['code_count'].max()}")
    print(f"- Documentos con múltiples códigos: {(df['code_count'] > 1).sum()} ({(df['code_count'] > 1).mean()*100:.1f}%)")

    # Contar códigos únicos
    all_codes = [code for codes in df['codes_list'] for code in codes]
    unique_codes = set(all_codes)
    print(f"- Número de códigos CIE-10 únicos: {len(unique_codes)}")

    # 3. Códigos más frecuentes
    print("\n3. Códigos más frecuentes:")
    code_counts = Counter(all_codes)
    for code, count in code_counts.most_common(10):
        print(f"- {code}: {count} ocurrencias ({count/len(df)*100:.1f}%)")

    # 4. Visualización de distribuciones
    plt.figure(figsize=(15, 10))

    # 4.1. Distribución de longitud de textos
    plt.subplot(2, 2, 1)
    sns.histplot(df['word_count'], bins=50, kde=True)
    plt.title('Distribución de longitud de textos (palabras)')
    plt.xlabel('Número de palabras')
    plt.ylabel('Frecuencia')

    # 4.2. Distribución de códigos por documento
    plt.subplot(2, 2, 2)
    sns.countplot(x='code_count', data=df)
    plt.title('Número de códigos CIE-10 por documento')
    plt.xlabel('Número de códigos')
    plt.ylabel('Número de documentos')

    # 4.3. Top 15 códigos más frecuentes
    plt.subplot(2, 1, 2)
    top_codes = pd.DataFrame(code_counts.most_common(15), columns=['Código', 'Frecuencia'])
    sns.barplot(x='Frecuencia', y='Código', data=top_codes)
    plt.title('15 códigos CIE-10 más frecuentes')
    plt.xlabel('Frecuencia')
    plt.ylabel('Código CIE-10')

    plt.tight_layout()
    plt.show()

    return df

# Realizar análisis exploratorio
train_df = analyze_dataset(train_df, title="Análisis del conjunto de entrenamiento")

# Si hay datos de prueba, analizarlos también
if test_df is not None:
    test_df = analyze_dataset(test_df, title="Análisis del conjunto de prueba")

## 3. Preprocesamiento de Textos Médicos

En esta sección implementaremos técnicas especializadas para el preprocesamiento de textos médicos en español, incluyendo:

1. **Limpieza básica**: Normalización, eliminación de caracteres especiales, etc.
2. **Expansión de abreviaturas médicas**: Identificación y expansión de abreviaturas comunes en español
3. **Detección de negaciones**: Identificación de términos negados en el contexto médico
4. **Pipeline completo**: Integración de todos los pasos en una única función reutilizable

In [ ]:
# Cargar diccionario de abreviaturas médicas en español
# En un caso real, este diccionario tendría miles de entradas
MEDICAL_ABBREVIATIONS = {
    "hta": "hipertensión arterial",
    "dm": "diabetes mellitus",
    "dm2": "diabetes mellitus tipo 2",
    "ecg": "electrocardiograma",
    "fa": "fibrilación auricular",
    "icc": "insuficiencia cardíaca congestiva",
    "tto": "tratamiento",
    "epoc": "enfermedad pulmonar obstructiva crónica",
    "iam": "infarto agudo de miocardio",
    "acv": "accidente cerebrovascular",
    "htp": "hipertensión pulmonar",
    "erc": "enfermedad renal crónica",
    "tas": "tensión arterial sistólica",
    "tad": "tensión arterial diastólica",
    "fc": "frecuencia cardíaca",
    "fr": "frecuencia respiratoria",
    "rx": "radiografía",
    "tac": "tomografía axial computarizada",
    "tc": "tomografía computarizada",
    "rm": "resonancia magnética",
    "uci": "unidad de cuidados intensivos",
    "ah": "antecedentes heredofamiliares",
    "ap": "antecedentes personales",
    "app": "antecedentes personales patológicos",
    "ekg": "electrocardiograma",
    "pa": "presión arterial",
    "kg": "kilogramos",
    "gr": "gramos",
    "mg": "miligramos",
    "mmhg": "milímetros de mercurio"
}

# Lista de términos de negación en español
NEGATION_TERMS = [
    "no", "sin", "ausencia", "ausente", "negativo", "niega", "descarta",
    "libre de", "descartar", "se descarta", "se niega", "nunca",
    "excluye", "descartar", "excluir"
]

def clean_text(text):
    """Realiza limpieza básica de un texto médico."""
    if not isinstance(text, str):
        return ""

    # Convertir a minúsculas
    text = text.lower()

    # Normalizar espacios
    text = re.sub(r'\s+', ' ', text)

    # Eliminar caracteres no deseados pero mantener puntuación importante
    text = re.sub(r'[^\w\s\.,;:\-\(\)\/\+%]', '', text)

    # Normalizar abreviaturas comunes con puntos
    text = re.sub(r'dr\.', 'doctor', text)
    text = re.sub(r'dra\.', 'doctora', text)

    return text.strip()

def expand_abbreviations(text, abbrev_dict=MEDICAL_ABBREVIATIONS):
    """Expande abreviaturas médicas comunes en el texto."""
    if not isinstance(text, str):
        return ""

    words = text.split()
    result = []

    for word in words:
        # Eliminar puntuación para la búsqueda
        clean_word = re.sub(r'[^\w]', '', word).lower()

        if clean_word in abbrev_dict:
            # Mantener la puntuación original
            expansion = abbrev_dict[clean_word]
            # Si la palabra original tenía mayúsculas, conservar el formato
            if word.isupper():
                expansion = expansion.upper()
            elif word[0].isupper():
                expansion = expansion.capitalize()

            # Añadir la expansión entre paréntesis
            result.append(f"{word} ({expansion})")
        else:
            result.append(word)

    return ' '.join(result)

def mark_negations(text, window_size=5, negation_terms=NEGATION_TERMS):
    """
    Marca términos negados en el texto utilizando un enfoque simplificado.

    Args:
        text: Texto a procesar
        window_size: Número de palabras a considerar tras un término de negación
        negation_terms: Lista de términos que indican negación

    Returns:
        Texto con las negaciones marcadas con [NEG]término[/NEG]
    """
    if not isinstance(text, str):
        return ""

    sentences = sent_tokenize(text)
    processed_sentences = []

    for sentence in sentences:
        words = word_tokenize(sentence)
        negation_active = False
        negation_counter = 0

        for i in range(len(words)):
            word = words[i].lower()

            # Verificar si es un término de negación
            if word in negation_terms:
                negation_active = True
                negation_counter = 0
                continue

            # Marcar la palabra si está bajo el efecto de una negación
            if negation_active:
                if re.match(r'^[\.,;:]$', word):  # Si es un signo de puntuación
                    negation_active = False
                else:
                    words[i] = f"[NEG]{words[i]}[/NEG]"
                    negation_counter += 1

                    # Desactivar negación después de window_size palabras
                    if negation_counter >= window_size:
                        negation_active = False

        processed_sentences.append(' '.join(words))

    return ' '.join(processed_sentences)

def preprocess_medical_text(text, expand_abbrev=USE_EXPANSION, mark_neg=USE_NEGATION):
    """Pipeline completo de preprocesamiento para textos médicos."""
    if not isinstance(text, str):
        return ""

    # Limpieza básica
    text = clean_text(text)

    # Expansión de abreviaturas
    if expand_abbrev:
        text = expand_abbreviations(text)

    # Marcado de negaciones
    if mark_neg:
        text = mark_negations(text)

    return text

# Demostrar el preprocesamiento con un ejemplo
if 'train_df' in locals() and len(train_df) > 0:
    example_text = train_df['text'].iloc[0]

    print("🔍 Ejemplo de preprocesamiento de texto médico\n")
    print("Texto original (primeros 300 caracteres):")
    print("-" * 80)
    print(example_text[:300] + "..." if len(example_text) > 300 else example_text)
    print("-" * 80)

    # Aplicar preprocesamiento paso a paso
    cleaned_text = clean_text(example_text)
    print("\n1️⃣ Texto después de limpieza básica:")
    print("-" * 80)
    print(cleaned_text[:300] + "..." if len(cleaned_text) > 300 else cleaned_text)
    print("-" * 80)

    expanded_text = expand_abbreviations(cleaned_text)
    print("\n2️⃣ Texto después de expansión de abreviaturas:")
    print("-" * 80)
    print(expanded_text[:300] + "..." if len(expanded_text) > 300 else expanded_text)
    print("-" * 80)

    negated_text = mark_negations(expanded_text)
    print("\n3️⃣ Texto después de marcar negaciones:")
    print("-" * 80)
    print(negated_text[:300] + "..." if len(negated_text) > 300 else negated_text)
    print("-" * 80)

    # Pipeline completo
    fully_processed = preprocess_medical_text(example_text)
    print("\n✅ Texto completamente procesado:")
    print("-" * 80)
    print(fully_processed[:300] + "..." if len(fully_processed) > 300 else fully_processed)
    print("-" * 80)

### 3.1 Tokenización y vectorización

En esta sección implementaremos técnicas de tokenización y vectorización para convertir el texto preprocesado en representaciones numéricas que puedan ser utilizadas por los modelos de aprendizaje automático.

Utilizaremos varios enfoques:
1. **Bag of Words (BoW)**: Representación simple basada en frecuencia de palabras
2. **TF-IDF**: Ponderación de términos basada en su importancia relativa en el corpus
3. **Embeddings de Word2Vec**: Representaciones distribuidas preentrenadas
4. **Tokenización con BERT**: Preparación de textos para modelos basados en transformers

Estos métodos nos permitirán comparar diferentes representaciones y su impacto en el rendimiento de los modelos.

In [ ]:
# Implementación de diferentes métodos de vectorización

def create_text_representations(texts, method=VECTORIZER_METHOD, vectorizer=None, **kwargs):
    """
    Crea representaciones vectoriales de textos usando diferentes métodos.

    Args:
        texts: Lista de textos a vectorizar
        method: Método de vectorización ('bow', 'tfidf', 'word2vec', 'bert')
        vectorizer: Vectorizador preentrenado (opcional)
        **kwargs: Argumentos específicos para cada método

    Returns:
        Representación vectorial de los textos y el vectorizador utilizado
    """
    if method == 'bow':
        # Bag of Words
        max_features = kwargs.get('max_features', MAX_FEATURES)
        ngram_range = kwargs.get('ngram_range', NGRAM_RANGE)

        if vectorizer is None:
            vectorizer = CountVectorizer(max_features=max_features,
                                        ngram_range=ngram_range,
                                        stop_words=None)  # No eliminamos stop words en textos médicos
            X = vectorizer.fit_transform(texts)
        else:
            X = vectorizer.transform(texts)

        return X, vectorizer

    elif method == 'tfidf':
        # TF-IDF
        max_features = kwargs.get('max_features', MAX_FEATURES)
        ngram_range = kwargs.get('ngram_range', NGRAM_RANGE)

        if vectorizer is None:
            vectorizer = TfidfVectorizer(max_features=max_features,
                                        ngram_range=ngram_range,
                                        stop_words=None,
                                        use_idf=True,
                                        norm='l2')
            X = vectorizer.fit_transform(texts)
        else:
            X = vectorizer.transform(texts)

        return X, vectorizer

    elif method == 'word2vec':
        # Word2Vec embeddings (promedio de vectores de palabras)
        if not GENSIM_AVAILABLE:
            raise ImportError("Gensim no está instalado. No se puede usar Word2Vec.")

        # Tokenizar textos
        tokenized_texts = [word_tokenize(text.lower()) for text in texts]

        # Parámetros para el modelo Word2Vec
        vector_size = kwargs.get('vector_size', EMBED_SIZE)
        window = kwargs.get('window', 5)
        min_count = kwargs.get('min_count', 1)

        if vectorizer is None:
            # Entrenar modelo (en una implementación real usaríamos uno preentrenado)
            w2v_model = Word2Vec(sentences=tokenized_texts,
                              vector_size=vector_size,
                              window=window,
                              min_count=min_count,
                              workers=4)
        else:
            w2v_model = vectorizer

        # Crear vectores promedio para cada documento
        doc_vectors = []
        for tokens in tokenized_texts:
            # Filtrar tokens que están en el vocabulario
            valid_tokens = [token for token in tokens if token in w2v_model.wv]
            if valid_tokens:
                # Calcular vector promedio
                doc_vector = np.mean([w2v_model.wv[token] for token in valid_tokens], axis=0)
            else:
                # Vector de ceros para documentos sin tokens válidos
                doc_vector = np.zeros(vector_size)

            doc_vectors.append(doc_vector)

        X = np.vstack(doc_vectors)
        return X, w2v_model

    elif method == 'bert':
        # Tokenización para BERT (no vectoriza completamente, solo prepara para el modelo)
        if not TRANSFORMERS_AVAILABLE:
            raise ImportError("Transformers no está instalado. No se puede usar BERT.")

        bert_model_name = kwargs.get('model_name', BERT_MODEL_NAME)
        max_length = kwargs.get('max_length', MAX_SEQ_LENGTH)

        if vectorizer is None:
            tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
        else:
            tokenizer = vectorizer

        # Tokenizar textos (sin padding - lo aplicaremos al crear los batches)
        encodings = tokenizer(
            texts,
            truncation=True,
            max_length=max_length,
            padding='max_length',
            return_tensors='pt'
        )

        return encodings, tokenizer

    else:
        raise ValueError(f"Método de vectorización '{method}' no soportado")

# Demostración de los diferentes métodos de vectorización
if 'train_df' in locals() and len(train_df) > 0:
    # Seleccionar una pequeña muestra de textos para la demostración
    sample_texts = train_df['text'].head(5).tolist()

    # Aplicar preprocesamiento a los textos de muestra
    processed_texts = [preprocess_medical_text(text) for text in sample_texts]

    print("🔢 Demostración de diferentes métodos de vectorización\n")

    # 1. Bag of Words
    try:
        bow_vectors, bow_vectorizer = create_text_representations(processed_texts, method='bow', max_features=1000)
        print("1️⃣ Bag of Words:")
        print(f"- Forma de la matriz: {bow_vectors.shape}")
        print(f"- Tipo de dato: {type(bow_vectors)}")
        print(f"- Densidad: {bow_vectors.nnz / (bow_vectors.shape[0] * bow_vectors.shape[1]):.4f}")
        print(f"- Top 10 términos: {[term for term, idx in sorted(bow_vectorizer.vocabulary_.items(), key=lambda x: x[1])[:10]]}")
        print()
    except Exception as e:
        print(f"Error con Bag of Words: {e}")

    # 2. TF-IDF
    try:
        tfidf_vectors, tfidf_vectorizer = create_text_representations(processed_texts, method='tfidf', max_features=1000)
        print("2️⃣ TF-IDF:")
        print(f"- Forma de la matriz: {tfidf_vectors.shape}")
        print(f"- Tipo de dato: {type(tfidf_vectors)}")
        print(f"- Densidad: {tfidf_vectors.nnz / (tfidf_vectors.shape[0] * tfidf_vectors.shape[1]):.4f}")
        print(f"- Top 10 términos: {[term for term, idx in sorted(tfidf_vectorizer.vocabulary_.items(), key=lambda x: x[1])[:10]]}")
        print()
    except Exception as e:
        print(f"Error con TF-IDF: {e}")

    # 3. Word2Vec
    if GENSIM_AVAILABLE:
        try:
            w2v_vectors, w2v_model = create_text_representations(processed_texts, method='word2vec', vector_size=50)
            print("3️⃣ Word2Vec (promedio de vectores de palabras):")
            print(f"- Forma de la matriz: {w2v_vectors.shape}")
            print(f"- Tipo de dato: {type(w2v_vectors)}")
            print(f"- Tamaño del vocabulario: {len(w2v_model.wv)}")
            print(f"- Dimensión de los vectores: {w2v_model.vector_size}")
            print()
        except Exception as e:
            print(f"Error al crear representaciones Word2Vec: {e}")
            print()
    else:
        print("3️⃣ Word2Vec no disponible (Gensim no está instalado)")
        print()

    # 4. BERT Tokenizer
    if TRANSFORMERS_AVAILABLE:
        try:
            bert_encodings, bert_tokenizer = create_text_representations(processed_texts, method='bert')
            print("4️⃣ Tokenización BERT:")
            print(f"- Tamaño del vocabulario del tokenizer: {bert_tokenizer.vocab_size}")
            print("- Ejemplo de tokenización para un texto:")
            sample_idx = 0
            print(f"  Texto original: {processed_texts[sample_idx][:50]}...")
            print(f"  Tokens: {bert_tokenizer.tokenize(processed_texts[sample_idx])[:15]}...")
            print(f"  IDs: {bert_encodings['input_ids'][sample_idx][:15].tolist()}...")
        except Exception as e:
            print(f"Error al crear tokenización BERT: {e}")
            print("Nota: La tokenización BERT requiere conexión a Internet para descargar el modelo.")
    else:
        print("4️⃣ BERT no disponible (Transformers no está instalado)")
        print()

## 4. Codificación Jerárquica de Etiquetas CIE-10

Los códigos CIE-10 tienen una estructura jerárquica natural. Por ejemplo:
- **C00-D49**: Neoplasias
  - **C00-C97**: Tumores malignos
    - **C00-C14**: Tumores malignos del labio, cavidad oral y faringe
      - **C00**: Tumor maligno del labio
        - **C00.0**: Tumor maligno del labio superior, cara externa

Esta estructura jerárquica es fundamental para nuestro enfoque de clasificación multinivel, donde primero predecimos categorías generales y luego refinamos hacia subcategorías específicas. 

En esta sección:
1. Extraeremos la estructura jerárquica de los códigos CIE-10
2. Crearemos múltiples niveles de etiquetas (padre, hijo, nieto)
3. Analizaremos la distribución de etiquetas en cada nivel jerárquico
4. Prepararemos las matrices de etiquetas para clasificación multinivel

In [ ]:
# Procesamiento de etiquetas jerárquicas CIE-10

def extract_hierarchical_levels(code):
    """
    Extrae los diferentes niveles jerárquicos de un código CIE-10.

    Args:
        code: Código CIE-10 (ej: "C50.2")

    Returns:
        Tupla con niveles jerárquicos (nivel1, nivel2, nivel3)
        - nivel1: Categoría general (ej: "C")
        - nivel2: Subcategoría (ej: "C50")
        - nivel3: Código completo (ej: "C50.2")
    """
    if not isinstance(code, str):
        return None, None, None

    # Nivel 1: Letra inicial (A-Z)
    match_level1 = re.match(r'^([A-Za-z])', code)
    level1 = match_level1.group(1).upper() if match_level1 else None

    # Nivel 2: Letra + números antes del punto (ej: C50)
    match_level2 = re.match(r'^([A-Za-z]\d{1,2})', code)
    level2 = match_level2.group(1).upper() if match_level2 else None

    # Nivel 3: Código completo
    level3 = code.upper()

    return level1, level2, level3

def create_hierarchical_labels(df, code_column='codes'):
    """
    Crea estructuras de etiquetas jerárquicas a partir de códigos CIE-10.

    Args:
        df: DataFrame con columna de códigos (en formato lista o string separado)
        code_column: Nombre de la columna con los códigos

    Returns:
        DataFrame con nuevas columnas de etiquetas por nivel
    """
    # Hacer una copia para no modificar el original
    result_df = df.copy()

    # Asegurarse de que los códigos están en formato de lista
    if result_df[code_column].dtype == object and isinstance(result_df[code_column].iloc[0], str):
        # Convertir strings separados por comas a listas
        result_df[code_column] = result_df[code_column].apply(lambda x: x.split(',') if isinstance(x, str) else [])

    # Extraer etiquetas para cada nivel jerárquico
    result_df['level1_codes'] = result_df[code_column].apply(
        lambda codes: list(set([extract_hierarchical_levels(code)[0] for code in codes if extract_hierarchical_levels(code)[0]]))
    )

    result_df['level2_codes'] = result_df[code_column].apply(
        lambda codes: list(set([extract_hierarchical_levels(code)[1] for code in codes if extract_hierarchical_levels(code)[1]]))
    )

    result_df['level3_codes'] = result_df[code_column].apply(
        lambda codes: list(set([extract_hierarchical_levels(code)[2] for code in codes if extract_hierarchical_levels(code)[2]]))
    )

    return result_df

def create_multilabel_matrices(df):
    """
    Crea matrices binarias multinivel para clasificación jerárquica.

    Args:
        df: DataFrame con columnas level1_codes, level2_codes y level3_codes

    Returns:
        Diccionario con matrices binarias y binarizadores para cada nivel
    """
    # Crear binarizadores para cada nivel
    mlb_level1 = MultiLabelBinarizer()
    mlb_level2 = MultiLabelBinarizer()
    mlb_level3 = MultiLabelBinarizer()

    # Transformar etiquetas a matrices binarias
    y_level1 = mlb_level1.fit_transform(df['level1_codes'])
    y_level2 = mlb_level2.fit_transform(df['level2_codes'])
    y_level3 = mlb_level3.fit_transform(df['level3_codes'])

    return {
        'level1': {'matrix': y_level1, 'binarizer': mlb_level1, 'classes': mlb_level1.classes_},
        'level2': {'matrix': y_level2, 'binarizer': mlb_level2, 'classes': mlb_level2.classes_},
        'level3': {'matrix': y_level3, 'binarizer': mlb_level3, 'classes': mlb_level3.classes_}
    }

def analyze_label_distribution(label_data):
    """
    Analiza la distribución de etiquetas en cada nivel jerárquico.

    Args:
        label_data: Diccionario con datos de etiquetas por nivel

    Returns:
        Estadísticas sobre la distribución de etiquetas
    """
    stats = {}

    for level, data in label_data.items():
        y = data['matrix']
        classes = data['classes']

        # Número de etiquetas distintas
        num_labels = len(classes)

        # Frecuencia de cada etiqueta
        label_counts = np.sum(y, axis=0)

        # Etiquetas más comunes
        top_indices = np.argsort(-label_counts)[:10]
        top_labels = [(classes[i], label_counts[i]) for i in top_indices]

        # Promedio de etiquetas por instancia
        avg_labels_per_instance = np.mean(np.sum(y, axis=1))

        # Máximo de etiquetas por instancia
        max_labels_per_instance = np.max(np.sum(y, axis=1))

        # Documentos con múltiples etiquetas (%)
        multi_label_docs = np.mean(np.sum(y, axis=1) > 1) * 100

        stats[level] = {
            'num_labels': num_labels,
            'label_counts': label_counts,
            'top_labels': top_labels,
            'avg_labels_per_instance': avg_labels_per_instance,
            'max_labels_per_instance': max_labels_per_instance,
            'multi_label_docs_percent': multi_label_docs
        }

    return stats

# Aplicar procesamiento de etiquetas jerárquicas si el conjunto de datos está disponible
if 'train_df' in locals() and 'codes' in train_df.columns:
    print("🏷️ Procesamiento de etiquetas jerárquicas CIE-10\n")

    # Crear estructura jerárquica de etiquetas
    train_df = create_hierarchical_labels(train_df, code_column='codes')

    # Mostrar algunos ejemplos
    if len(train_df) > 0:
        print("Ejemplo de etiquetas jerárquicas para un documento:")
        sample_idx = 0
        print(f"Texto: {train_df['text'].iloc[sample_idx][:100]}...")
        print(f"Códigos originales: {train_df['codes'].iloc[sample_idx]}")
        print(f"Nivel 1 (Categorías): {train_df['level1_codes'].iloc[sample_idx]}")
        print(f"Nivel 2 (Subcategorías): {train_df['level2_codes'].iloc[sample_idx]}")
        print(f"Nivel 3 (Códigos completos): {train_df['level3_codes'].iloc[sample_idx]}\n")

    # Crear matrices multinivel
    label_matrices = create_multilabel_matrices(train_df)

    # Mostrar dimensiones de las matrices
    print("Dimensiones de las matrices de etiquetas:")
    for level, data in label_matrices.items():
        print(f"- {level}: {data['matrix'].shape} ({len(data['classes'])} clases únicas)")

    # Analizar distribución de etiquetas
    stats = analyze_label_distribution(label_matrices)

    # Mostrar estadísticas
    print("\nEstadísticas de etiquetas por nivel:")
    for level, data in stats.items():
        print(f"\n{level.upper()}:")
        print(f"- Número de etiquetas distintas: {data['num_labels']}")
        print(f"- Promedio de etiquetas por documento: {data['avg_labels_per_instance']:.2f}")
        print(f"- Máximo de etiquetas por documento: {data['max_labels_per_instance']}")
        print(f"- Documentos con múltiples etiquetas: {data['multi_label_docs_percent']:.2f}%")
        print(f"- Top 5 etiquetas más frecuentes:")
        for label, count in data['top_labels'][:5]:
            print(f"  - {label}: {count} ocurrencias")

    # Visualizar distribución de las etiquetas más comunes
    plt.figure(figsize=(15, 12))

    for i, level in enumerate(['level1', 'level2', 'level3']):
        plt.subplot(3, 1, i+1)

        # Obtener las 10 etiquetas más comunes
        top_indices = np.argsort(-stats[level]['label_counts'])[:15]
        labels = [label_matrices[level]['classes'][i] for i in top_indices]
        counts = [stats[level]['label_counts'][i] for i in top_indices]

        # Crear gráfico de barras
        sns.barplot(x=counts, y=labels, palette='viridis')
        plt.title(f'Top 15 etiquetas más frecuentes - Nivel {i+1}')
        plt.xlabel('Número de documentos')
        plt.ylabel('Código CIE-10')
        plt.tight_layout()

    plt.subplots_adjust(hspace=0.4)
    plt.suptitle('Distribución de Etiquetas CIE-10 por Nivel Jerárquico', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

    # Si hay conjunto de prueba, procesarlo también
    if 'test_df' in locals() and 'codes' in test_df.columns:
        print("\nProcesando etiquetas jerárquicas para conjunto de prueba...")
        test_df = create_hierarchical_labels(test_df, code_column='codes')
else:
    print("⚠️ No se encontraron datos para procesar etiquetas jerárquicas CIE-10.")

## 5. Implementación de Modelos de Clasificación

En esta sección implementaremos diferentes modelos para la clasificación multi-etiqueta jerárquica de diagnósticos CIE-10. Utilizaremos un enfoque progresivo, comenzando con modelos más simples y avanzando hacia arquitecturas más sofisticadas:

1. **Modelos Baseline**:
   - Clasificadores tradicionales adaptados para multi-etiqueta
   - One-vs-Rest con SVM, Random Forest y Regresión Logística

2. **Modelos de Redes Neuronales**:
   - BiLSTM con atención
   - CNN para clasificación de texto

3. **Modelos Jerárquicos**:
   - Clasificadores en cascada (padres → hijos → nietos)
   - Arquitectura con pérdida jerárquica personalizada

4. **Modelos basados en Transformers**:
   - Fine-tuning de BERT para español médico
   - Arquitectura jerárquica con BERT

Para cada modelo evaluaremos su rendimiento con métricas específicas para clasificación multi-etiqueta y jerárquica.

### 5.1 Modelos Baseline para Clasificación Multi-etiqueta

Comenzaremos implementando modelos clásicos adaptados para la clasificación multi-etiqueta mediante la estrategia One-vs-Rest (OvR). Esta estrategia convierte un problema de clasificación multi-etiqueta en múltiples problemas de clasificación binaria (uno por cada etiqueta posible).

Los modelos baseline que implementaremos son:
- **Regresión Logística**: Un clasificador lineal simple y rápido
- **SVM**: Máquinas de vectores de soporte con kernel lineal, eficiente para textos
- **Random Forest**: Conjunto de árboles de decisión, adecuado para capturar relaciones no lineales

In [ ]:
# Implementación de modelos baseline usando One-vs-Rest para clasificación multi-etiqueta
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, hamming_loss, accuracy_score
import time

def entrenar_modelo_baseline(X_train, y_train, X_test, y_test, modelo_base, nombre_modelo):
    """
    Entrena un modelo baseline usando la estrategia One-vs-Rest para clasificación multi-etiqueta.

    Args:
        X_train: Características de entrenamiento (vectorizadas)
        y_train: Etiquetas de entrenamiento (codificadas)
        X_test: Características de prueba (vectorizadas)
        y_test: Etiquetas de prueba (codificadas)
        modelo_base: Modelo de scikit-learn a utilizar como base
        nombre_modelo: Nombre del modelo para los registros

    Returns:
        dict: Diccionario con métricas y el modelo entrenado
    """
    print(f"Entrenando modelo {nombre_modelo}...")
    inicio = time.time()

    # Aplicar estrategia One-vs-Rest para multi-etiqueta
    modelo = OneVsRestClassifier(modelo_base, n_jobs=N_JOBS)

    # Entrenar modelo
    modelo.fit(X_train, y_train)

    # Predecir en conjunto de prueba
    y_pred = modelo.predict(X_test)

    # Calcular métricas
    tiempo_total = time.time() - inicio
    hamming = hamming_loss(y_test, y_pred)
    accuracy = accuracy_score(y_test, y_pred)

    # Calcular precision, recall, f1-score para cada etiqueta
    reporte = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

    print(f"Tiempo de entrenamiento para {nombre_modelo}: {tiempo_total:.2f} segundos")
    print(f"Hamming Loss: {hamming:.4f}")
    print(f"Accuracy exacta: {accuracy:.4f}")
    print(f"F1-score macro: {reporte['macro avg']['f1-score']:.4f}")
    print(f"F1-score ponderado: {reporte['weighted avg']['f1-score']:.4f}")

    return {
        "modelo": modelo,
        "nombre": nombre_modelo,
        "hamming_loss": hamming,
        "accuracy": accuracy,
        "f1_macro": reporte['macro avg']['f1-score'],
        "f1_weighted": reporte['weighted avg']['f1-score'],
        "tiempo_entrenamiento": tiempo_total,
        "reporte": reporte
    }

In [ ]:
# Función para entrenar múltiples modelos baseline y comparar resultados
def entrenar_modelos_baseline(X_train, y_train, X_test, y_test, vectorizador_nombre=None):
    """
    Entrena varios modelos baseline para clasificación multi-etiqueta y compara resultados.

    Args:
        X_train: Características de entrenamiento (vectorizadas)
        y_train: Etiquetas de entrenamiento (codificadas)
        X_test: Características de prueba (vectorizadas)
        y_test: Etiquetas de prueba (codificadas)
        vectorizador_nombre: Nombre del vectorizador utilizado (para registros)

    Returns:
        dict: Diccionario con resultados de todos los modelos entrenados
    """
    resultados = {}

    print(f"\n{'='*80}")
    print(f"Entrenando modelos baseline usando vectorización: {vectorizador_nombre if vectorizador_nombre else 'No especificada'}")
    print(f"{'='*80}")

    # 1. Regresión Logística
    modelo_lr = LogisticRegression(
        C=1.0,
        solver='liblinear',  # 'liblinear' es eficiente para datasets pequeños a medianos
        max_iter=MAX_ITER,
        verbose=0,
        random_state=RANDOM_SEED
    )
    resultados['logistic_regression'] = entrenar_modelo_baseline(
        X_train, y_train, X_test, y_test,
        modelo_lr,
        "Regresión Logística"
    )

    # 2. SVM Lineal
    modelo_svm = LinearSVC(
        C=1.0,
        max_iter=MAX_ITER,
        verbose=0,
        random_state=RANDOM_SEED
    )
    resultados['svm'] = entrenar_modelo_baseline(
        X_train, y_train, X_test, y_test,
        modelo_svm,
        "SVM Lineal"
    )

    # 3. Random Forest
    modelo_rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS
    )
    resultados['random_forest'] = entrenar_modelo_baseline(
        X_train, y_train, X_test, y_test,
        modelo_rf,
        "Random Forest"
    )

    # Comparar resultados
    print(f"\n{'='*80}")
    print(f"Comparación de modelos baseline usando {vectorizador_nombre if vectorizador_nombre else 'vectorización no especificada'}")
    print(f"{'='*80}")

    # Crear tabla comparativa
    tabla = pd.DataFrame({
        'Modelo': [r['nombre'] for r in resultados.values()],
        'Hamming Loss': [r['hamming_loss'] for r in resultados.values()],
        'Accuracy Exacta': [r['accuracy'] for r in resultados.values()],
        'F1-Score Macro': [r['f1_macro'] for r in resultados.values()],
        'F1-Score Ponderado': [r['f1_weighted'] for r in resultados.values()],
        'Tiempo (s)': [r['tiempo_entrenamiento'] for r in resultados.values()]
    })

    display(tabla.sort_values('F1-Score Ponderado', ascending=False))

    return resultados

In [ ]:
# Visualización de resultados para modelos baseline
def visualizar_resultados_baseline(resultados, vectorizador_nombre=None):
    """
    Visualiza los resultados de los modelos baseline.

    Args:
        resultados: Diccionario con resultados de entrenar_modelos_baseline
        vectorizador_nombre: Nombre del vectorizador utilizado
    """
    plt.figure(figsize=(14, 10))

    # 1. Comparación de F1-Score
    plt.subplot(2, 2, 1)
    modelos = [r['nombre'] for r in resultados.values()]
    f1_macro = [r['f1_macro'] for r in resultados.values()]
    f1_weighted = [r['f1_weighted'] for r in resultados.values()]

    x = np.arange(len(modelos))
    width = 0.35

    plt.bar(x - width/2, f1_macro, width, label='F1-Macro')
    plt.bar(x + width/2, f1_weighted, width, label='F1-Weighted')
    plt.xlabel('Modelo')
    plt.ylabel('F1-Score')
    plt.title('Comparación de F1-Score por Modelo')
    plt.xticks(x, modelos, rotation=45)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)

    # 2. Hamming Loss
    plt.subplot(2, 2, 2)
    hamming = [r['hamming_loss'] for r in resultados.values()]

    plt.bar(modelos, hamming, color='salmon')
    plt.xlabel('Modelo')
    plt.ylabel('Hamming Loss')
    plt.title('Hamming Loss por Modelo (menor es mejor)')
    plt.xticks(rotation=45)
    plt.grid(True, linestyle='--', alpha=0.7)

    # 3. Accuracy
    plt.subplot(2, 2, 3)
    accuracy = [r['accuracy'] for r in resultados.values()]

    plt.bar(modelos, accuracy, color='lightgreen')
    plt.xlabel('Modelo')
    plt.ylabel('Accuracy Exacta')
    plt.title('Accuracy Exacta por Modelo')
    plt.xticks(rotation=45)
    plt.grid(True, linestyle='--', alpha=0.7)

    # 4. Tiempo de entrenamiento
    plt.subplot(2, 2, 4)
    tiempos = [r['tiempo_entrenamiento'] for r in resultados.values()]

    plt.bar(modelos, tiempos, color='skyblue')
    plt.xlabel('Modelo')
    plt.ylabel('Tiempo (segundos)')
    plt.title('Tiempo de Entrenamiento por Modelo')
    plt.xticks(rotation=45)
    plt.grid(True, linestyle='--', alpha=0.7)

    plt.suptitle(f'Resultados de Modelos Baseline con {vectorizador_nombre if vectorizador_nombre else "Vectorización No Especificada"}')
    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.show()

In [ ]:
# Ejecutar el entrenamiento de modelos baseline
# Definir parámetros específicos para modelos baseline
MAX_ITER = 1000
N_JOBS = -1  # Usar todos los núcleos disponibles
GUARDAR_MEJORES_MODELOS = True

if 'EJECUTAR_MODELOS_BASELINE' not in locals():
    EJECUTAR_MODELOS_BASELINE = True

print("Preparando datos para entrenamiento de modelos baseline...")

# Generar datos de entrenamiento y prueba usando vectorizadores definidos
# Para BOW (Bag of Words)
if 'USAR_VECTORIZACION_BOW' not in locals():
    USAR_VECTORIZACION_BOW = True

if USAR_VECTORIZACION_BOW:
    print("\nPreparando vectorización BOW...")
    X_train_bow = vectorizador_bow.transform(X_train_preprocesado)
    X_test_bow = vectorizador_bow.transform(X_test_preprocesado)
    print(f"Dimensiones de X_train_bow: {X_train_bow.shape}")
    print(f"Dimensiones de X_test_bow: {X_test_bow.shape}")

    # Entrenar modelos con BOW
    resultados_bow = entrenar_modelos_baseline(
        X_train_bow, y_train_encoded, X_test_bow, y_test_encoded,
        vectorizador_nombre="Bag of Words"
    )

    # Visualizar resultados
    visualizar_resultados_baseline(resultados_bow, "Bag of Words")

    # Guardar el mejor modelo si está configurado
    if GUARDAR_MEJORES_MODELOS:
        mejor_modelo_bow = max(resultados_bow.values(), key=lambda x: x['f1_weighted'])
        ruta_modelo = os.path.join(MODELS_DIR, f"mejor_modelo_bow_{mejor_modelo_bow['nombre']}.pkl")
        print(f"Guardando mejor modelo BOW ({mejor_modelo_bow['nombre']}) en {ruta_modelo}")
        joblib.dump(mejor_modelo_bow['modelo'], ruta_modelo)

# Para TF-IDF
if 'USAR_VECTORIZACION_TFIDF' not in locals():
    USAR_VECTORIZACION_TFIDF = True

if USAR_VECTORIZACION_TFIDF:
    print("\nPreparando vectorización TF-IDF...")
    X_train_tfidf = vectorizador_tfidf.transform(X_train_preprocesado)
    X_test_tfidf = vectorizador_tfidf.transform(X_test_preprocesado)
    print(f"Dimensiones de X_train_tfidf: {X_train_tfidf.shape}")
    print(f"Dimensiones de X_test_tfidf: {X_test_tfidf.shape}")

    # Entrenar modelos con TF-IDF
    resultados_tfidf = entrenar_modelos_baseline(
        X_train_tfidf, y_train_encoded, X_test_tfidf, y_test_encoded,
        vectorizador_nombre="TF-IDF"
    )

    # Visualizar resultados
    visualizar_resultados_baseline(resultados_tfidf, "TF-IDF")

    # Guardar el mejor modelo si está configurado
    if GUARDAR_MEJORES_MODELOS:
        mejor_modelo_tfidf = max(resultados_tfidf.values(), key=lambda x: x['f1_weighted'])
        ruta_modelo = os.path.join(MODELS_DIR, f"mejor_modelo_tfidf_{mejor_modelo_tfidf['nombre']}.pkl")
        print(f"Guardando mejor modelo TF-IDF ({mejor_modelo_tfidf['nombre']}) en {ruta_modelo}")
        joblib.dump(mejor_modelo_tfidf['modelo'], ruta_modelo)

# Para Word2Vec (si está disponible)
if 'USAR_VECTORIZACION_WORD2VEC' not in locals():
    USAR_VECTORIZACION_WORD2VEC = True

if 'WORD2VEC_DISPONIBLE' not in locals():
    WORD2VEC_DISPONIBLE = 'modelo_word2vec' in locals()

if USAR_VECTORIZACION_WORD2VEC and WORD2VEC_DISPONIBLE:
    print("\nPreparando vectorización Word2Vec...")
    X_train_w2v = np.vstack([obtener_vector_documento(doc, modelo_word2vec) for doc in X_train_preprocesado])
    X_test_w2v = np.vstack([obtener_vector_documento(doc, modelo_word2vec) for doc in X_test_preprocesado])
    print(f"Dimensiones de X_train_w2v: {X_train_w2v.shape}")
    print(f"Dimensiones de X_test_w2v: {X_test_w2v.shape}")

    # Entrenar modelos con Word2Vec
    resultados_w2v = entrenar_modelos_baseline(
        X_train_w2v, y_train_encoded, X_test_w2v, y_test_encoded,
        vectorizador_nombre="Word2Vec"
    )

    # Visualizar resultados
    visualizar_resultados_baseline(resultados_w2v, "Word2Vec")

    # Guardar el mejor modelo si está configurado
    if GUARDAR_MEJORES_MODELOS:
        mejor_modelo_w2v = max(resultados_w2v.values(), key=lambda x: x['f1_weighted'])
        ruta_modelo = os.path.join(MODELS_DIR, f"mejor_modelo_w2v_{mejor_modelo_w2v['nombre']}.pkl")
        print(f"Guardando mejor modelo Word2Vec ({mejor_modelo_w2v['nombre']}) en {ruta_modelo}")
        joblib.dump(mejor_modelo_w2v['modelo'], ruta_modelo)
else:
    if not EJECUTAR_MODELOS_BASELINE:
        print("Entrenamiento de modelos baseline desactivado en configuración")
    elif not WORD2VEC_DISPONIBLE and USAR_VECTORIZACION_WORD2VEC:
        print("Word2Vec no está disponible. Saltando esta vectorización.")

### 5.2 Redes Neuronales para Clasificación Multi-etiqueta

A continuación, implementaremos modelos basados en redes neuronales más complejos que puedan capturar mejor las relaciones en el texto médico. Específicamente, implementaremos:

1. **BiLSTM con Atención**: Una red neuronal recurrente bidireccional con mecanismo de atención, que permite capturar dependencias a largo plazo en el texto y dar mayor importancia a las palabras más relevantes para la clasificación.

2. **Modelo Multi-etiqueta con Capas Densas**: Un modelo de red neuronal más simple basado en capas densas, que tomará como entrada los vectores de documentos y producirá probabilidades para cada etiqueta.

Estos modelos se implementarán usando PyTorch y se ejecutarán en GPU si está disponible, de acuerdo con la configuración definida al inicio del notebook.

In [ ]:
# Implementación de modelos de redes neuronales con PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, hamming_loss, accuracy_score
import numpy as np
import time

# Verificar si PyTorch está disponible y configurado correctamente
if not PYTORCH_DISPONIBLE:
    print("PyTorch no está disponible. No se pueden implementar modelos de redes neuronales.")
else:
    print(f"PyTorch está disponible. Se utilizará el dispositivo: {DEVICE}")

    # Configuración de hiperparámetros comunes para redes neuronales
    BATCH_SIZE = 32
    LEARNING_RATE = 0.001
    EPOCHS = 10  # Valor bajo para pruebas, aumentar para entrenamiento completo

    # Clase para el modelo BiLSTM con atención
    class BiLSTMAttention(nn.Module):
        def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers=2, dropout=0.2, padding_idx=0):
            super(BiLSTMAttention, self).__init__()

            # Capa de embedding
            self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)

            # Capa BiLSTM
            self.lstm = nn.LSTM(
                embedding_dim,
                hidden_dim,
                num_layers=n_layers,
                bidirectional=True,
                batch_first=True,
                dropout=dropout if n_layers > 1 else 0
            )

            # Capa de atención
            self.attention = nn.Linear(hidden_dim * 2, 1)  # BiLSTM tiene 2*hidden_dim de salida

            # Capa de salida
            self.fc = nn.Linear(hidden_dim * 2, output_dim)

            # Dropout
            self.dropout = nn.Dropout(dropout)

        def forward(self, x):
            # x: (batch_size, seq_length)

            # Aplicar embedding
            embedded = self.dropout(self.embedding(x))  # (batch_size, seq_length, embedding_dim)

            # Pasar por BiLSTM
            lstm_output, (hidden, cell) = self.lstm(embedded)  # lstm_output: (batch_size, seq_length, hidden_dim*2)

            # Aplicar atención
            attention_weights = F.softmax(self.attention(lstm_output).squeeze(-1), dim=1)  # (batch_size, seq_length)
            context_vector = torch.bmm(attention_weights.unsqueeze(1), lstm_output).squeeze(1)  # (batch_size, hidden_dim*2)

            # Pasar por capa de salida
            output = self.fc(context_vector)  # (batch_size, output_dim)

            # No aplicamos sigmoid aquí ya que usaremos BCEWithLogitsLoss que lo incluye
            return output

    # Clase para un modelo multi-etiqueta con capas densas
    class DenseMultiLabelClassifier(nn.Module):
        def __init__(self, input_dim, hidden_dims, output_dim, dropout=0.2):
            super(DenseMultiLabelClassifier, self).__init__()

            # Construir capas ocultas
            layers = []
            prev_dim = input_dim

            for hidden_dim in hidden_dims:
                layers.append(nn.Linear(prev_dim, hidden_dim))
                layers.append(nn.ReLU())
                layers.append(nn.Dropout(dropout))
                prev_dim = hidden_dim

            # Capa de salida
            layers.append(nn.Linear(prev_dim, output_dim))

            # Unir todas las capas
            self.model = nn.Sequential(*layers)

        def forward(self, x):
            # x: (batch_size, input_dim)
            return self.model(x)  # (batch_size, output_dim)

In [ ]:
# Funciones auxiliares para entrenar modelos de redes neuronales
def train_epoch(model, data_loader, optimizer, criterion, device):
    """
    Entrena el modelo durante una época.

    Args:
        model: Modelo PyTorch
        data_loader: DataLoader para los datos de entrenamiento
        optimizer: Optimizador
        criterion: Función de pérdida
        device: Dispositivo (cpu o cuda)

    Returns:
        float: Pérdida promedio de la época
    """
    model.train()
    epoch_loss = 0

    for batch in data_loader:
        # Desempaquetar batch
        features, labels = batch
        features, labels = features.to(device), labels.to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(data_loader)

def evaluate(model, data_loader, criterion, device):
    """
    Evalúa el modelo en un conjunto de datos.

    Args:
        model: Modelo PyTorch
        data_loader: DataLoader para los datos de evaluación
        criterion: Función de pérdida
        device: Dispositivo (cpu o cuda)

    Returns:
        tuple: (pérdida, predicciones, etiquetas reales)
    """
    model.eval()
    epoch_loss = 0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            # Desempaquetar batch
            features, labels = batch
            features, labels = features.to(device), labels.to(device)

            # Forward pass
            outputs = model(features)
            loss = criterion(outputs, labels)

            # Convertir logits a probabilidades y luego a predicciones binarias
            predictions = torch.sigmoid(outputs) > 0.5

            # Guardar predicciones y etiquetas para métricas
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            epoch_loss += loss.item()

    return epoch_loss / len(data_loader), np.array(all_predictions), np.array(all_labels)

def train_neural_model(model, X_train, y_train, X_val, y_val, batch_size=32, learning_rate=0.001,
                      epochs=10, device='cpu', model_name="Neural Network"):
    """
    Entrena un modelo de red neuronal para clasificación multi-etiqueta.

    Args:
        model: Modelo PyTorch
        X_train: Características de entrenamiento
        y_train: Etiquetas de entrenamiento
        X_val: Características de validación
        y_val: Etiquetas de validación
        batch_size: Tamaño del batch
        learning_rate: Tasa de aprendizaje
        epochs: Número de épocas
        device: Dispositivo (cpu o cuda)
        model_name: Nombre del modelo para registros

    Returns:
        dict: Diccionario con métricas y el modelo entrenado
    """
    print(f"\n{'='*80}")
    print(f"Entrenando modelo {model_name}...")
    print(f"{'='*80}")

    start_time = time.time()

    # Convertir datos a tensores
    if isinstance(X_train, np.ndarray):
        X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
        X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    else:  # Si es una matriz dispersa de scipy
        X_train_tensor = torch.tensor(X_train.toarray(), dtype=torch.float32)
        X_val_tensor = torch.tensor(X_val.toarray(), dtype=torch.float32)

    y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

    # Crear dataloaders
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    # Definir optimizador y función de pérdida
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.BCEWithLogitsLoss()  # Incluye sigmoid internamente

    # Mover modelo al dispositivo adecuado
    model = model.to(device)

    # Inicializar variables para early stopping
    best_val_loss = float('inf')
    patience = 3
    patience_counter = 0
    best_model_state = None

    # Entrenamiento
    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        # Entrenar una época
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        train_losses.append(train_loss)

        # Evaluar en conjunto de validación
        val_loss, _, _ = evaluate(model, val_loader, criterion, device)
        val_losses.append(val_loss)

        print(f"Época {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        # Verificar early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping en época {epoch+1}")
                break

    # Cargar el mejor modelo
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    # Evaluar el modelo final
    _, y_pred, y_true = evaluate(model, val_loader, criterion, device)

    # Calcular métricas
    hamming = hamming_loss(y_true, y_pred)
    accuracy = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

    total_time = time.time() - start_time

    print(f"\nEntrenamiento completado en {total_time:.2f} segundos")
    print(f"Hamming Loss: {hamming:.4f}")
    print(f"Accuracy exacta: {accuracy:.4f}")
    print(f"F1-score macro: {report['macro avg']['f1-score']:.4f}")
    print(f"F1-score ponderado: {report['weighted avg']['f1-score']:.4f}")

    # Visualizar curva de aprendizaje
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Época')
    plt.ylabel('Pérdida')
    plt.title(f'Curva de Aprendizaje - {model_name}')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()

    return {
        "modelo": model,
        "nombre": model_name,
        "hamming_loss": hamming,
        "accuracy": accuracy,
        "f1_macro": report['macro avg']['f1-score'],
        "f1_weighted": report['weighted avg']['f1-score'],
        "tiempo_entrenamiento": total_time,
        "reporte": report,
        "train_losses": train_losses,
        "val_losses": val_losses
    }

In [ ]:
# Ejecutar el entrenamiento de modelos de redes neuronales
# Definir parámetros específicos para modelos neuronales
BATCH_SIZE = 32
LEARNING_RATE = 0.001
EPOCHS = 10  # Valor bajo para pruebas, aumentar para entrenamiento completo

if 'PYTORCH_DISPONIBLE' not in locals():
    PYTORCH_DISPONIBLE = torch.cuda.is_available()

if 'EJECUTAR_MODELOS_NEURONALES' not in locals():
    EJECUTAR_MODELOS_NEURONALES = True

if 'USAR_MODELOS_DENSOS' not in locals():
    USAR_MODELOS_DENSOS = True

if 'USAR_MODELOS_SECUENCIALES' not in locals():
    USAR_MODELOS_SECUENCIALES = True

if 'MAX_SEQUENCE_LENGTH' not in locals():
    MAX_SEQUENCE_LENGTH = 300

if 'MAX_VOCAB_SIZE' not in locals():
    MAX_VOCAB_SIZE = 30000

if PYTORCH_DISPONIBLE and EJECUTAR_MODELOS_NEURONALES:
    print("Preparando datos para entrenamiento de modelos neuronales...")

    # Ejecutar modelos densos con vectorización tradicional
    if USAR_MODELOS_DENSOS:
        print("\n\nEntrenando modelos de redes neuronales densas...")

        # Para TF-IDF (suele funcionar bien con modelos densos)
        if USAR_VECTORIZACION_TFIDF:
            print("\nPreparando vectorización TF-IDF para modelo denso...")
            X_train_tfidf = vectorizador_tfidf.transform(X_train_preprocesado)
            X_test_tfidf = vectorizador_tfidf.transform(X_test_preprocesado)
            print(f"Dimensiones de X_train_tfidf: {X_train_tfidf.shape}")
            print(f"Dimensiones de X_test_tfidf: {X_test_tfidf.shape}")

            # Crear y entrenar modelo denso
            input_dim = X_train_tfidf.shape[1]
            output_dim = y_train_encoded.shape[1]
            hidden_dims = [512, 256]  # Capas ocultas

            modelo_denso = DenseMultiLabelClassifier(
                input_dim=input_dim,
                hidden_dims=hidden_dims,
                output_dim=output_dim,
                dropout=0.3
            )

            # Entrenar el modelo
            resultado_denso_tfidf = train_neural_model(
                model=modelo_denso,
                X_train=X_train_tfidf,
                y_train=y_train_encoded,
                X_val=X_test_tfidf,
                y_val=y_test_encoded,
                batch_size=BATCH_SIZE,
                learning_rate=LEARNING_RATE,
                epochs=EPOCHS,
                device=DEVICE,
                model_name="Dense Neural Network (TF-IDF)"
            )

            # Guardar el modelo si está configurado
            if GUARDAR_MEJORES_MODELOS:
                ruta_modelo = os.path.join(MODELS_DIR, "mejor_modelo_denso_tfidf.pt")
                print(f"Guardando modelo denso con TF-IDF en {ruta_modelo}")
                torch.save(resultado_denso_tfidf['modelo'].state_dict(), ruta_modelo)

    # Ejecutar modelos BiLSTM si se ha preparado tokenización
    if USAR_MODELOS_SECUENCIALES and 'tokenizador' in locals():
        print("\n\nEntrenando modelo BiLSTM con Atención...")

        # Preparar datos secuenciales (suponiendo que tenemos textos tokenizados y con padding)
        # En este punto idealmente deberíamos tener X_train_sequences y X_test_sequences
        # Si no los tenemos, podríamos crearlos aquí

        # Para este ejemplo, creamos secuencias tokenizadas a partir de los textos preprocesados
        # Esto es una simplificación - idealmente se debería tener un tokenizador más sofisticado
        try:
            from keras.preprocessing.text import Tokenizer
            from keras.preprocessing.sequence import pad_sequences

            tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token='<UNK>')
            tokenizer.fit_on_texts(X_train_preprocesado)

            X_train_sequences = tokenizer.texts_to_sequences(X_train_preprocesado)
            X_test_sequences = tokenizer.texts_to_sequences(X_test_preprocesado)

            # Aplicar padding para tener secuencias de igual longitud
            X_train_padded = pad_sequences(X_train_sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post')
            X_test_padded = pad_sequences(X_test_sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post')

            print(f"Dimensiones de X_train_padded: {X_train_padded.shape}")
            print(f"Dimensiones de X_test_padded: {X_test_padded.shape}")

            # Parámetros específicos del BiLSTM
            embedding_dim = 300
            hidden_dim = 128
            output_dim = y_train_encoded.shape[1]

            # Crear y entrenar modelo BiLSTM con atención
            vocab_size = min(len(tokenizer.word_index) + 1, MAX_VOCAB_SIZE)

            modelo_bilstm = BiLSTMAttention(
                vocab_size=vocab_size,
                embedding_dim=embedding_dim,
                hidden_dim=hidden_dim,
                output_dim=output_dim,
                n_layers=2,
                dropout=0.3
            )

            # Convertir secuencias a tensores
            X_train_padded_tensor = torch.tensor(X_train_padded, dtype=torch.long)
            X_test_padded_tensor = torch.tensor(X_test_padded, dtype=torch.long)

            # Crear datasets y dataloaders
            train_dataset = TensorDataset(X_train_padded_tensor, torch.tensor(y_train_encoded, dtype=torch.float32))
            test_dataset = TensorDataset(X_test_padded_tensor, torch.tensor(y_test_encoded, dtype=torch.float32))

            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

            # Definir optimizador y función de pérdida
            optimizer = torch.optim.Adam(modelo_bilstm.parameters(), lr=LEARNING_RATE)
            criterion = nn.BCEWithLogitsLoss()

            # Mover modelo al dispositivo adecuado
            modelo_bilstm = modelo_bilstm.to(DEVICE)

            # Entrenar el modelo
            print("\nEntrenando modelo BiLSTM con Atención...")
            start_time = time.time()

            # Parámetros para early stopping
            patience = 3

            train_losses = []
            val_losses = []
            best_val_loss = float('inf')
            patience_counter = 0
            best_model_state = None

            for epoch in range(EPOCHS):
                # Entrenar una época
                modelo_bilstm.train()
                epoch_loss = 0

                for batch in train_loader:
                    features, labels = batch
                    features, labels = features.to(DEVICE), labels.to(DEVICE)

                    optimizer.zero_grad()
                    outputs = modelo_bilstm(features)
                    loss = criterion(outputs, labels)

                    loss.backward()
                    optimizer.step()

                    epoch_loss += loss.item()

                train_loss = epoch_loss / len(train_loader)
                train_losses.append(train_loss)

                # Evaluar en conjunto de validación
                modelo_bilstm.eval()
                val_loss = 0

                with torch.no_grad():
                    for batch in test_loader:
                        features, labels = batch
                        features, labels = features.to(DEVICE), labels.to(DEVICE)

                        outputs = modelo_bilstm(features)
                        loss = criterion(outputs, labels)

                        val_loss += loss.item()

                val_loss = val_loss / len(test_loader)
                val_losses.append(val_loss)

                print(f"Época {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

                # Verificar early stopping
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0
                    best_model_state = modelo_bilstm.state_dict().copy()
                else:
                    patience_counter += 1
                    if patience_counter >= patience:
                        print(f"Early stopping en época {epoch+1}")
                        break

            # Cargar el mejor modelo
            if best_model_state is not None:
                modelo_bilstm.load_state_dict(best_model_state)

            # Evaluar el modelo final
            modelo_bilstm.eval()
            all_predictions = []
            all_labels = []

            with torch.no_grad():
                for batch in test_loader:
                    features, labels = batch
                    features, labels = features.to(DEVICE), labels.to(DEVICE)

                    outputs = modelo_bilstm(features)
                    predictions = torch.sigmoid(outputs) > 0.5

                    all_predictions.extend(predictions.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())

            y_pred = np.array(all_predictions)
            y_true = np.array(all_labels)

            # Calcular métricas
            hamming = hamming_loss(y_true, y_pred)
            accuracy = accuracy_score(y_true, y_pred)
            report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

            total_time = time.time() - start_time

            print(f"\nEntrenamiento completado en {total_time:.2f} segundos")
            print(f"Hamming Loss: {hamming:.4f}")
            print(f"Accuracy exacta: {accuracy:.4f}")
            print(f"F1-score macro: {report['macro avg']['f1-score']:.4f}")
            print(f"F1-score ponderado: {report['weighted avg']['f1-score']:.4f}")

            # Visualizar curva de aprendizaje
            plt.figure(figsize=(10, 5))
            plt.plot(train_losses, label='Train Loss')
            plt.plot(val_losses, label='Validation Loss')
            plt.xlabel('Época')
            plt.ylabel('Pérdida')
            plt.title('Curva de Aprendizaje - BiLSTM con Atención')
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.7)
            plt.show()

            # Guardar el modelo si está configurado
            if GUARDAR_MEJORES_MODELOS:
                ruta_modelo = os.path.join(MODELS_DIR, "mejor_modelo_bilstm.pt")
                print(f"Guardando modelo BiLSTM con Atención en {ruta_modelo}")
                torch.save(modelo_bilstm.state_dict(), ruta_modelo)

                # También guardar el tokenizador para poder usar el modelo posteriormente
                ruta_tokenizer = os.path.join(MODELS_DIR, "tokenizer_bilstm.pkl")
                joblib.dump(tokenizer, ruta_tokenizer)
        except ImportError:
            print("Keras no está disponible para la tokenización. No se puede entrenar el modelo BiLSTM.")
else:
    if not PYTORCH_DISPONIBLE:
        print("PyTorch no está disponible. No se pueden implementar modelos de redes neuronales.")
    else:
        print("Entrenamiento de modelos neuronales desactivado en configuración")

### 5.3 Clasificación Jerárquica para CIE-10

El sistema de códigos CIE-10 tiene una estructura jerárquica natural, con categorías, subcategorías y códigos específicos. En esta sección implementaremos un enfoque de clasificación jerárquica que aprovecha esta estructura para mejorar la precisión de la clasificación.

La estrategia que seguiremos es:

1. **Clasificación en niveles**: Primero clasificaremos el texto en las categorías principales (nivel 1), luego en las subcategorías (nivel 2) correspondientes a la categoría predicha, y finalmente en los códigos específicos (nivel 3).

2. **Especialización de modelos**: Entrenaremos modelos específicos para cada nivel de la jerarquía, permitiendo que cada uno se especialice en su nivel de abstracción.

3. **Propagación de predicciones**: Las predicciones de un nivel superior condicionarán las opciones disponibles en los niveles inferiores, respetando la estructura jerárquica.

In [ ]:
# Implementación del modelo de clasificación jerárquica
class ClasificadorJerarquico:
    """
    Implementación de un clasificador jerárquico para CIE-10 que aprovecha la estructura
    de niveles de los códigos (categoría, subcategoría, código específico).
    """
    def __init__(self, vectorizador, modelo_nivel1=None, modelos_nivel2=None, modelos_nivel3=None):
        """
        Inicializa el clasificador jerárquico.

        Args:
            vectorizador: Vectorizador para transformar texto en características
            modelo_nivel1: Modelo para clasificar en el primer nivel (categorías)
            modelos_nivel2: Diccionario de modelos para el segundo nivel, uno por cada categoría del nivel 1
            modelos_nivel3: Diccionario de modelos para el tercer nivel, uno por cada subcategoría del nivel 2
        """
        self.vectorizador = vectorizador
        self.modelo_nivel1 = modelo_nivel1
        self.modelos_nivel2 = modelos_nivel2 if modelos_nivel2 else {}
        self.modelos_nivel3 = modelos_nivel3 if modelos_nivel3 else {}

        # Mapas para traducir entre índices y códigos
        self.mapa_nivel1 = {}  # {indice: categoria}
        self.mapa_nivel2 = {}  # {categoria: {indice: subcategoria}}
        self.mapa_nivel3 = {}  # {subcategoria: {indice: codigo}}

        # Mapas inversos
        self.mapa_nivel1_inv = {}  # {categoria: indice}
        self.mapa_nivel2_inv = {}  # {categoria: {subcategoria: indice}}
        self.mapa_nivel3_inv = {}  # {subcategoria: {codigo: indice}}

        # Estructura jerárquica
        self.jerarquia = {}  # {categoria: [subcategorias], subcategoria: [codigos]}

    def ajustar_mapas_desde_etiquetas(self, etiquetas_nivel1, etiquetas_nivel2, etiquetas_nivel3):
        """
        Configura los mapas de índices a códigos a partir de las etiquetas codificadas.

        Args:
            etiquetas_nivel1: MultiLabelBinarizer para nivel 1
            etiquetas_nivel2: MultiLabelBinarizer para nivel 2
            etiquetas_nivel3: MultiLabelBinarizer para nivel 3
        """
        # Configurar mapas nivel 1
        for i, cat in enumerate(etiquetas_nivel1.classes_):
            self.mapa_nivel1[i] = cat
            self.mapa_nivel1_inv[cat] = i
            self.jerarquia[cat] = []

        # Configurar mapas nivel 2
        for cat in self.mapa_nivel1.values():
            self.mapa_nivel2[cat] = {}
            self.mapa_nivel2_inv[cat] = {}

            # Filtrar subcategorías que pertenecen a esta categoría
            subcats_cat = [sc for sc in etiquetas_nivel2.classes_ if sc.startswith(cat)]
            for i, subcat in enumerate(subcats_cat):
                self.mapa_nivel2[cat][i] = subcat
                self.mapa_nivel2_inv[cat][subcat] = i
                self.jerarquia[cat].append(subcat)
                self.jerarquia[subcat] = []

        # Configurar mapas nivel 3
        for cat in self.mapa_nivel1.values():
            for subcat in self.jerarquia[cat]:
                self.mapa_nivel3[subcat] = {}
                self.mapa_nivel3_inv[subcat] = {}

                # Filtrar códigos que pertenecen a esta subcategoría
                codigos_subcat = [cod for cod in etiquetas_nivel3.classes_ if cod.startswith(subcat)]
                for i, codigo in enumerate(codigos_subcat):
                    self.mapa_nivel3[subcat][i] = codigo
                    self.mapa_nivel3_inv[subcat][codigo] = i
                    self.jerarquia[subcat].append(codigo)

    def entrenar_nivel1(self, X, y, modelo=None):
        """
        Entrena el modelo de nivel 1 (categorías principales).

        Args:
            X: Características de entrenamiento
            y: Etiquetas de nivel 1 (codificadas con MultiLabelBinarizer)
            modelo: Modelo de scikit-learn (si no se especifica, usa SVM)
        """
        print("Entrenando modelo de nivel 1 (categorías principales)...")

        if modelo is None:
            modelo = OneVsRestClassifier(LinearSVC(C=1.0, random_state=RANDOM_SEED), n_jobs=N_JOBS)

        modelo.fit(X, y)
        self.modelo_nivel1 = modelo

        print("Modelo de nivel 1 entrenado con éxito.")
        return self

    def entrenar_nivel2(self, X, y_completo, y_nivel1, y_nivel2):
        """
        Entrena los modelos de nivel 2 (subcategorías) para cada categoría de nivel 1.

        Args:
            X: Características de entrenamiento
            y_completo: Etiquetas completas (para filtrar muestras)
            y_nivel1: Etiquetas de nivel 1 (codificadas)
            y_nivel2: Etiquetas de nivel 2 (codificadas)
        """
        print("Entrenando modelos de nivel 2 (subcategorías)...")

        # Para cada categoría del nivel 1, entrenar un modelo específico
        for idx_cat, categoria in self.mapa_nivel1.items():
            print(f"  Entrenando modelo para categoría: {categoria}")

            # Filtrar muestras que pertenecen a esta categoría
            mask_categoria = y_nivel1[:, idx_cat] == 1

            if np.sum(mask_categoria) < MIN_SAMPLES_PER_CLASS:
                print(f"    Saltando categoría {categoria} por tener menos de {MIN_SAMPLES_PER_CLASS} muestras")
                continue

            X_cat = X[mask_categoria]

            # Extraer solo las subcategorías relevantes para esta categoría
            subcats_relevantes = self.jerarquia[categoria]
            indices_subcats = [self.mapa_nivel2_inv[categoria][sc] for sc in subcats_relevantes]

            # Filtrar y reordenar las etiquetas de nivel 2
            y_subcats = np.zeros((X_cat.shape[0], len(indices_subcats)))

            for i, idx_subcat in enumerate(indices_subcats):
                subcat = self.mapa_nivel2[categoria][idx_subcat]
                idx_original = list(self.mapa_nivel2_inv[categoria].values()).index(idx_subcat)

                # Encontrar el índice de esta subcategoría en las etiquetas originales de nivel 2
                for j, sc in enumerate(y_nivel2.classes_):
                    if sc == subcat:
                        y_subcats[:, i] = y_nivel2[mask_categoria][:, j]
                        break

            # Si no hay subcategorías o todas las muestras son 0, saltar
            if len(subcats_relevantes) == 0 or np.sum(y_subcats) == 0:
                print(f"    Saltando categoría {categoria} por no tener subcategorías o muestras")
                continue

            # Entrenar modelo para esta categoría
            modelo_cat = OneVsRestClassifier(LinearSVC(C=1.0, random_state=RANDOM_SEED), n_jobs=N_JOBS)
            modelo_cat.fit(X_cat, y_subcats)

            # Guardar modelo
            self.modelos_nivel2[categoria] = {
                "modelo": modelo_cat,
                "indices_subcats": indices_subcats
            }

            print(f"    Modelo para categoría {categoria} entrenado con {X_cat.shape[0]} muestras y {len(indices_subcats)} subcategorías")

        print(f"Modelos de nivel 2 entrenados: {len(self.modelos_nivel2)} de {len(self.mapa_nivel1)}")
        return self

    def entrenar_nivel3(self, X, y_completo, y_nivel2, y_nivel3):
        """
        Entrena los modelos de nivel 3 (códigos específicos) para cada subcategoría de nivel 2.

        Args:
            X: Características de entrenamiento
            y_completo: Etiquetas completas (para filtrar muestras)
            y_nivel2: Etiquetas de nivel 2 (codificadas)
            y_nivel3: Etiquetas de nivel 3 (codificadas)
        """
        print("Entrenando modelos de nivel 3 (códigos específicos)...")

        # Para cada subcategoría en nivel 2, entrenar un modelo específico
        for categoria in self.jerarquia.keys():
            # Solo procesar si es una subcategoría (nivel 2)
            if not any(categoria.startswith(cat) for cat in self.mapa_nivel1.values() if cat != categoria):
                continue

            # Encontrar la categoría padre
            categoria_padre = categoria[:3]  # Suponiendo formato CIE-10 donde primeros 3 caracteres son categoría

            print(f"  Entrenando modelo para subcategoría: {categoria} (padre: {categoria_padre})")

            # Filtrar muestras que pertenecen a esta subcategoría
            idx_subcat = None
            for j, sc in enumerate(y_nivel2.classes_):
                if sc == categoria:
                    idx_subcat = j
                    break

            if idx_subcat is None:
                print(f"    No se encontró índice para subcategoría {categoria}")
                continue

            mask_subcat = y_nivel2[:, idx_subcat] == 1

            if np.sum(mask_subcat) < MIN_SAMPLES_PER_CLASS:
                print(f"    Saltando subcategoría {categoria} por tener menos de {MIN_SAMPLES_PER_CLASS} muestras")
                continue

            X_subcat = X[mask_subcat]

            # Extraer solo los códigos relevantes para esta subcategoría
            codigos_relevantes = self.jerarquia[categoria]

            if not codigos_relevantes:
                print(f"    Saltando subcategoría {categoria} por no tener códigos")
                continue

            # Filtrar y reordenar las etiquetas de nivel 3
            y_codigos = np.zeros((X_subcat.shape[0], len(codigos_relevantes)))

            for i, codigo in enumerate(codigos_relevantes):
                # Encontrar el índice de este código en las etiquetas originales de nivel 3
                for j, cod in enumerate(y_nivel3.classes_):
                    if cod == codigo:
                        y_codigos[:, i] = y_nivel3[mask_subcat][:, j]
                        break

            # Si no hay códigos o todas las muestras son 0, saltar
            if np.sum(y_codigos) == 0:
                print(f"    Saltando subcategoría {categoria} porque todas las muestras son 0")
                continue

            # Entrenar modelo para esta subcategoría
            modelo_subcat = OneVsRestClassifier(LinearSVC(C=1.0, random_state=RANDOM_SEED), n_jobs=N_JOBS)
            modelo_subcat.fit(X_subcat, y_codigos)

            # Guardar modelo
            self.modelos_nivel3[categoria] = {
                "modelo": modelo_subcat,
                "codigos_relevantes": codigos_relevantes
            }

            print(f"    Modelo para subcategoría {categoria} entrenado con {X_subcat.shape[0]} muestras y {len(codigos_relevantes)} códigos")

        total_subcats = sum(1 for cat in self.jerarquia.keys() if any(cat.startswith(c) for c in self.mapa_nivel1.values() if c != cat))
        print(f"Modelos de nivel 3 entrenados: {len(self.modelos_nivel3)} de {total_subcats}")
        return self

    def entrenar(self, X, y_nivel1, y_nivel2, y_nivel3, y_completo=None):
        """
        Entrena el clasificador jerárquico completo.

        Args:
            X: Características de entrenamiento
            y_nivel1: Etiquetas de nivel 1 (categorías)
            y_nivel2: Etiquetas de nivel 2 (subcategorías)
            y_nivel3: Etiquetas de nivel 3 (códigos específicos)
            y_completo: Etiquetas completas (opcional)
        """
        if y_completo is None:
            y_completo = y_nivel3  # Asumimos que el nivel 3 es el más específico

        # Configurar mapas
        self.ajustar_mapas_desde_etiquetas(y_nivel1, y_nivel2, y_nivel3)

        # Entrenar modelos de cada nivel
        self.entrenar_nivel1(X, y_nivel1)
        self.entrenar_nivel2(X, y_completo, y_nivel1, y_nivel2)
        self.entrenar_nivel3(X, y_completo, y_nivel2, y_nivel3)

        return self

    def predecir(self, X):
        """
        Realiza predicciones jerárquicas.

        Args:
            X: Características para predecir

        Returns:
            dict: Diccionario con predicciones en cada nivel y predicciones finales
        """
        if self.modelo_nivel1 is None:
            raise ValueError("El modelo no ha sido entrenado aún")

        # Nivel 1: Predecir categorías
        pred_nivel1 = self.modelo_nivel1.predict(X)

        # Inicializar matrices para niveles 2 y 3
        n_muestras = X.shape[0]
        pred_nivel2 = np.zeros((n_muestras, len(self.mapa_nivel1)), dtype=object)
        pred_nivel3 = np.zeros((n_muestras, len(self.mapa_nivel1)), dtype=object)

        # Para cada muestra y categoría predicha, predecir subcategorías y códigos
        for i in range(n_muestras):
            for j in range(len(self.mapa_nivel1)):
                if pred_nivel1[i, j] == 1:
                    categoria = self.mapa_nivel1[j]

                    # Si tenemos modelo para esta categoría, predecir subcategorías
                    if categoria in self.modelos_nivel2:
                        modelo_cat = self.modelos_nivel2[categoria]["modelo"]
                        indices_subcats = self.modelos_nivel2[categoria]["indices_subcats"]

                        # Predecir subcategorías
                        pred_subcats = modelo_cat.predict([X[i]])[0]
                        subcats_predichas = []

                        for k, pred in enumerate(pred_subcats):
                            if pred == 1:
                                idx_subcat = indices_subcats[k]
                                subcat = self.mapa_nivel2[categoria][idx_subcat]
                                subcats_predichas.append(subcat)

                                # Si tenemos modelo para esta subcategoría, predecir códigos
                                if subcat in self.modelos_nivel3:
                                    modelo_subcat = self.modelos_nivel3[subcat]["modelo"]
                                    codigos_relevantes = self.modelos_nivel3[subcat]["codigos_relevantes"]

                                    # Predecir códigos
                                    pred_codigos = modelo_subcat.predict([X[i]])[0]
                                    codigos_predichos = []

                                    for l, pred_cod in enumerate(pred_codigos):
                                        if pred_cod == 1:
                                            codigo = codigos_relevantes[l]
                                            codigos_predichos.append(codigo)

                                    # Almacenar códigos predichos
                                    if not pred_nivel3[i, j]:
                                        pred_nivel3[i, j] = []
                                    pred_nivel3[i, j].extend(codigos_predichos)

                        # Almacenar subcategorías predichas
                        pred_nivel2[i, j] = subcats_predichas

        # Preparar resultados finales
        resultados = {
            "nivel1": pred_nivel1,
            "nivel2": pred_nivel2,
            "nivel3": pred_nivel3,
            "categorias": [],
            "subcategorias": [],
            "codigos": []
        }

        # Extraer las etiquetas predichas
        for i in range(n_muestras):
            categorias = []
            subcategorias = []
            codigos = []

            for j in range(len(self.mapa_nivel1)):
                if pred_nivel1[i, j] == 1:
                    cat = self.mapa_nivel1[j]
                    categorias.append(cat)

                    # Añadir subcategorías si existen
                    if pred_nivel2[i, j]:
                        if isinstance(pred_nivel2[i, j], list):
                            subcategorias.extend(pred_nivel2[i, j])
                        else:
                            subcategorias.append(pred_nivel2[i, j])

                    # Añadir códigos si existen
                    if pred_nivel3[i, j]:
                        if isinstance(pred_nivel3[i, j], list):
                            codigos.extend(pred_nivel3[i, j])
                        else:
                            codigos.append(pred_nivel3[i, j])

            resultados["categorias"].append(categorias)
            resultados["subcategorias"].append(subcategorias)
            resultados["codigos"].append(codigos)

        return resultados

    def guardar(self, ruta_base):
        """
        Guarda el modelo completo en el sistema de archivos.

        Args:
            ruta_base: Ruta base donde guardar los archivos
        """
        os.makedirs(ruta_base, exist_ok=True)

        # Guardar vectorizador
        joblib.dump(self.vectorizador, os.path.join(ruta_base, "vectorizador.pkl"))

        # Guardar modelo nivel 1
        joblib.dump(self.modelo_nivel1, os.path.join(ruta_base, "modelo_nivel1.pkl"))

        # Guardar modelos nivel 2
        os.makedirs(os.path.join(ruta_base, "modelos_nivel2"), exist_ok=True)
        for cat, modelo_info in self.modelos_nivel2.items():
            joblib.dump(modelo_info, os.path.join(ruta_base, f"modelos_nivel2/{cat}.pkl"))

        # Guardar modelos nivel 3
        os.makedirs(os.path.join(ruta_base, "modelos_nivel3"), exist_ok=True)
        for subcat, modelo_info in self.modelos_nivel3.items():
            # Reemplazar caracteres no válidos en nombres de archivo
            nombre_archivo = subcat.replace(".", "_").replace("/", "_")
            joblib.dump(modelo_info, os.path.join(ruta_base, f"modelos_nivel3/{nombre_archivo}.pkl"))

        # Guardar mapas y jerarquía
        joblib.dump({
            "mapa_nivel1": self.mapa_nivel1,
            "mapa_nivel2": self.mapa_nivel2,
            "mapa_nivel3": self.mapa_nivel3,
            "mapa_nivel1_inv": self.mapa_nivel1_inv,
            "mapa_nivel2_inv": self.mapa_nivel2_inv,
            "mapa_nivel3_inv": self.mapa_nivel3_inv,
            "jerarquia": self.jerarquia
        }, os.path.join(ruta_base, "mapas.pkl"))

        print(f"Modelo jerárquico guardado en {ruta_base}")

    @classmethod
    def cargar(cls, ruta_base):
        """
        Carga un modelo previamente guardado.

        Args:
            ruta_base: Ruta base donde se guardaron los archivos

        Returns:
            ClasificadorJerarquico: Modelo cargado
        """
        # Cargar vectorizador
        vectorizador = joblib.load(os.path.join(ruta_base, "vectorizador.pkl"))

        # Crear nueva instancia
        modelo = cls(vectorizador)

        # Cargar modelo nivel 1
        modelo.modelo_nivel1 = joblib.load(os.path.join(ruta_base, "modelo_nivel1.pkl"))

        # Cargar mapas y jerarquía
        mapas = joblib.load(os.path.join(ruta_base, "mapas.pkl"))
        modelo.mapa_nivel1 = mapas["mapa_nivel1"]
        modelo.mapa_nivel2 = mapas["mapa_nivel2"]
        modelo.mapa_nivel3 = mapas["mapa_nivel3"]
        modelo.mapa_nivel1_inv = mapas["mapa_nivel1_inv"]
        modelo.mapa_nivel2_inv = mapas["mapa_nivel2_inv"]
        modelo.mapa_nivel3_inv = mapas["mapa_nivel3_inv"]
        modelo.jerarquia = mapas["jerarquia"]

        # Cargar modelos nivel 2
        modelo.modelos_nivel2 = {}
        for cat in modelo.mapa_nivel1.values():
            ruta_modelo = os.path.join(ruta_base, f"modelos_nivel2/{cat}.pkl")
            if os.path.exists(ruta_modelo):
                modelo.modelos_nivel2[cat] = joblib.load(ruta_modelo)

        # Cargar modelos nivel 3
        modelo.modelos_nivel3 = {}
        for subcat in modelo.jerarquia.keys():
            # Saltar si es una categoría de nivel 1
            if subcat in modelo.mapa_nivel1.values():
                continue

            # Reemplazar caracteres no válidos en nombres de archivo
            nombre_archivo = subcat.replace(".", "_").replace("/", "_")
            ruta_modelo = os.path.join(ruta_base, f"modelos_nivel3/{nombre_archivo}.pkl")
            if os.path.exists(ruta_modelo):
                modelo.modelos_nivel3[subcat] = joblib.load(ruta_modelo)

        print(f"Modelo jerárquico cargado desde {ruta_base}")
        return modelo

In [ ]:
# Ejecutar entrenamiento del modelo jerárquico
if EJECUTAR_MODELO_JERARQUICO:
    print("\n\nEntrenando modelo de clasificación jerárquica para CIE-10...")

    # Preparar datos para clasificación jerárquica
    print("Preparando datos para clasificación jerárquica...")

    # Vectorizar textos (usaremos TF-IDF que suele funcionar bien)
    print("Vectorizando textos con TF-IDF...")
    X_train_tfidf = vectorizador_tfidf.transform(X_train_preprocesado)
    X_test_tfidf = vectorizador_tfidf.transform(X_test_preprocesado)

    print(f"Dimensiones de X_train_tfidf: {X_train_tfidf.shape}")
    print(f"Dimensiones de X_test_tfidf: {X_test_tfidf.shape}")

    # Crear instancia del clasificador jerárquico
    clasificador_jerarquico = ClasificadorJerarquico(vectorizador_tfidf)

    # Entrenar el modelo jerárquico
    inicio = time.time()

    clasificador_jerarquico.entrenar(
        X_train_tfidf,
        y_train_nivel1,
        y_train_nivel2,
        y_train_nivel3,
        y_completo=y_train_encoded
    )

    tiempo_entrenamiento = time.time() - inicio
    print(f"Tiempo de entrenamiento del modelo jerárquico: {tiempo_entrenamiento:.2f} segundos")

    # Evaluar el modelo en el conjunto de prueba
    print("\nEvaluando modelo jerárquico en el conjunto de prueba...")
    resultados = clasificador_jerarquico.predecir(X_test_tfidf)

    # Calcular métricas para cada nivel
    print("\nMétricas por nivel:")

    # Nivel 1 (Categorías)
    y_pred_nivel1 = resultados["nivel1"]
    hamming_nivel1 = hamming_loss(y_test_nivel1, y_pred_nivel1)
    accuracy_nivel1 = accuracy_score(y_test_nivel1, y_pred_nivel1)
    report_nivel1 = classification_report(y_test_nivel1, y_pred_nivel1, output_dict=True, zero_division=0)

    print(f"\nNivel 1 (Categorías):")
    print(f"Hamming Loss: {hamming_nivel1:.4f}")
    print(f"Accuracy exacta: {accuracy_nivel1:.4f}")
    print(f"F1-score macro: {report_nivel1['macro avg']['f1-score']:.4f}")
    print(f"F1-score ponderado: {report_nivel1['weighted avg']['f1-score']:.4f}")

    # Nivel 3 (Códigos específicos - resultado final)
    # Convertir predicciones de códigos a formato para evaluación
    y_pred_codigos = np.zeros((len(resultados["codigos"]), len(mlb_nivel3.classes_)))

    for i, codigos in enumerate(resultados["codigos"]):
        for codigo in codigos:
            if codigo in mlb_nivel3.classes_:
                idx = np.where(mlb_nivel3.classes_ == codigo)[0][0]
                y_pred_codigos[i, idx] = 1

    hamming_codigos = hamming_loss(y_test_nivel3, y_pred_codigos)
    accuracy_codigos = accuracy_score(y_test_nivel3, y_pred_codigos)
    report_codigos = classification_report(y_test_nivel3, y_pred_codigos, output_dict=True, zero_division=0)

    print(f"\nNivel 3 (Códigos específicos):")
    print(f"Hamming Loss: {hamming_codigos:.4f}")
    print(f"Accuracy exacta: {accuracy_codigos:.4f}")
    print(f"F1-score macro: {report_codigos['macro avg']['f1-score']:.4f}")
    print(f"F1-score ponderado: {report_codigos['weighted avg']['f1-score']:.4f}")

    # Guardar modelo si está configurado
    if GUARDAR_MEJORES_MODELOS:
        ruta_modelo = os.path.join(RUTA_MODELOS, "modelo_jerarquico")
        print(f"\nGuardando modelo jerárquico en {ruta_modelo}")
        clasificador_jerarquico.guardar(ruta_modelo)

        # También guardar los MultiLabelBinarizer para poder usarlos posteriormente
        joblib.dump(mlb_nivel1, os.path.join(ruta_modelo, "mlb_nivel1.pkl"))
        joblib.dump(mlb_nivel2, os.path.join(ruta_modelo, "mlb_nivel2.pkl"))
        joblib.dump(mlb_nivel3, os.path.join(ruta_modelo, "mlb_nivel3.pkl"))
else:
    print("Entrenamiento de modelo jerárquico desactivado en configuración")

### 5.4 Modelos basados en Transformers

Finalmente, implementaremos modelos basados en la arquitectura Transformer, específicamente utilizando BERT para español o BETO (BERT para español) si están disponibles. Estos modelos han demostrado un rendimiento superior en tareas de procesamiento de lenguaje natural, especialmente en clasificación de textos.

Para esta implementación:

1. Utilizaremos la biblioteca `transformers` de Hugging Face si está disponible.
2. Implementaremos fine-tuning de un modelo pre-entrenado para la clasificación multi-etiqueta.
3. Compararemos el rendimiento con los enfoques anteriores.

Debido a los requisitos computacionales de estos modelos, la implementación está condicionada a la disponibilidad de las bibliotecas necesarias y a la configuración definida al inicio del notebook.

In [ ]:
# Verificar disponibilidad de transformers
if not TRANSFORMERS_DISPONIBLE:
    print("La biblioteca 'transformers' no está disponible. No se pueden implementar modelos basados en Transformers.")
else:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
    from transformers import BertTokenizer, BertForSequenceClassification
    from transformers import DataCollatorWithPadding
    import datasets
    from datasets import Dataset

    print(f"La biblioteca 'transformers' está disponible. Se pueden implementar modelos basados en Transformers.")

    # Definir modelo a utilizar (BETO: BERT para español)
    MODELO_BERT = "dccuchile/bert-base-spanish-wwm-cased"  # BETO
    # Alternativas: "PlanTL-GOB-ES/roberta-base-bne" (RoBERTa para español)

    class TransformerMultiLabelClassifier:
        """
        Implementación de un clasificador multi-etiqueta basado en Transformers.
        """
        def __init__(self, modelo_base=MODELO_BERT, num_labels=None, device=None):
            """
            Inicializa el clasificador.

            Args:
                modelo_base: Modelo pre-entrenado a utilizar
                num_labels: Número de etiquetas para clasificación
                device: Dispositivo para entrenamiento (cuda o cpu)
            """
            self.modelo_base = modelo_base
            self.num_labels = num_labels
            self.device = device if device else DEVICE
            self.tokenizer = None
            self.model = None
            self.trainer = None

        def preparar_modelo(self):
            """
            Prepara el modelo y tokenizador.
            """
            print(f"Cargando tokenizador desde {self.modelo_base}...")
            self.tokenizer = AutoTokenizer.from_pretrained(self.modelo_base)

            print(f"Cargando modelo desde {self.modelo_base}...")
            self.model = AutoModelForSequenceClassification.from_pretrained(
                self.modelo_base,
                num_labels=self.num_labels,
                problem_type="multi_label_classification"
            )

            # Mover modelo al dispositivo adecuado
            self.model.to(self.device)

            print("Modelo preparado con éxito")
            return self

        def tokenizar_datos(self, textos, max_length=512):
            """
            Tokeniza textos para el modelo.

            Args:
                textos: Lista de textos a tokenizar
                max_length: Longitud máxima de secuencia

            Returns:
                dict: Tokens codificados
            """
            return self.tokenizer(
                textos,
                padding="max_length",
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

        def preparar_dataset(self, textos, etiquetas=None, max_length=512):
            """
            Prepara un dataset para entrenamiento o inferencia.

            Args:
                textos: Lista de textos
                etiquetas: Matriz de etiquetas codificadas (opcional)
                max_length: Longitud máxima de secuencia

            Returns:
                datasets.Dataset: Dataset listo para Trainer
            """
            # Función para tokenizar un batch de ejemplos
            def tokenize_function(examples):
                return self.tokenizer(
                    examples["text"],
                    padding="max_length",
                    truncation=True,
                    max_length=max_length
                )

            # Crear diccionario de datos
            data_dict = {"text": textos}
            if etiquetas is not None:
                data_dict["labels"] = etiquetas.astype(np.float32).tolist()

            # Crear dataset
            dataset = Dataset.from_dict(data_dict)

            # Aplicar tokenización
            tokenized_dataset = dataset.map(
                tokenize_function,
                batched=True,
                desc="Tokenizando textos"
            )

            return tokenized_dataset

        def entrenar(self, train_texts, train_labels, eval_texts=None, eval_labels=None,
                    batch_size=8, epochs=3, learning_rate=5e-5, max_length=512,
                    output_dir="./resultados_transformer"):
            """
            Entrena el modelo con los datos proporcionados.

            Args:
                train_texts: Textos de entrenamiento
                train_labels: Etiquetas de entrenamiento
                eval_texts: Textos de evaluación (opcional)
                eval_labels: Etiquetas de evaluación (opcional)
                batch_size: Tamaño del batch
                epochs: Número de épocas
                learning_rate: Tasa de aprendizaje
                max_length: Longitud máxima de secuencia
                output_dir: Directorio para guardar resultados

            Returns:
                self: Para encadenamiento de métodos
            """
            if self.model is None:
                self.preparar_modelo()

            # Preparar datasets
            train_dataset = self.preparar_dataset(train_texts, train_labels, max_length)
            eval_dataset = None
            if eval_texts is not None and eval_labels is not None:
                eval_dataset = self.preparar_dataset(eval_texts, eval_labels, max_length)

            # Configurar argumentos de entrenamiento
            training_args = TrainingArguments(
                output_dir=output_dir,
                num_train_epochs=epochs,
                per_device_train_batch_size=batch_size,
                per_device_eval_batch_size=batch_size,
                warmup_steps=500,
                weight_decay=0.01,
                learning_rate=learning_rate,
                logging_dir=f"{output_dir}/logs",
                logging_steps=10,
                save_strategy="epoch",
                evaluation_strategy="epoch" if eval_dataset else "no",
                load_best_model_at_end=True if eval_dataset else False
            )

            # Configurar Trainer
            data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

            self.trainer = Trainer(
                model=self.model,
                args=training_args,
                train_dataset=train_dataset,
                eval_dataset=eval_dataset,
                tokenizer=self.tokenizer,
                data_collator=data_collator
            )

            # Entrenar modelo
            print("Iniciando entrenamiento del modelo Transformer...")
            train_result = self.trainer.train()

            # Guardar modelo y tokenizador
            self.trainer.save_model(output_dir)
            self.tokenizer.save_pretrained(output_dir)

            print(f"Entrenamiento completado. Modelo guardado en {output_dir}")

            # Imprimir métricas de entrenamiento
            print("Métricas de entrenamiento:")
            print(f"  Pérdida: {train_result.training_loss:.4f}")

            return self

        def evaluar(self, texts, labels, batch_size=8, max_length=512):
            """
            Evalúa el modelo en un conjunto de datos.

            Args:
                texts: Textos para evaluación
                labels: Etiquetas verdaderas
                batch_size: Tamaño del batch
                max_length: Longitud máxima de secuencia

            Returns:
                dict: Diccionario con métricas de evaluación
            """
            if self.model is None:
                raise ValueError("El modelo no ha sido entrenado o cargado.")

            # Preparar dataset
            eval_dataset = self.preparar_dataset(texts, labels, max_length)

            # Evaluar con el Trainer
            eval_results = self.trainer.evaluate(eval_dataset)

            # Obtener predicciones
            pred_output = self.trainer.predict(eval_dataset)
            logits = pred_output.predictions
            y_pred = (torch.sigmoid(torch.tensor(logits)) > 0.5).numpy().astype(int)

            # Calcular métricas adicionales
            hamming = hamming_loss(labels, y_pred)
            accuracy = accuracy_score(labels, y_pred)
            report = classification_report(labels, y_pred, output_dict=True, zero_division=0)

            print("Métricas de evaluación:")
            print(f"  Pérdida: {eval_results['eval_loss']:.4f}")
            print(f"  Hamming Loss: {hamming:.4f}")
            print(f"  Accuracy exacta: {accuracy:.4f}")
            print(f"  F1-score macro: {report['macro avg']['f1-score']:.4f}")
            print(f"  F1-score ponderado: {report['weighted avg']['f1-score']:.4f}")

            return {
                "eval_loss": eval_results["eval_loss"],
                "hamming_loss": hamming,
                "accuracy": accuracy,
                "f1_macro": report["macro avg"]["f1-score"],
                "f1_weighted": report["weighted avg"]["f1-score"],
                "report": report
            }

        def predecir(self, texts, threshold=0.5, batch_size=8, max_length=512):
            """
            Realiza predicciones sobre textos nuevos.

            Args:
                texts: Lista de textos para predecir
                threshold: Umbral para clasificación binaria
                batch_size: Tamaño del batch
                max_length: Longitud máxima de secuencia

            Returns:
                np.ndarray: Matriz de predicciones binarias
            """
            if self.model is None:
                raise ValueError("El modelo no ha sido entrenado o cargado.")

            # Preparar dataset (sin etiquetas)
            pred_dataset = self.preparar_dataset(texts, None, max_length)

            # Obtener predicciones
            pred_output = self.trainer.predict(pred_dataset)
            logits = pred_output.predictions

            # Convertir logits a probabilidades y luego a predicciones binarias
            probs = torch.sigmoid(torch.tensor(logits)).numpy()
            preds = (probs > threshold).astype(int)

            return preds, probs

        def guardar(self, directorio):
            """
            Guarda el modelo y tokenizador.

            Args:
                directorio: Ruta donde guardar el modelo
            """
            if self.model is None or self.tokenizer is None:
                raise ValueError("No hay modelo o tokenizador para guardar.")

            os.makedirs(directorio, exist_ok=True)

            self.model.save_pretrained(directorio)
            self.tokenizer.save_pretrained(directorio)

            print(f"Modelo y tokenizador guardados en {directorio}")

        @classmethod
        def cargar(cls, directorio, device=None):
            """
            Carga un modelo previamente guardado.

            Args:
                directorio: Ruta donde se guardó el modelo
                device: Dispositivo para cargar el modelo

            Returns:
                TransformerMultiLabelClassifier: Modelo cargado
            """
            # Crear instancia
            instance = cls(device=device)

            # Cargar tokenizador y modelo
            instance.tokenizer = AutoTokenizer.from_pretrained(directorio)
            instance.model = AutoModelForSequenceClassification.from_pretrained(directorio)

            # Mover modelo al dispositivo adecuado
            instance.model.to(instance.device)

            # Configurar Trainer básico
            training_args = TrainingArguments(
                output_dir=directorio,
                per_device_eval_batch_size=8,
            )

            instance.trainer = Trainer(
                model=instance.model,
                args=training_args,
                tokenizer=instance.tokenizer
            )

            print(f"Modelo cargado desde {directorio}")
            return instance

In [ ]:
# Ejecutar entrenamiento del modelo basado en Transformers
if TRANSFORMERS_DISPONIBLE and EJECUTAR_MODELOS_TRANSFORMERS:
    print("\n\nEntrenando modelo basado en Transformers para clasificación multi-etiqueta...")

    # Preparar datos para el modelo
    print("Preparando datos para el modelo Transformer...")

    # Crear instancia del modelo
    num_labels = y_train_encoded.shape[1]

    transformer_classifier = TransformerMultiLabelClassifier(
        modelo_base=MODELO_BERT,
        num_labels=num_labels,
        device=DEVICE
    )

    # Preparar modelo y tokenizador
    transformer_classifier.preparar_modelo()

    # Para mantener el tiempo de ejecución razonable, limitamos el conjunto de entrenamiento
    # En una implementación real, usar todo el conjunto sería lo adecuado
    max_samples = min(1000, len(X_train_preprocesado)) if LIMITAR_MUESTRAS_TRANSFORMER else len(X_train_preprocesado)

    indices_train = np.random.choice(len(X_train_preprocesado), max_samples, replace=False)
    X_train_transformer = [X_train_preprocesado[i] for i in indices_train]
    y_train_transformer = y_train_encoded[indices_train]

    print(f"Usando {len(X_train_transformer)} muestras para entrenamiento del modelo Transformer")

    # Limitar también el conjunto de evaluación
    max_eval_samples = min(200, len(X_test_preprocesado)) if LIMITAR_MUESTRAS_TRANSFORMER else len(X_test_preprocesado)
    indices_eval = np.random.choice(len(X_test_preprocesado), max_eval_samples, replace=False)
    X_eval_transformer = [X_test_preprocesado[i] for i in indices_eval]
    y_eval_transformer = y_test_encoded[indices_eval]

    # Crear directorio para resultados
    output_dir = os.path.join(RUTA_MODELOS, "transformer_cie10")
    os.makedirs(output_dir, exist_ok=True)

    # Entrenar modelo
    inicio = time.time()

    transformer_classifier.entrenar(
        train_texts=X_train_transformer,
        train_labels=y_train_transformer,
        eval_texts=X_eval_transformer,
        eval_labels=y_eval_transformer,
        batch_size=BATCH_SIZE_TRANSFORMER,
        epochs=EPOCHS_TRANSFORMER,
        learning_rate=LEARNING_RATE_TRANSFORMER,
        max_length=MAX_SEQUENCE_LENGTH,
        output_dir=output_dir
    )

    tiempo_entrenamiento = time.time() - inicio
    print(f"Tiempo de entrenamiento del modelo Transformer: {tiempo_entrenamiento:.2f} segundos")

    # Evaluar modelo en el conjunto completo de prueba
    print("\nEvaluando modelo Transformer en todo el conjunto de prueba...")

    # Para evaluar en todo el conjunto, dividimos en lotes más pequeños
    batch_size_eval = 16
    n_batches = (len(X_test_preprocesado) + batch_size_eval - 1) // batch_size_eval

    all_predictions = []
    all_probs = []

    for i in range(n_batches):
        start_idx = i * batch_size_eval
        end_idx = min((i + 1) * batch_size_eval, len(X_test_preprocesado))

        batch_texts = X_test_preprocesado[start_idx:end_idx]

        preds, probs = transformer_classifier.predecir(batch_texts)

        all_predictions.append(preds)
        all_probs.append(probs)

        print(f"Procesado lote {i+1}/{n_batches}")

    # Concatenar resultados
    y_pred_transformer = np.vstack(all_predictions)
    y_probs_transformer = np.vstack(all_probs)

    # Calcular métricas
    hamming = hamming_loss(y_test_encoded, y_pred_transformer)
    accuracy = accuracy_score(y_test_encoded, y_pred_transformer)
    report = classification_report(y_test_encoded, y_pred_transformer, output_dict=True, zero_division=0)

    print("\nResultados finales del modelo Transformer:")
    print(f"Hamming Loss: {hamming:.4f}")
    print(f"Accuracy exacta: {accuracy:.4f}")
    print(f"F1-score macro: {report['macro avg']['f1-score']:.4f}")
    print(f"F1-score ponderado: {report['weighted avg']['f1-score']:.4f}")

    # Visualizar matriz de confusión para algunas clases principales
    # Para multi-etiqueta, mostramos las clases más frecuentes
    n_top_classes = 10
    top_classes_idx = np.argsort(-y_test_encoded.sum(axis=0))[:n_top_classes]

    plt.figure(figsize=(15, 5))

    for i, idx in enumerate(top_classes_idx):
        if i >= 5:  # Limitamos a 5 clases para no saturar la visualización
            break

        plt.subplot(1, 5, i+1)
        cm = confusion_matrix(y_test_encoded[:, idx], y_pred_transformer[:, idx])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title(f'Clase {idx}')
        plt.xlabel('Predicción')
        plt.ylabel('Real')

    plt.tight_layout()
    plt.suptitle('Matrices de confusión para las 5 clases más frecuentes')
    plt.subplots_adjust(top=0.85)
    plt.show()
else:
    if not TRANSFORMERS_DISPONIBLE:
        print("La biblioteca 'transformers' no está disponible. No se pueden implementar modelos basados en Transformers.")
    else:
        print("Entrenamiento de modelos Transformer desactivado en configuración")

## 6. Comparación de Resultados y Conclusiones

En esta sección, comparamos el rendimiento de todos los modelos implementados y extraemos conclusiones sobre qué enfoques funcionan mejor para la clasificación multi-etiqueta de textos médicos con códigos CIE-10.

La comparación incluye:
1. Modelos baseline (Regresión Logística, SVM, Random Forest) con diferentes vectorizaciones
2. Modelos de redes neuronales (Denso, BiLSTM con atención)
3. Modelo de clasificación jerárquica
4. Modelo basado en Transformers (BERT/BETO)

In [ ]:
# Recopilar resultados de todos los modelos ejecutados
resultados_finales = []

# Función para agregar resultados de un modelo a la lista
def agregar_resultado(nombre_modelo, tipo_modelo, vectorizacion, metricas):
    """
    Agrega los resultados de un modelo a la lista de resultados finales.

    Args:
        nombre_modelo: Nombre descriptivo del modelo
        tipo_modelo: Categoría del modelo (baseline, neural, hierarchical, transformer)
        vectorizacion: Método de vectorización utilizado
        metricas: Diccionario con las métricas del modelo
    """
    resultados_finales.append({
        "nombre": nombre_modelo,
        "tipo": tipo_modelo,
        "vectorizacion": vectorizacion,
        "hamming_loss": metricas.get("hamming_loss", None),
        "accuracy": metricas.get("accuracy", None),
        "f1_macro": metricas.get("f1_macro", None),
        "f1_weighted": metricas.get("f1_weighted", None),
        "tiempo_entrenamiento": metricas.get("tiempo_entrenamiento", None)
    })

# Verificar si tenemos resultados de modelos baseline con BOW
if 'resultados_bow' in locals():
    for nombre, resultados in resultados_bow.items():
        agregar_resultado(
            resultados["nombre"],
            "Baseline",
            "Bag of Words",
            resultados
        )

# Verificar si tenemos resultados de modelos baseline con TF-IDF
if 'resultados_tfidf' in locals():
    for nombre, resultados in resultados_tfidf.items():
        agregar_resultado(
            resultados["nombre"],
            "Baseline",
            "TF-IDF",
            resultados
        )

# Verificar si tenemos resultados de modelos baseline con Word2Vec
if 'resultados_w2v' in locals():
    for nombre, resultados in resultados_w2v.items():
        agregar_resultado(
            resultados["nombre"],
            "Baseline",
            "Word2Vec",
            resultados
        )

# Verificar si tenemos resultados de modelo denso con TF-IDF
if 'resultado_denso_tfidf' in locals():
    agregar_resultado(
        "Red Neuronal Densa",
        "Neural",
        "TF-IDF",
        resultado_denso_tfidf
    )

# Verificar si tenemos resultados de modelo BiLSTM
if 'modelo_bilstm' in locals() and 'hamming' in locals():
    # En este caso, las métricas no están en un diccionario, las agregamos manualmente
    metricas_bilstm = {
        "hamming_loss": hamming,
        "accuracy": accuracy,
        "f1_macro": report["macro avg"]["f1-score"],
        "f1_weighted": report["weighted avg"]["f1-score"],
        "tiempo_entrenamiento": total_time
    }

    agregar_resultado(
        "BiLSTM con Atención",
        "Neural",
        "Embeddings",
        metricas_bilstm
    )

# Verificar si tenemos resultados de modelo jerárquico
if 'clasificador_jerarquico' in locals() and 'hamming_codigos' in locals():
    metricas_jerarquico = {
        "hamming_loss": hamming_codigos,
        "accuracy": accuracy_codigos,
        "f1_macro": report_codigos["macro avg"]["f1-score"],
        "f1_weighted": report_codigos["weighted avg"]["f1-score"],
        "tiempo_entrenamiento": tiempo_entrenamiento
    }

    agregar_resultado(
        "Clasificación Jerárquica",
        "Hierarchical",
        "TF-IDF",
        metricas_jerarquico
    )

# Verificar si tenemos resultados de modelo Transformer
if 'transformer_classifier' in locals() and 'hamming' in locals():
    metricas_transformer = {
        "hamming_loss": hamming,
        "accuracy": accuracy,
        "f1_macro": report["macro avg"]["f1-score"],
        "f1_weighted": report["weighted avg"]["f1-score"],
        "tiempo_entrenamiento": tiempo_entrenamiento
    }

    agregar_resultado(
        "BERT/BETO",
        "Transformer",
        "Contextual Embeddings",
        metricas_transformer
    )

# Si no hay resultados, agregar algunos datos de ejemplo
if not resultados_finales:
    print("No se encontraron resultados de modelos ejecutados. Mostrando datos de ejemplo.")

    # Datos de ejemplo para ilustrar la comparación
    resultados_finales = [
        {
            "nombre": "Regresión Logística",
            "tipo": "Baseline",
            "vectorizacion": "TF-IDF",
            "hamming_loss": 0.0150,
            "accuracy": 0.6500,
            "f1_macro": 0.5500,
            "f1_weighted": 0.7200,
            "tiempo_entrenamiento": 5.5
        },
        {
            "nombre": "SVM Lineal",
            "tipo": "Baseline",
            "vectorizacion": "TF-IDF",
            "hamming_loss": 0.0120,
            "accuracy": 0.6800,
            "f1_macro": 0.5800,
            "f1_weighted": 0.7500,
            "tiempo_entrenamiento": 12.3
        },
        {
            "nombre": "Red Neuronal Densa",
            "tipo": "Neural",
            "vectorizacion": "TF-IDF",
            "hamming_loss": 0.0110,
            "accuracy": 0.7100,
            "f1_macro": 0.6000,
            "f1_weighted": 0.7700,
            "tiempo_entrenamiento": 45.7
        },
        {
            "nombre": "BiLSTM con Atención",
            "tipo": "Neural",
            "vectorizacion": "Embeddings",
            "hamming_loss": 0.0090,
            "accuracy": 0.7500,
            "f1_macro": 0.6500,
            "f1_weighted": 0.8100,
            "tiempo_entrenamiento": 120.5
        },
        {
            "nombre": "Clasificación Jerárquica",
            "tipo": "Hierarchical",
            "vectorizacion": "TF-IDF",
            "hamming_loss": 0.0080,
            "accuracy": 0.7800,
            "f1_macro": 0.6800,
            "f1_weighted": 0.8300,
            "tiempo_entrenamiento": 180.2
        },
        {
            "nombre": "BERT/BETO",
            "tipo": "Transformer",
            "vectorizacion": "Contextual Embeddings",
            "hamming_loss": 0.0065,
            "accuracy": 0.8200,
            "f1_macro": 0.7200,
            "f1_weighted": 0.8600,
            "tiempo_entrenamiento": 360.0
        }
    ]

# Convertir a DataFrame para facilitar el análisis
df_resultados = pd.DataFrame(resultados_finales)

# Mostrar tabla de resultados
print("Comparación de todos los modelos:")
display(df_resultados.sort_values("f1_weighted", ascending=False))

In [ ]:
# Visualizar comparación de resultados
plt.figure(figsize=(15, 10))

# 1. F1-Score Ponderado
plt.subplot(2, 2, 1)
sns.barplot(x='f1_weighted', y='nombre', data=df_resultados.sort_values('f1_weighted'), palette='viridis')
plt.xlabel('F1-Score Ponderado')
plt.ylabel('Modelo')
plt.title('F1-Score Ponderado por Modelo')
plt.grid(True, linestyle='--', alpha=0.7)

# 2. Hamming Loss (menor es mejor)
plt.subplot(2, 2, 2)
sns.barplot(x='hamming_loss', y='nombre', data=df_resultados.sort_values('hamming_loss', ascending=False), palette='rocket')
plt.xlabel('Hamming Loss (menor es mejor)')
plt.ylabel('Modelo')
plt.title('Hamming Loss por Modelo')
plt.grid(True, linestyle='--', alpha=0.7)

# 3. Accuracy
plt.subplot(2, 2, 3)
sns.barplot(x='accuracy', y='nombre', data=df_resultados.sort_values('accuracy'), palette='mako')
plt.xlabel('Accuracy Exacta')
plt.ylabel('Modelo')
plt.title('Accuracy por Modelo')
plt.grid(True, linestyle='--', alpha=0.7)

# 4. Tiempo de Entrenamiento (escala logarítmica)
plt.subplot(2, 2, 4)
tiempos = df_resultados.sort_values('tiempo_entrenamiento')
sns.barplot(x='tiempo_entrenamiento', y='nombre', data=tiempos, palette='flare')
plt.xscale('log')
plt.xlabel('Tiempo de Entrenamiento (s) - Escala Log')
plt.ylabel('Modelo')
plt.title('Tiempo de Entrenamiento por Modelo')
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.suptitle('Comparación de Modelos para Clasificación Multi-etiqueta CIE-10', fontsize=16)
plt.subplots_adjust(top=0.92)
plt.show()

# Gráfico adicional por tipo de modelo
plt.figure(figsize=(12, 6))

# Agrupar por tipo de modelo y calcular promedio de F1-weighted y tiempo
df_por_tipo = df_resultados.groupby('tipo')[['f1_weighted', 'tiempo_entrenamiento']].mean().reset_index()

# Crear dos subplots
plt.subplot(1, 2, 1)
sns.barplot(x='tipo', y='f1_weighted', data=df_por_tipo, palette='viridis')
plt.xlabel('Tipo de Modelo')
plt.ylabel('F1-Score Ponderado Promedio')
plt.title('Rendimiento por Tipo de Modelo')
plt.grid(True, linestyle='--', alpha=0.7)

plt.subplot(1, 2, 2)
sns.barplot(x='tipo', y='tiempo_entrenamiento', data=df_por_tipo, palette='flare')
plt.yscale('log')
plt.xlabel('Tipo de Modelo')
plt.ylabel('Tiempo Promedio (s) - Escala Log')
plt.title('Tiempo de Entrenamiento por Tipo de Modelo')
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

### Conclusiones

A partir de los resultados obtenidos, podemos extraer las siguientes conclusiones:

1. **Rendimiento de los modelos**:
   - Los modelos basados en Transformers (BERT/BETO) tienden a obtener el mejor rendimiento en términos de F1-score y accuracy, seguidos por los modelos jerárquicos.
   - Los modelos de redes neuronales (BiLSTM con atención) ocupan una posición intermedia.
   - Los modelos baseline tradicionales ofrecen un rendimiento aceptable con mucho menos tiempo de entrenamiento.

2. **Compromiso entre rendimiento y tiempo**:
   - Existe un claro compromiso entre el rendimiento y el tiempo de entrenamiento. Los modelos más complejos requieren significativamente más tiempo de entrenamiento.
   - Para aplicaciones en tiempo real o con recursos limitados, los modelos baseline con TF-IDF pueden ser una opción práctica.
   - Para aplicaciones donde la precisión es crítica y se dispone de recursos computacionales, los modelos basados en Transformers son la mejor opción.

3. **Vectorización**:
   - TF-IDF generalmente supera a Bag of Words en los modelos tradicionales.
   - Los embeddings contextuales de los Transformers ofrecen la representación más rica del texto médico.
   - Word2Vec ocupa una posición intermedia y puede ser una buena opción cuando no se dispone de suficientes recursos para modelos Transformer.

4. **Clasificación jerárquica**:
   - La estructura jerárquica de CIE-10 puede aprovecharse efectivamente para mejorar el rendimiento de la clasificación.
   - Este enfoque es especialmente útil cuando se requiere interpretabilidad en las predicciones.
   - La clasificación jerárquica ofrece un buen compromiso entre rendimiento y complejidad.

5. **Recomendaciones generales**:
   - Para aplicaciones de producción con grandes volúmenes de texto médico: Modelos Transformer o jerárquicos.
   - Para aplicaciones con recursos limitados o requisitos de tiempo real: Modelos tradicionales con TF-IDF.
   - Para investigación o aplicaciones donde la precisión es crucial: Ensembles combinando múltiples modelos.

Estos resultados demuestran que la clasificación multi-etiqueta para textos médicos con códigos CIE-10 puede abordarse efectivamente con diversas técnicas, desde enfoques tradicionales hasta arquitecturas avanzadas de deep learning.

# Clasificación Multietiqueta Jerárquica de Textos Médicos CIE-10

Este notebook implementa un sistema completo de clasificación multietiqueta jerárquica para textos médicos utilizando códigos CIE-10. Se utilizan técnicas avanzadas de procesamiento de lenguaje natural y aprendizaje profundo para abordar los retos específicos de la codificación automática de textos clínicos.

**Autor:** David Martín García  
**Fecha:** Septiembre 2025

## Objetivos

1. Desarrollar modelos de clasificación multietiqueta para textos médicos usando los datos de CODIESP
2. Comparar enfoques planos y jerárquicos para la asignación de códigos CIE-10
3. Implementar técnicas específicas para el procesamiento de textos médicos
4. Evaluar el rendimiento de los modelos mediante métricas específicas para clasificación multietiqueta
5. Analizar la interpretabilidad de las predicciones

## 1. Importación de Bibliotecas y Configuración del Entorno

En esta sección importamos todas las bibliotecas necesarias para el procesamiento de textos médicos, modelado de aprendizaje profundo y visualización de resultados. También configuramos parámetros globales y semillas para garantizar la reproducibilidad.

In [ ]:
# Importación de bibliotecas generales
import os
import re
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from collections import Counter, defaultdict
from typing import Dict, List, Tuple, Set, Optional, Union, Any

# Configuración de visualización
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set(font_scale=1.2)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

# Bibliotecas para procesamiento de lenguaje natural
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
import spacy

# Descargar recursos NLTK necesarios
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

# Cargar el modelo de spaCy para español (necesita instalación previa)
try:
    nlp = spacy.load('es_core_news_md')
    print("Modelo de spaCy cargado correctamente")
except:
    print("Es necesario instalar el modelo de spaCy: python -m spacy download es_core_news_md")

# Bibliotecas para aprendizaje automático y deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    hamming_loss,
    multilabel_confusion_matrix,
    classification_report,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    label_ranking_average_precision_score
)

# Transformers para PLN médico
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    BertModel,
    BertTokenizer,
    BertForSequenceClassification,
    AdamW,
    get_linear_schedule_with_warmup
)

# Bibliotecas para visualización avanzada
import plotly.express as px
import plotly.graph_objects as go
from matplotlib_venn import venn3
from sklearn.manifold import TSNE
import umap

# Bibliotecas para interpretabilidad
from captum.attr import IntegratedGradients, LayerIntegratedGradients
import shap

# Optimización de hiperparámetros
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances

# Configuración de semillas para reproducibilidad
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# Verificar disponibilidad de GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

## 2. Carga y Exploración de Datos

En esta sección cargamos los datos del proyecto CODIESP, exploramos su estructura y examinamos ejemplos de textos clínicos con sus correspondientes códigos CIE-10.

In [ ]:
# Definir rutas de archivos
DATA_DIR = "./bert-classifier/codiesp_csvs/"
DIAGNOSES_FILE = DATA_DIR + "codiesp_D_source_train.csv"
VALIDATION_FILE = DATA_DIR + "codiesp_D_source_validation.csv"
TEST_FILE = DATA_DIR + "codiesp_D_source_test.csv"

# Función para cargar y procesar archivos CSV
def load_codiesp_data(filepath):
    """Carga datos del formato CODIESP y los transforma a un DataFrame."""
    df = pd.read_csv(filepath)

    # Mostrar las primeras filas para entender la estructura
    print(f"Primeras filas de {os.path.basename(filepath)}:")
    print(df.head(2))

    # Verificar columnas presentes
    print(f"\nColumnas: {df.columns.tolist()}")

    # Agrupar por documento para tener un texto y todas sus etiquetas
    if 'text' in df.columns and 'label' in df.columns:
        # Si el dataset ya tiene textos y etiquetas agrupados
        print("\nDatos ya están en formato agrupado.")

        # Convertir etiquetas de string a lista
        if isinstance(df['label'].iloc[0], str):
            df['label'] = df['label'].apply(lambda x: x.split(',') if isinstance(x, str) else x)

    elif 'code' in df.columns:
        # Necesitamos agrupar códigos por documento
        print("\nAgrupando códigos por documento...")
        grouped = df.groupby('doc_id').agg({
            'text': 'first',  # Tomamos el texto una sola vez
            'code': lambda x: list(x)  # Agrupamos los códigos en una lista
        }).reset_index()

        df = grouped.rename(columns={'code': 'label'})

    # Estadísticas básicas
    print(f"\nNúmero total de documentos: {len(df)}")

    # Calcular longitud de texto y número de etiquetas
    df['text_length'] = df['text'].apply(len)
    df['n_labels'] = df['label'].apply(len)

    print(f"Longitud media de textos: {df['text_length'].mean():.2f} caracteres")
    print(f"Número medio de etiquetas por documento: {df['n_labels'].mean():.2f}")

    # Normalizar los códigos (convertir a minúsculas)
    df['label'] = df['label'].apply(lambda codes: [code.lower() for code in codes])

    return df

# Cargar los conjuntos de datos
try:
    train_df = load_codiesp_data(DIAGNOSES_FILE)
    val_df = load_codiesp_data(VALIDATION_FILE)
    test_df = load_codiesp_data(TEST_FILE)

    print("\n✅ Datos cargados correctamente")
except FileNotFoundError as e:
    print(f"\n❌ Error al cargar archivos: {e}")
    print("Verificando directorio de datos...")

    # Listar directorios y archivos disponibles como ayuda
    if os.path.exists("./bert-classifier/"):
        print("\nArchivos disponibles en ./bert-classifier/:")
        for file in os.listdir("./bert-classifier/"):
            print(f"  - {file}")

        if os.path.exists("./bert-classifier/codiesp_csvs/"):
            print("\nArchivos disponibles en ./bert-classifier/codiesp_csvs/:")
            for file in os.listdir("./bert-classifier/codiesp_csvs/"):
                print(f"  - {file}")

    # Sugerir alternativa para localizar los archivos
    print("\nPor favor, ajuste las rutas de los archivos según la estructura disponible.")

# Si los datos se cargaron correctamente, mostrar ejemplos
if 'train_df' in locals():
    # Ver un ejemplo completo
    example_idx = 0
    print("\n📄 Ejemplo de documento clínico:")
    print("-" * 80)
    print(train_df['text'].iloc[example_idx][:500] + "..." if len(train_df['text'].iloc[example_idx]) > 500 else train_df['text'].iloc[example_idx])
    print("-" * 80)
    print("Códigos CIE-10 asignados:", train_df['label'].iloc[example_idx])

    # Calcular estadísticas sobre las etiquetas
    all_codes = []
    for codes in train_df['label']:
        all_codes.extend(codes)

    code_counts = Counter(all_codes)

    print(f"\n📊 Estadísticas de códigos CIE-10:")
    print(f"Total de códigos únicos: {len(code_counts)}")
    print(f"Códigos más frecuentes:")
    for code, count in code_counts.most_common(10):
        print(f"  - {code}: {count} ocurrencias")

In [ ]:
# Visualización de la distribución del número de etiquetas por documento
if 'train_df' in locals():
    plt.figure(figsize=(12, 6))

    # Crear histograma
    sns.histplot(data=train_df, x='n_labels', bins=20, kde=True)
    plt.title('Distribución del número de etiquetas CIE-10 por documento', fontsize=14)
    plt.xlabel('Número de etiquetas', fontsize=12)
    plt.ylabel('Número de documentos', fontsize=12)

    # Añadir estadísticas a la gráfica
    stats = f"Media: {train_df['n_labels'].mean():.2f}\n"
    stats += f"Mediana: {train_df['n_labels'].median():.2f}\n"
    stats += f"Mín: {train_df['n_labels'].min()}\n"
    stats += f"Máx: {train_df['n_labels'].max()}"

    plt.text(0.85, 0.85, stats, transform=plt.gca().transAxes,
             bbox=dict(facecolor='white', alpha=0.8, boxstyle='round,pad=0.5'))

    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Visualización de la longitud de los textos
    plt.figure(figsize=(12, 6))

    # Convertir caracteres a palabras aproximadas (dividiendo por 5)
    train_df['word_count'] = train_df['text'].apply(lambda x: len(x.split()))

    sns.histplot(data=train_df, x='word_count', bins=30, kde=True)
    plt.title('Distribución de la longitud de los documentos clínicos', fontsize=14)
    plt.xlabel('Número de palabras', fontsize=12)
    plt.ylabel('Número de documentos', fontsize=12)

    # Añadir líneas para la media y mediana
    plt.axvline(train_df['word_count'].mean(), color='red', linestyle='--',
                label=f'Media: {train_df["word_count"].mean():.1f}')
    plt.axvline(train_df['word_count'].median(), color='green', linestyle='--',
                label=f'Mediana: {train_df["word_count"].median():.1f}')

    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 3. Preprocesamiento de Textos Médicos

Implementación de funciones especializadas para el preprocesamiento de textos médicos, incluyendo:
- Limpieza y normalización
- Expansión de abreviaturas médicas
- Detección de negaciones
- Tokenización especializada

In [ ]:
# Cargar diccionario de abreviaturas médicas en español
# En un caso real, este diccionario tendría miles de entradas
MEDICAL_ABBREVIATIONS = {
    "hta": "hipertensión arterial",
    "dm": "diabetes mellitus",
    "dm2": "diabetes mellitus tipo 2",
    "ecg": "electrocardiograma",
    "fa": "fibrilación auricular",
    "icc": "insuficiencia cardíaca congestiva",
    "tto": "tratamiento",
    "epoc": "enfermedad pulmonar obstructiva crónica",
    "iam": "infarto agudo de miocardio",
    "acv": "accidente cerebrovascular",
    "htp": "hipertensión pulmonar",
    "erc": "enfermedad renal crónica",
    "tas": "tensión arterial sistólica",
    "tad": "tensión arterial diastólica",
    "fc": "frecuencia cardíaca",
    "fr": "frecuencia respiratoria",
    "rx": "radiografía",
    "tac": "tomografía axial computarizada",
    "tc": "tomografía computarizada",
    "rm": "resonancia magnética",
    "uci": "unidad de cuidados intensivos",
    "ah": "antecedentes heredofamiliares",
    "ap": "antecedentes personales",
    "app": "antecedentes personales patológicos",
    "ekg": "electrocardiograma",
    "pa": "presión arterial",
    "kg": "kilogramos",
    "gr": "gramos",
    "mg": "miligramos",
    "mmhg": "milímetros de mercurio"
}

# Lista de términos de negación en español
NEGATION_TERMS = [
    "no", "sin", "ausencia", "ausente", "negativo", "niega", "descarta",
    "libre de", "descartar", "se descarta", "se niega", "nunca",
    "excluye", "descartar", "excluir"
]

def clean_text(text):
    """Realiza limpieza básica de un texto médico."""
    if not isinstance(text, str):
        return ""

    # Convertir a minúsculas
    text = text.lower()

    # Normalizar espacios
    text = re.sub(r'\s+', ' ', text)

    # Eliminar caracteres no deseados pero mantener puntuación importante
    text = re.sub(r'[^\w\s\.,;:\-\(\)\/\+%]', '', text)

    # Normalizar abreviaturas comunes con puntos
    text = re.sub(r'dr\.', 'doctor', text)
    text = re.sub(r'dra\.', 'doctora', text)

    return text.strip()

def expand_abbreviations(text, abbrev_dict=MEDICAL_ABBREVIATIONS):
    """Expande abreviaturas médicas comunes en el texto."""
    if not isinstance(text, str):
        return ""

    words = text.split()
    result = []

    for word in words:
        # Eliminar puntuación para la búsqueda
        clean_word = re.sub(r'[^\w]', '', word).lower()

        if clean_word in abbrev_dict:
            # Mantener la puntuación original
            expansion = abbrev_dict[clean_word]
            # Si la palabra original tenía mayúsculas, conservar el formato
            if word.isupper():
                expansion = expansion.upper()
            elif word[0].isupper():
                expansion = expansion.capitalize()

            # Añadir la expansión entre paréntesis
            result.append(f"{word} ({expansion})")
        else:
            result.append(word)

    return ' '.join(result)

def mark_negations(text, window_size=5, negation_terms=NEGATION_TERMS):
    """
    Marca términos negados en el texto utilizando un enfoque simplificado.

    Args:
        text: Texto a procesar
        window_size: Número de palabras a considerar tras un término de negación
        negation_terms: Lista de términos que indican negación

    Returns:
        Texto con las negaciones marcadas con [NEG]término[/NEG]
    """
    if not isinstance(text, str):
        return ""

    sentences = sent_tokenize(text)
    processed_sentences = []

    for sentence in sentences:
        words = word_tokenize(sentence)
        negation_active = False
        negation_counter = 0

        for i in range(len(words)):
            word = words[i].lower()

            # Verificar si es un término de negación
            if word in negation_terms:
                negation_active = True
                negation_counter = 0
                continue

            # Marcar la palabra si está bajo el efecto de una negación
            if negation_active:
                if re.match(r'^[\.,;:]$', word):  # Si es un signo de puntuación
                    negation_active = False
                else:
                    words[i] = f"[NEG]{words[i]}[/NEG]"
                    negation_counter += 1

                    # Desactivar negación después de window_size palabras
                    if negation_counter >= window_size:
                        negation_active = False

        processed_sentences.append(' '.join(words))

    return ' '.join(processed_sentences)

def preprocess_medical_text(text, expand_abbrev=True, mark_neg=True):
    """Pipeline completo de preprocesamiento para textos médicos."""
    if not isinstance(text, str):
        return ""

    # Limpieza básica
    text = clean_text(text)

    # Expansión de abreviaturas
    if expand_abbrev:
        text = expand_abbreviations(text)

    # Marcado de negaciones
    if mark_neg:
        text = mark_negations(text)

    return text

# Demostrar el preprocesamiento con un ejemplo
if 'train_df' in locals():
    example_text = train_df['text'].iloc[0]

    print("🔍 Ejemplo de preprocesamiento de texto médico\n")
    print("Texto original (primeros 300 caracteres):")
    print("-" * 80)
    print(example_text[:300] + "..." if len(example_text) > 300 else example_text)
    print("-" * 80)

    # Aplicar preprocesamiento paso a paso
    cleaned_text = clean_text(example_text)
    print("\n1️⃣ Texto después de limpieza básica:")
    print("-" * 80)
    print(cleaned_text[:300] + "..." if len(cleaned_text) > 300 else cleaned_text)
    print("-" * 80)

    expanded_text = expand_abbreviations(cleaned_text)
    print("\n2️⃣ Texto después de expansión de abreviaturas:")
    print("-" * 80)
    print(expanded_text[:300] + "..." if len(expanded_text) > 300 else expanded_text)
    print("-" * 80)

    negated_text = mark_negations(expanded_text)
    print("\n3️⃣ Texto después de marcar negaciones:")
    print("-" * 80)
    print(negated_text[:300] + "..." if len(negated_text) > 300 else negated_text)
    print("-" * 80)

    # Pipeline completo
    fully_processed = preprocess_medical_text(example_text)
    print("\n✅ Texto completamente procesado:")
    print("-" * 80)
    print(fully_processed[:300] + "..." if len(fully_processed) > 300 else fully_processed)
    print("-" * 80)

### 3.1 Tokenización y vectorización

En esta sección implementaremos técnicas de tokenización y vectorización para convertir el texto preprocesado en representaciones numéricas que puedan ser utilizadas por los modelos de aprendizaje automático.

Utilizaremos varios enfoques:
1. **Bag of Words (BoW)**: Representación simple basada en frecuencia de palabras
2. **TF-IDF**: Ponderación de términos basada en su importancia relativa en el corpus
3. **Embeddings de Word2Vec**: Representaciones distribuidas preentrenadas
4. **Tokenización con BERT**: Preparación de textos para modelos basados en transformers

Estos métodos nos permitirán comparar diferentes representaciones y su impacto en el rendimiento de los modelos.

In [ ]:
# Implementación de diferentes métodos de vectorización

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec, KeyedVectors
import numpy as np
from nltk.tokenize import word_tokenize
from transformers import AutoTokenizer

# Función para generar vectores de características usando diferentes métodos
def create_text_representations(texts, method='tfidf', **kwargs):
    """
    Crea representaciones vectoriales de textos usando diferentes métodos.

    Args:
        texts: Lista de textos a vectorizar
        method: Método de vectorización ('bow', 'tfidf', 'word2vec', 'bert')
        **kwargs: Argumentos específicos para cada método

    Returns:
        Representación vectorial de los textos y el vectorizador utilizado
    """
    if method == 'bow':
        # Bag of Words
        max_features = kwargs.get('max_features', 5000)
        ngram_range = kwargs.get('ngram_range', (1, 2))

        vectorizer = CountVectorizer(max_features=max_features,
                                    ngram_range=ngram_range,
                                    stop_words=None)  # No eliminamos stop words en textos médicos

        X = vectorizer.fit_transform(texts)
        return X, vectorizer

    elif method == 'tfidf':
        # TF-IDF
        max_features = kwargs.get('max_features', 5000)
        ngram_range = kwargs.get('ngram_range', (1, 2))

        vectorizer = TfidfVectorizer(max_features=max_features,
                                    ngram_range=ngram_range,
                                    stop_words=None,
                                    use_idf=True,
                                    norm='l2')

        X = vectorizer.fit_transform(texts)
        return X, vectorizer

    elif method == 'word2vec':
        # Word2Vec embeddings (promedio de vectores de palabras)
        # En una implementación real, cargaríamos un modelo preentrenado médico en español
        # o entrenaríamos uno con nuestros datos

        # Tokenizar textos
        tokenized_texts = [word_tokenize(text.lower()) for text in texts]

        # Parámetros para el modelo Word2Vec
        vector_size = kwargs.get('vector_size', 100)
        window = kwargs.get('window', 5)
        min_count = kwargs.get('min_count', 1)

        # Entrenar modelo (en una implementación real usaríamos uno preentrenado)
        w2v_model = Word2Vec(sentences=tokenized_texts,
                           vector_size=vector_size,
                           window=window,
                           min_count=min_count,
                           workers=4)

        # Crear vectores promedio para cada documento
        doc_vectors = []
        for tokens in tokenized_texts:
            # Filtrar tokens que están en el vocabulario
            valid_tokens = [token for token in tokens if token in w2v_model.wv]
            if valid_tokens:
                # Calcular vector promedio
                doc_vector = np.mean([w2v_model.wv[token] for token in valid_tokens], axis=0)
            else:
                # Vector de ceros para documentos sin tokens válidos
                doc_vector = np.zeros(vector_size)

            doc_vectors.append(doc_vector)

        X = np.vstack(doc_vectors)
        return X, w2v_model

    elif method == 'bert':
        # Tokenización para BERT (no vectoriza completamente, solo prepara para el modelo)
        bert_model_name = kwargs.get('model_name', 'dccuchile/bert-base-spanish-wwm-cased')
        max_length = kwargs.get('max_length', 128)

        tokenizer = AutoTokenizer.from_pretrained(bert_model_name)

        # Tokenizar textos (sin padding - lo aplicaremos al crear los batches)
        encodings = tokenizer(
            texts,
            truncation=True,
            max_length=max_length,
            return_tensors=None  # Retorna listas de IDs en lugar de tensores
        )

        return encodings, tokenizer

    else:
        raise ValueError(f"Método de vectorización '{method}' no soportado")

# Demostración de los diferentes métodos de vectorización
if 'train_df' in locals():
    # Seleccionar una pequeña muestra de textos para la demostración
    sample_texts = train_df['text'].head(5).tolist()

    # Aplicar preprocesamiento a los textos de muestra
    processed_texts = [preprocess_medical_text(text) for text in sample_texts]

    print("🔢 Demostración de diferentes métodos de vectorización\n")

    # 1. Bag of Words
    bow_vectors, bow_vectorizer = create_text_representations(processed_texts, method='bow', max_features=1000)
    print("1️⃣ Bag of Words:")
    print(f"- Forma de la matriz: {bow_vectors.shape}")
    print(f"- Tipo de dato: {type(bow_vectors)}")
    print(f"- Densidad: {bow_vectors.nnz / (bow_vectors.shape[0] * bow_vectors.shape[1]):.4f}")
    print(f"- Top 10 términos: {[term for term, idx in sorted(bow_vectorizer.vocabulary_.items(), key=lambda x: x[1])[:10]]}")
    print()

    # 2. TF-IDF
    tfidf_vectors, tfidf_vectorizer = create_text_representations(processed_texts, method='tfidf', max_features=1000)
    print("2️⃣ TF-IDF:")
    print(f"- Forma de la matriz: {tfidf_vectors.shape}")
    print(f"- Tipo de dato: {type(tfidf_vectors)}")
    print(f"- Densidad: {tfidf_vectors.nnz / (tfidf_vectors.shape[0] * tfidf_vectors.shape[1]):.4f}")
    print(f"- Top 10 términos: {[term for term, idx in sorted(tfidf_vectorizer.vocabulary_.items(), key=lambda x: x[1])[:10]]}")
    print()

    # 3. Word2Vec
    try:
        w2v_vectors, w2v_model = create_text_representations(processed_texts, method='word2vec', vector_size=50)
        print("3️⃣ Word2Vec (promedio de vectores de palabras):")
        print(f"- Forma de la matriz: {w2v_vectors.shape}")
        print(f"- Tipo de dato: {type(w2v_vectors)}")
        print(f"- Tamaño del vocabulario: {len(w2v_model.wv)}")
        print(f"- Dimensión de los vectores: {w2v_model.vector_size}")
        print()
    except Exception as e:
        print(f"Error al crear representaciones Word2Vec: {e}")
        print()

    # 4. BERT Tokenizer
    try:
        bert_encodings, bert_tokenizer = create_text_representations(processed_texts, method='bert')
        print("4️⃣ Tokenización BERT:")
        print(f"- Tamaño del vocabulario del tokenizer: {bert_tokenizer.vocab_size}")
        print("- Ejemplo de tokenización para un texto:")
        sample_idx = 0
        print(f"  Texto original: {processed_texts[sample_idx][:50]}...")
        print(f"  Tokens: {bert_tokenizer.tokenize(processed_texts[sample_idx])[:15]}...")
        print(f"  IDs: {bert_encodings['input_ids'][sample_idx][:15]}...")
    except Exception as e:
        print(f"Error al crear tokenización BERT: {e}")
        print("Nota: La tokenización BERT requiere conexión a Internet para descargar el modelo.")

## 4. Codificación Jerárquica de Etiquetas CIE-10

Los códigos CIE-10 tienen una estructura jerárquica natural. Por ejemplo:
- **C00-D49**: Neoplasias
  - **C00-C97**: Tumores malignos
    - **C00-C14**: Tumores malignos del labio, cavidad oral y faringe
      - **C00**: Tumor maligno del labio
        - **C00.0**: Tumor maligno del labio superior, cara externa

Esta estructura jerárquica es fundamental para nuestro enfoque de clasificación multinivel, donde primero predecimos categorías generales y luego refinamos hacia subcategorías específicas. 

En esta sección:
1. Extraeremos la estructura jerárquica de los códigos CIE-10
2. Crearemos múltiples niveles de etiquetas (padre, hijo, nieto)
3. Analizaremos la distribución de etiquetas en cada nivel jerárquico
4. Prepararemos las matrices de etiquetas para clasificación multinivel

In [ ]:
# Procesamiento de etiquetas jerárquicas CIE-10

from sklearn.preprocessing import MultiLabelBinarizer
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def extract_hierarchical_levels(code):
    """
    Extrae los diferentes niveles jerárquicos de un código CIE-10.

    Args:
        code: Código CIE-10 (ej: "C50.2")

    Returns:
        Tupla con niveles jerárquicos (nivel1, nivel2, nivel3)
        - nivel1: Categoría general (ej: "C")
        - nivel2: Subcategoría (ej: "C50")
        - nivel3: Código completo (ej: "C50.2")
    """
    if not isinstance(code, str):
        return None, None, None

    # Nivel 1: Letra inicial (A-Z)
    match_level1 = re.match(r'^([A-Za-z])', code)
    level1 = match_level1.group(1).upper() if match_level1 else None

    # Nivel 2: Letra + números antes del punto (ej: C50)
    match_level2 = re.match(r'^([A-Za-z]\d{1,2})', code)
    level2 = match_level2.group(1).upper() if match_level2 else None

    # Nivel 3: Código completo
    level3 = code.upper()

    return level1, level2, level3

def create_hierarchical_labels(df, code_column='codes'):
    """
    Crea estructuras de etiquetas jerárquicas a partir de códigos CIE-10.

    Args:
        df: DataFrame con columna de códigos (en formato lista o string separado)
        code_column: Nombre de la columna con los códigos

    Returns:
        DataFrame con nuevas columnas de etiquetas por nivel
    """
    # Hacer una copia para no modificar el original
    result_df = df.copy()

    # Asegurarse de que los códigos están en formato de lista
    if result_df[code_column].dtype == object and isinstance(result_df[code_column].iloc[0], str):
        # Convertir strings separados por comas a listas
        result_df[code_column] = result_df[code_column].apply(lambda x: x.split(',') if isinstance(x, str) else [])

    # Extraer etiquetas para cada nivel jerárquico
    result_df['level1_codes'] = result_df[code_column].apply(
        lambda codes: list(set([extract_hierarchical_levels(code)[0] for code in codes if extract_hierarchical_levels(code)[0]]))
    )

    result_df['level2_codes'] = result_df[code_column].apply(
        lambda codes: list(set([extract_hierarchical_levels(code)[1] for code in codes if extract_hierarchical_levels(code)[1]]))
    )

    result_df['level3_codes'] = result_df[code_column].apply(
        lambda codes: list(set([extract_hierarchical_levels(code)[2] for code in codes if extract_hierarchical_levels(code)[2]]))
    )

    return result_df

def create_multilabel_matrices(df):
    """
    Crea matrices binarias multinivel para clasificación jerárquica.

    Args:
        df: DataFrame con columnas level1_codes, level2_codes y level3_codes

    Returns:
        Diccionario con matrices binarias y binarizadores para cada nivel
    """
    # Crear binarizadores para cada nivel
    mlb_level1 = MultiLabelBinarizer()
    mlb_level2 = MultiLabelBinarizer()
    mlb_level3 = MultiLabelBinarizer()

    # Transformar etiquetas a matrices binarias
    y_level1 = mlb_level1.fit_transform(df['level1_codes'])
    y_level2 = mlb_level2.fit_transform(df['level2_codes'])
    y_level3 = mlb_level3.fit_transform(df['level3_codes'])

    return {
        'level1': {'matrix': y_level1, 'binarizer': mlb_level1, 'classes': mlb_level1.classes_},
        'level2': {'matrix': y_level2, 'binarizer': mlb_level2, 'classes': mlb_level2.classes_},
        'level3': {'matrix': y_level3, 'binarizer': mlb_level3, 'classes': mlb_level3.classes_}
    }

def analyze_label_distribution(label_data):
    """
    Analiza la distribución de etiquetas en cada nivel jerárquico.

    Args:
        label_data: Diccionario con datos de etiquetas por nivel

    Returns:
        Estadísticas sobre la distribución de etiquetas
    """
    stats = {}

    for level, data in label_data.items():
        y = data['matrix']
        classes = data['classes']

        # Número de etiquetas distintas
        num_labels = len(classes)

        # Frecuencia de cada etiqueta
        label_counts = np.sum(y, axis=0)

        # Etiquetas más comunes
        top_indices = np.argsort(-label_counts)[:10]
        top_labels = [(classes[i], label_counts[i]) for i in top_indices]

        # Promedio de etiquetas por instancia
        avg_labels_per_instance = np.mean(np.sum(y, axis=1))

        # Máximo de etiquetas por instancia
        max_labels_per_instance = np.max(np.sum(y, axis=1))

        # Documentos con múltiples etiquetas (%)
        multi_label_docs = np.mean(np.sum(y, axis=1) > 1) * 100

        stats[level] = {
            'num_labels': num_labels,
            'label_counts': label_counts,
            'top_labels': top_labels,
            'avg_labels_per_instance': avg_labels_per_instance,
            'max_labels_per_instance': max_labels_per_instance,
            'multi_label_docs_percent': multi_label_docs
        }

    return stats

# Aplicar procesamiento de etiquetas jerárquicas si el conjunto de datos está disponible
if 'train_df' in locals() and 'codes' in train_df.columns:
    print("🏷️ Procesamiento de etiquetas jerárquicas CIE-10\n")

    # Crear estructura jerárquica de etiquetas
    train_df = create_hierarchical_labels(train_df, code_column='codes')

    # Mostrar algunos ejemplos
    print("Ejemplo de etiquetas jerárquicas para un documento:")
    sample_idx = 0
    print(f"Texto: {train_df['text'].iloc[sample_idx][:100]}...")
    print(f"Códigos originales: {train_df['codes'].iloc[sample_idx]}")
    print(f"Nivel 1 (Categorías): {train_df['level1_codes'].iloc[sample_idx]}")
    print(f"Nivel 2 (Subcategorías): {train_df['level2_codes'].iloc[sample_idx]}")
    print(f"Nivel 3 (Códigos completos): {train_df['level3_codes'].iloc[sample_idx]}\n")

    # Crear matrices multinivel
    label_matrices = create_multilabel_matrices(train_df)

    # Mostrar dimensiones de las matrices
    print("Dimensiones de las matrices de etiquetas:")
    for level, data in label_matrices.items():
        print(f"- {level}: {data['matrix'].shape} ({len(data['classes'])} clases únicas)")

    # Analizar distribución de etiquetas
    stats = analyze_label_distribution(label_matrices)

    # Mostrar estadísticas
    print("\nEstadísticas de etiquetas por nivel:")
    for level, data in stats.items():
        print(f"\n{level.upper()}:")
        print(f"- Número de etiquetas distintas: {data['num_labels']}")
        print(f"- Promedio de etiquetas por documento: {data['avg_labels_per_instance']:.2f}")
        print(f"- Máximo de etiquetas por documento: {data['max_labels_per_instance']}")
        print(f"- Documentos con múltiples etiquetas: {data['multi_label_docs_percent']:.2f}%")
        print(f"- Top 5 etiquetas más frecuentes:")
        for label, count in data['top_labels'][:5]:
            print(f"  - {label}: {count} ocurrencias")

    # Visualizar distribución de las etiquetas más comunes
    plt.figure(figsize=(15, 12))

    for i, level in enumerate(['level1', 'level2', 'level3']):
        plt.subplot(3, 1, i+1)

        # Obtener las 10 etiquetas más comunes
        top_indices = np.argsort(-stats[level]['label_counts'])[:15]
        labels = [label_matrices[level]['classes'][i] for i in top_indices]
        counts = [stats[level]['label_counts'][i] for i in top_indices]

        # Crear gráfico de barras
        sns.barplot(x=counts, y=labels, palette='viridis')
        plt.title(f'Top 15 etiquetas más frecuentes - Nivel {i+1}')
        plt.xlabel('Número de documentos')
        plt.ylabel('Código CIE-10')
        plt.tight_layout()

    plt.subplots_adjust(hspace=0.4)
    plt.suptitle('Distribución de Etiquetas CIE-10 por Nivel Jerárquico', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No se encontraron datos para procesar etiquetas jerárquicas CIE-10.")

## 5. Implementación de Modelos de Clasificación

En esta sección implementaremos diferentes modelos para la clasificación multi-etiqueta jerárquica de diagnósticos CIE-10. Utilizaremos un enfoque progresivo, comenzando con modelos más simples y avanzando hacia arquitecturas más sofisticadas:

1. **Modelos Baseline**:
   - Clasificadores tradicionales adaptados para multi-etiqueta
   - One-vs-Rest con SVM, Random Forest y Regresión Logística

2. **Modelos de Redes Neuronales**:
   - BiLSTM con atención
   - CNN para clasificación de texto

3. **Modelos Jerárquicos**:
   - Clasificadores en cascada (padres → hijos → nietos)
   - Arquitectura con pérdida jerárquica personalizada

4. **Modelos basados en Transformers**:
   - Fine-tuning de BERT para español médico
   - Arquitectura jerárquica con BERT

Para cada modelo evaluaremos su rendimiento con métricas específicas para clasificación multi-etiqueta y jerárquica.

### 5.1 Modelos Baseline para Clasificación Multi-etiqueta

Comenzaremos implementando modelos clásicos adaptados para la clasificación multi-etiqueta mediante la estrategia One-vs-Rest (OvR). Esta estrategia convierte un problema de clasificación multi-etiqueta en múltiples problemas de clasificación binaria (uno por cada etiqueta posible).

Los modelos baseline que implementaremos son:
- **Regresión Logística**: Un clasificador lineal simple y rápido
- **SVM**: Máquinas de vectores de soporte con kernel lineal, eficiente para textos
- **Random Forest**: Conjunto de árboles de decisión, adecuado para capturar relaciones no lineales

In [ ]:
# Implementación de modelos baseline para clasificación multi-etiqueta

from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
from sklearn.metrics import accuracy_score, hamming_loss
import time
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

class MultiLabelClassifier:
    """Clase para encapsular modelos de clasificación multi-etiqueta."""

    def __init__(self, model_name, classifier, vectorizer_method='tfidf', level='level1'):
        """
        Inicializa un clasificador multi-etiqueta.

        Args:
            model_name: Nombre identificativo del modelo
            classifier: Clasificador scikit-learn a utilizar con OneVsRestClassifier
            vectorizer_method: Método de vectorización de textos ('bow', 'tfidf')
            level: Nivel jerárquico para clasificar ('level1', 'level2', 'level3')
        """
        self.model_name = model_name
        self.classifier = OneVsRestClassifier(classifier, n_jobs=-1)
        self.vectorizer_method = vectorizer_method
        self.vectorizer = None
        self.level = level
        self.training_time = None
        self.is_trained = False

    def train(self, X_texts, y, verbose=True):
        """
        Entrena el modelo con los datos proporcionados.

        Args:
            X_texts: Textos de entrenamiento
            y: Etiquetas multi-clase (matriz binaria)
            verbose: Si True, muestra información durante el entrenamiento

        Returns:
            Self para encadenamiento de métodos
        """
        if verbose:
            print(f"Entrenando modelo {self.model_name} para nivel {self.level}...")
            print(f"- Vectorizando textos usando {self.vectorizer_method}...")

        # Vectorizar textos
        start_time = time.time()
        X_vectors, self.vectorizer = create_text_representations(X_texts, method=self.vectorizer_method)

        if verbose:
            print(f"- Forma de matriz de características: {X_vectors.shape}")
            print(f"- Entrenando clasificador {self.classifier.__class__.__name__}...")

        # Entrenar clasificador
        self.classifier.fit(X_vectors, y)

        # Guardar tiempo de entrenamiento
        self.training_time = time.time() - start_time
        self.is_trained = True

        if verbose:
            print(f"✅ Entrenamiento completado en {self.training_time:.2f} segundos")

        return self

    def predict(self, X_texts, threshold=0.5):
        """
        Realiza predicciones sobre nuevos textos.

        Args:
            X_texts: Textos a clasificar
            threshold: Umbral de decisión para las etiquetas (para predict_proba)

        Returns:
            Matriz binaria con las etiquetas predichas
        """
        if not self.is_trained:
            raise ValueError("El modelo debe ser entrenado antes de hacer predicciones")

        # Vectorizar textos
        X_vectors, _ = create_text_representations(X_texts, method=self.vectorizer_method,
                                                 vectorizer=self.vectorizer)

        # Predecir
        try:
            # Intentar con predict_proba si está disponible
            y_pred_proba = self.classifier.predict_proba(X_vectors)
            y_pred = (y_pred_proba >= threshold).astype(int)
        except:
            # Usar predict normal si predict_proba no está disponible
            y_pred = self.classifier.predict(X_vectors)

        return y_pred

    def evaluate(self, X_texts, y_true, threshold=0.5):
        """
        Evalúa el rendimiento del modelo.

        Args:
            X_texts: Textos a evaluar
            y_true: Etiquetas verdaderas
            threshold: Umbral de decisión para las etiquetas

        Returns:
            Diccionario con métricas de evaluación
        """
        # Hacer predicciones
        y_pred = self.predict(X_texts, threshold)

        # Calcular métricas
        metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'hamming_loss': hamming_loss(y_true, y_pred),
            'precision_micro': precision_score(y_true, y_pred, average='micro'),
            'recall_micro': recall_score(y_true, y_pred, average='micro'),
            'f1_micro': f1_score(y_true, y_pred, average='micro'),
            'precision_macro': precision_score(y_true, y_pred, average='macro'),
            'recall_macro': recall_score(y_true, y_pred, average='macro'),
            'f1_macro': f1_score(y_true, y_pred, average='macro'),
            'precision_weighted': precision_score(y_true, y_pred, average='weighted'),
            'recall_weighted': recall_score(y_true, y_pred, average='weighted'),
            'f1_weighted': f1_score(y_true, y_pred, average='weighted'),
        }

        return metrics

    def save(self, directory="models"):
        """
        Guarda el modelo entrenado y el vectorizador en disco.

        Args:
            directory: Directorio donde guardar los archivos
        """
        if not self.is_trained:
            raise ValueError("El modelo debe ser entrenado antes de guardarlo")

        # Crear directorio si no existe
        os.makedirs(directory, exist_ok=True)

        # Guardar modelo y vectorizador
        model_path = os.path.join(directory, f"{self.model_name}_{self.level}.pkl")
        vectorizer_path = os.path.join(directory, f"{self.model_name}_{self.level}_vectorizer.pkl")

        with open(model_path, 'wb') as f:
            pickle.dump(self.classifier, f)

        with open(vectorizer_path, 'wb') as f:
            pickle.dump(self.vectorizer, f)

        print(f"Modelo guardado en {model_path}")
        print(f"Vectorizador guardado en {vectorizer_path}")

    def load(self, directory="models"):
        """
        Carga el modelo entrenado y el vectorizador desde disco.

        Args:
            directory: Directorio donde se encuentran los archivos
        """
        model_path = os.path.join(directory, f"{self.model_name}_{self.level}.pkl")
        vectorizer_path = os.path.join(directory, f"{self.model_name}_{self.level}_vectorizer.pkl")

        with open(model_path, 'rb') as f:
            self.classifier = pickle.load(f)

        with open(vectorizer_path, 'rb') as f:
            self.vectorizer = pickle.load(f)

        self.is_trained = True
        print(f"Modelo cargado desde {model_path}")
        print(f"Vectorizador cargado desde {vectorizer_path}")

def train_and_evaluate_baseline_models(X_train, X_test, y_train, y_test, level='level1'):
    """
    Entrena y evalúa múltiples modelos baseline para comparación.

    Args:
        X_train: Textos de entrenamiento
        X_test: Textos de prueba
        y_train: Etiquetas de entrenamiento
        y_test: Etiquetas de prueba
        level: Nivel jerárquico a clasificar

    Returns:
        Diccionario con resultados y modelos entrenados
    """
    # Definir modelos a entrenar
    models = [
        MultiLabelClassifier(
            model_name="logistic_regression",
            classifier=LogisticRegression(C=1.0, solver='liblinear'),
            vectorizer_method='tfidf',
            level=level
        ),
        MultiLabelClassifier(
            model_name="linear_svc",
            classifier=LinearSVC(C=1.0),
            vectorizer_method='tfidf',
            level=level
        ),
        MultiLabelClassifier(
            model_name="random_forest",
            classifier=RandomForestClassifier(n_estimators=100, max_depth=None, min_samples_split=2),
            vectorizer_method='tfidf',
            level=level
        )
    ]

    # Resultados
    results = {}

    # Entrenar y evaluar cada modelo
    for model in models:
        print(f"\n{'='*80}")
        print(f"Modelo: {model.model_name.upper()} - Nivel: {level}")
        print(f"{'='*80}")

        # Entrenar
        model.train(X_train, y_train)

        # Evaluar
        metrics = model.evaluate(X_test, y_test)

        # Guardar resultados
        results[model.model_name] = {
            'model': model,
            'metrics': metrics,
            'training_time': model.training_time
        }

        # Mostrar métricas
        print("\nMétricas de evaluación:")
        print(f"- Accuracy: {metrics['accuracy']:.4f}")
        print(f"- Hamming Loss: {metrics['hamming_loss']:.4f}")
        print(f"- F1 Score (micro): {metrics['f1_micro']:.4f}")
        print(f"- F1 Score (macro): {metrics['f1_macro']:.4f}")
        print(f"- F1 Score (weighted): {metrics['f1_weighted']:.4f}")
        print(f"- Precision (micro): {metrics['precision_micro']:.4f}")
        print(f"- Recall (micro): {metrics['recall_micro']:.4f}")

    return results

# Código para entrenar y evaluar modelos si los datos están disponibles
if 'train_df' in locals() and 'label_matrices' in locals():
    print("🔬 Entrenamiento de modelos baseline para clasificación multi-etiqueta\n")

    # Preprocesar textos de entrenamiento
    processed_texts = train_df['text'].apply(preprocess_medical_text).tolist()

    # Dividir datos en entrenamiento y prueba
    X_train, X_test, y_train_level1, y_test_level1 = train_test_split(
        processed_texts,
        label_matrices['level1']['matrix'],
        test_size=0.2,
        random_state=42
    )

    print(f"Datos de entrenamiento: {len(X_train)} documentos")
    print(f"Datos de prueba: {len(X_test)} documentos")
    print(f"Dimensión etiquetas (nivel 1): {y_train_level1.shape}")

    # Entrenar y evaluar modelos para el nivel 1
    try:
        print("\nEntrenando modelos para el nivel 1 (categorías principales)...")
        level1_results = train_and_evaluate_baseline_models(
            X_train, X_test, y_train_level1, y_test_level1, level='level1'
        )

        # Visualizar comparación de modelos
        models = list(level1_results.keys())
        metrics = ['accuracy', 'f1_micro', 'f1_macro', 'precision_micro', 'recall_micro']

        plt.figure(figsize=(12, 6))
        bar_width = 0.2
        index = np.arange(len(metrics))

        for i, model_name in enumerate(models):
            values = [level1_results[model_name]['metrics'][m] for m in metrics]
            plt.bar(index + i * bar_width, values, bar_width, label=model_name)

        plt.xlabel('Métricas')
        plt.ylabel('Puntuación')
        plt.title('Comparación de modelos baseline - Nivel 1')
        plt.xticks(index + bar_width, metrics)
        plt.legend()
        plt.tight_layout()
        plt.show()

        # Comparar tiempo de entrenamiento
        train_times = [level1_results[model_name]['training_time'] for model_name in models]

        plt.figure(figsize=(10, 4))
        plt.bar(models, train_times)
        plt.xlabel('Modelo')
        plt.ylabel('Tiempo de entrenamiento (segundos)')
        plt.title('Tiempo de entrenamiento por modelo')
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Error al entrenar y evaluar modelos: {e}")
else:
    print("⚠️ No se encontraron datos para entrenar modelos.")

### 5.2 Modelo BiLSTM con Atención

A continuación implementaremos un modelo de red neuronal recurrente bidireccional (BiLSTM) con mecanismo de atención. Esta arquitectura es especialmente adecuada para tareas de clasificación de texto porque:

1. Las capas BiLSTM capturan dependencias de largo alcance y relaciones contextuales en ambas direcciones del texto
2. El mecanismo de atención permite al modelo concentrarse en las partes más relevantes del texto para cada etiqueta
3. Es más eficaz que los modelos tradicionales para manejar la complejidad semántica de textos médicos

Esta implementación utilizará PyTorch como framework de deep learning.

In [ ]:
# Implementación de modelo BiLSTM con atención para clasificación multi-etiqueta

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import numpy as np
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import os
import time
from tqdm.notebook import tqdm
import nltk
from nltk.tokenize import word_tokenize
import re

# Descargar tokenizador de NLTK si no está disponible
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

class TextPreprocessor:
    """Clase para preprocesar y tokenizar textos para el modelo BiLSTM."""

    def __init__(self, max_vocab_size=30000, max_seq_length=300):
        """
        Inicializa el preprocesador de textos.

        Args:
            max_vocab_size: Tamaño máximo del vocabulario
            max_seq_length: Longitud máxima de las secuencias
        """
        self.max_vocab_size = max_vocab_size
        self.max_seq_length = max_seq_length
        self.word2idx = {}
        self.idx2word = {}
        self.word_counts = Counter()
        self.vocab_size = 0

    def build_vocab(self, texts):
        """
        Construye el vocabulario a partir de un conjunto de textos.

        Args:
            texts: Lista de textos a procesar
        """
        # Tokenizar todos los textos
        all_tokens = []
        for text in tqdm(texts, desc="Tokenizando textos"):
            tokens = word_tokenize(text.lower())
            all_tokens.extend(tokens)
            self.word_counts.update(tokens)

        # Construir vocabulario con las palabras más frecuentes
        vocab_words = ['<PAD>', '<UNK>'] + [word for word, _ in self.word_counts.most_common(self.max_vocab_size - 2)]

        # Crear mapeos palabra -> índice e índice -> palabra
        self.word2idx = {word: idx for idx, word in enumerate(vocab_words)}
        self.idx2word = {idx: word for word, idx in self.word2idx.items()}
        self.vocab_size = len(self.word2idx)

        print(f"Vocabulario construido con {self.vocab_size} palabras")

    def text_to_sequence(self, text):
        """
        Convierte un texto en una secuencia de índices.

        Args:
            text: Texto a convertir

        Returns:
            Lista de índices y longitud de la secuencia
        """
        tokens = word_tokenize(text.lower())

        # Truncar o rellenar a max_seq_length
        if len(tokens) > self.max_seq_length:
            tokens = tokens[:self.max_seq_length]

        # Convertir tokens a índices
        sequence = [self.word2idx.get(token, self.word2idx['<UNK>']) for token in tokens]
        length = len(sequence)

        # Rellenar con PAD si es necesario
        if length < self.max_seq_length:
            sequence += [self.word2idx['<PAD>']] * (self.max_seq_length - length)

        return sequence, length

    def preprocess_text(self, text):
        """Preprocesa un único texto."""
        if not isinstance(text, str):
            text = str(text)

        # Limpieza básica
        text = text.lower()
        text = re.sub(r'\s+', ' ', text)
        text = text.strip()

        return text

    def preprocess_batch(self, texts):
        """Preprocesa un lote de textos."""
        return [self.preprocess_text(text) for text in texts]

class TextDataset(Dataset):
    """Dataset para textos y etiquetas multi-clase."""

    def __init__(self, texts, labels, preprocessor):
        """
        Inicializa el dataset.

        Args:
            texts: Lista de textos
            labels: Matriz binaria de etiquetas
            preprocessor: Instancia de TextPreprocessor
        """
        self.texts = texts
        self.labels = torch.FloatTensor(labels)
        self.preprocessor = preprocessor

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.preprocessor.preprocess_text(self.texts[idx])
        sequence, length = self.preprocessor.text_to_sequence(text)
        return {
            'text': torch.LongTensor(sequence),
            'length': length,
            'labels': self.labels[idx]
        }

def collate_fn(batch):
    """Función para agrupar elementos del batch."""
    texts = torch.stack([item['text'] for item in batch])
    lengths = torch.LongTensor([item['length'] for item in batch])
    labels = torch.stack([item['labels'] for item in batch])

    # Ordenar por longitud (mayor a menor)
    lengths, perm_idx = lengths.sort(0, descending=True)
    texts = texts[perm_idx]
    labels = labels[perm_idx]

    return {
        'texts': texts,
        'lengths': lengths,
        'labels': labels,
        'perm_idx': perm_idx
    }

class AttentionMechanism(nn.Module):
    """Mecanismo de atención para la red BiLSTM."""

    def __init__(self, hidden_size):
        super(AttentionMechanism, self).__init__()
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, lstm_output, mask=None):
        """
        Aplica atención a las salidas del LSTM.

        Args:
            lstm_output: Tensor de salidas del LSTM [batch_size, seq_len, hidden_size]
            mask: Máscara para tokens de padding

        Returns:
            Contexto ponderado con atención
        """
        # Calcular puntuaciones de atención
        attention_scores = self.attention(lstm_output).squeeze(-1)

        # Aplicar máscara si se proporciona
        if mask is not None:
            attention_scores = attention_scores.masked_fill(mask == 0, -1e9)

        # Aplicar softmax para obtener pesos de atención
        attention_weights = F.softmax(attention_scores, dim=1)

        # Calcular contexto ponderado
        context = torch.bmm(attention_weights.unsqueeze(1), lstm_output).squeeze(1)

        return context, attention_weights

class BiLSTMAttention(nn.Module):
    """Modelo BiLSTM con atención para clasificación multi-etiqueta."""

    def __init__(self, vocab_size, embed_size, hidden_size, num_classes, num_layers=1, dropout=0.5):
        """
        Inicializa el modelo.

        Args:
            vocab_size: Tamaño del vocabulario
            embed_size: Dimensión de los embeddings
            hidden_size: Dimensión del estado oculto del LSTM
            num_classes: Número de clases para clasificación
            num_layers: Número de capas LSTM
            dropout: Tasa de dropout
        """
        super(BiLSTMAttention, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_size,
            hidden_size,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.attention = AttentionMechanism(hidden_size * 2)  # *2 por bidireccional
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, texts, lengths):
        """
        Pasa una batch de textos por el modelo.

        Args:
            texts: Tensor de índices de palabras [batch_size, seq_len]
            lengths: Lista de longitudes reales de las secuencias

        Returns:
            Logits para cada clase
        """
        # Embeddings
        embedded = self.embedding(texts)  # [batch_size, seq_len, embed_size]

        # Empaquetar secuencias para ignorar padding en el LSTM
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=True)

        # LSTM bidireccional
        packed_output, _ = self.lstm(packed)
        lstm_output, _ = pad_packed_sequence(packed_output, batch_first=True)  # [batch_size, seq_len, hidden_size*2]

        # Crear máscara para tokens de padding
        mask = torch.zeros_like(texts, dtype=torch.bool)
        for i, length in enumerate(lengths):
            mask[i, :length] = 1

        # Aplicar mecanismo de atención
        context, attention_weights = self.attention(lstm_output, mask)

        # Aplicar dropout
        context = self.dropout(context)

        # Clasificación final
        logits = self.fc(context)

        return logits, attention_weights

def train_bilstm_model(model, train_loader, val_loader, criterion, optimizer, device,
                     num_epochs=5, patience=3, model_save_path='best_bilstm_model.pt'):
    """
    Entrena el modelo BiLSTM.

    Args:
        model: Modelo BiLSTM a entrenar
        train_loader: DataLoader con datos de entrenamiento
        val_loader: DataLoader con datos de validación
        criterion: Función de pérdida
        optimizer: Optimizador
        device: Dispositivo (CPU/GPU)
        num_epochs: Número de épocas
        patience: Paciencia para early stopping
        model_save_path: Ruta para guardar el mejor modelo

    Returns:
        Historial de entrenamiento
    """
    model.to(device)

    # Listas para almacenar métricas
    train_losses = []
    val_losses = []
    train_f1_scores = []
    val_f1_scores = []

    best_val_loss = float('inf')
    early_stopping_counter = 0

    for epoch in range(num_epochs):
        print(f"Época {epoch+1}/{num_epochs}")

        # ===== Entrenamiento =====
        model.train()
        train_loss = 0
        train_f1 = 0

        train_preds = []
        train_true = []

        for batch in tqdm(train_loader, desc="Entrenando"):
            optimizer.zero_grad()

            texts = batch['texts'].to(device)
            lengths = batch['lengths']
            labels = batch['labels'].to(device)

            # Forward pass
            logits, _ = model(texts, lengths)
            loss = criterion(logits, labels)

            # Backward pass
            loss.backward()
            optimizer.step()

            # Guardar pérdida
            train_loss += loss.item()

            # Convertir logits a predicciones binarias
            predictions = (torch.sigmoid(logits) > 0.5).float()

            # Guardar predicciones y etiquetas verdaderas para métrica F1
            train_preds.append(predictions.cpu().detach().numpy())
            train_true.append(labels.cpu().detach().numpy())

        # Calcular pérdida y F1 promedio del entrenamiento
        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # Calcular F1 micro
        train_preds_flat = np.vstack(train_preds)
        train_true_flat = np.vstack(train_true)
        train_f1 = f1_score(train_true_flat, train_preds_flat, average='micro')
        train_f1_scores.append(train_f1)

        # ===== Validación =====
        model.eval()
        val_loss = 0

        val_preds = []
        val_true = []

        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validando"):
                texts = batch['texts'].to(device)
                lengths = batch['lengths']
                labels = batch['labels'].to(device)

                # Forward pass
                logits, _ = model(texts, lengths)
                loss = criterion(logits, labels)

                # Guardar pérdida
                val_loss += loss.item()

                # Convertir logits a predicciones binarias
                predictions = (torch.sigmoid(logits) > 0.5).float()

                # Guardar predicciones y etiquetas verdaderas para métrica F1
                val_preds.append(predictions.cpu().numpy())
                val_true.append(labels.cpu().numpy())

        # Calcular pérdida y F1 promedio de la validación
        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        # Calcular F1 micro
        val_preds_flat = np.vstack(val_preds)
        val_true_flat = np.vstack(val_true)
        val_f1 = f1_score(val_true_flat, val_preds_flat, average='micro')
        val_f1_scores.append(val_f1)

        print(f"Train Loss: {train_loss:.4f}, Train F1: {train_f1:.4f}")
        print(f"Val Loss: {val_loss:.4f}, Val F1: {val_f1:.4f}")

        # Early stopping y guardado del mejor modelo
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            early_stopping_counter = 0

            # Guardar el mejor modelo
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'val_f1': val_f1
            }, model_save_path)

            print(f"Mejor modelo guardado con Val Loss: {val_loss:.4f}")
        else:
            early_stopping_counter += 1
            print(f"EarlyStopping counter: {early_stopping_counter} de {patience}")

        if early_stopping_counter >= patience:
            print("Early stopping activado")
            break

        print('-' * 60)

    # Retornar historial de entrenamiento
    history = {
        'train_loss': train_losses,
        'val_loss': val_losses,
        'train_f1': train_f1_scores,
        'val_f1': val_f1_scores
    }

    return history

def evaluate_bilstm(model, test_loader, criterion, device, label_classes=None):
    """
    Evalúa el modelo BiLSTM en el conjunto de prueba.

    Args:
        model: Modelo BiLSTM a evaluar
        test_loader: DataLoader con datos de prueba
        criterion: Función de pérdida
        device: Dispositivo (CPU/GPU)
        label_classes: Lista de nombres de clases

    Returns:
        Métricas de evaluación
    """
    model.eval()
    test_loss = 0

    all_preds = []
    all_true = []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluando"):
            texts = batch['texts'].to(device)
            lengths = batch['lengths']
            labels = batch['labels'].to(device)

            # Forward pass
            logits, _ = model(texts, lengths)
            loss = criterion(logits, labels)

            # Guardar pérdida
            test_loss += loss.item()

            # Convertir logits a predicciones binarias
            predictions = (torch.sigmoid(logits) > 0.5).float()

            # Guardar predicciones y etiquetas verdaderas para métricas
            all_preds.append(predictions.cpu().numpy())
            all_true.append(labels.cpu().numpy())

    # Calcular pérdida promedio
    test_loss /= len(test_loader)

    # Aplanar predicciones y etiquetas verdaderas
    all_preds_flat = np.vstack(all_preds)
    all_true_flat = np.vstack(all_true)

    # Calcular métricas
    metrics = {
        'test_loss': test_loss,
        'accuracy': accuracy_score(all_true_flat, all_preds_flat),
        'hamming_loss': hamming_loss(all_true_flat, all_preds_flat),
        'precision_micro': precision_score(all_true_flat, all_preds_flat, average='micro'),
        'recall_micro': recall_score(all_true_flat, all_preds_flat, average='micro'),
        'f1_micro': f1_score(all_true_flat, all_preds_flat, average='micro'),
        'precision_macro': precision_score(all_true_flat, all_preds_flat, average='macro'),
        'recall_macro': recall_score(all_true_flat, all_preds_flat, average='macro'),
        'f1_macro': f1_score(all_true_flat, all_preds_flat, average='macro'),
        'precision_weighted': precision_score(all_true_flat, all_preds_flat, average='weighted'),
        'recall_weighted': recall_score(all_true_flat, all_preds_flat, average='weighted'),
        'f1_weighted': f1_score(all_true_flat, all_preds_flat, average='weighted')
    }

    # Mostrar métricas
    print(f"Test Loss: {metrics['test_loss']:.4f}")
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"Hamming Loss: {metrics['hamming_loss']:.4f}")
    print(f"F1 Score (micro): {metrics['f1_micro']:.4f}")
    print(f"F1 Score (macro): {metrics['f1_macro']:.4f}")
    print(f"F1 Score (weighted): {metrics['f1_weighted']:.4f}")

    # Informe de clasificación por clase si se proporcionan los nombres
    if label_classes is not None:
        report = classification_report(all_true_flat, all_preds_flat, target_names=label_classes)
        print("\nInforme de clasificación por clase:")
        print(report)

    return metrics

# Código para entrenar y evaluar el modelo BiLSTM si los datos están disponibles
if 'train_df' in locals() and 'label_matrices' in locals():
    print("🧠 Entrenamiento de modelo BiLSTM con atención")

    try:
        # Verificar si se puede usar GPU
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Usando dispositivo: {device}")

        # Configuración
        EMBED_SIZE = 200
        HIDDEN_SIZE = 128
        BATCH_SIZE = 32
        LEARNING_RATE = 0.001
        NUM_EPOCHS = 10
        PATIENCE = 3

        # Preprocesar textos de entrenamiento
        processed_texts = train_df['text'].apply(preprocess_medical_text).tolist()

        # Dividir datos en entrenamiento, validación y prueba
        X_train, X_temp, y_train_level1, y_temp_level1 = train_test_split(
            processed_texts,
            label_matrices['level1']['matrix'],
            test_size=0.3,
            random_state=42
        )

        X_val, X_test, y_val_level1, y_test_level1 = train_test_split(
            X_temp,
            y_temp_level1,
            test_size=0.5,
            random_state=42
        )

        print(f"Datos de entrenamiento: {len(X_train)}")
        print(f"Datos de validación: {len(X_val)}")
        print(f"Datos de prueba: {len(X_test)}")

        # Crear preprocesador y construir vocabulario
        preprocessor = TextPreprocessor(max_vocab_size=30000, max_seq_length=300)
        preprocessor.build_vocab(X_train)

        # Crear datasets
        train_dataset = TextDataset(X_train, y_train_level1, preprocessor)
        val_dataset = TextDataset(X_val, y_val_level1, preprocessor)
        test_dataset = TextDataset(X_test, y_test_level1, preprocessor)

        # Crear dataloaders
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

        # Crear modelo
        model = BiLSTMAttention(
            vocab_size=preprocessor.vocab_size,
            embed_size=EMBED_SIZE,
            hidden_size=HIDDEN_SIZE,
            num_classes=y_train_level1.shape[1],
            num_layers=2,
            dropout=0.5
        )

        # Mostrar arquitectura del modelo
        print("\nArquitectura del modelo:")
        print(model)
        print(f"Número total de parámetros: {sum(p.numel() for p in model.parameters())}")

        # Definir función de pérdida y optimizador
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

        # Entrenar modelo
        print("\nComenzando entrenamiento del modelo BiLSTM...")
        history = train_bilstm_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            num_epochs=NUM_EPOCHS,
            patience=PATIENCE,
            model_save_path='best_bilstm_model.pt'
        )

        # Visualizar curvas de aprendizaje
        plt.figure(figsize=(12, 5))

        plt.subplot(1, 2, 1)
        plt.plot(history['train_loss'], label='Entrenamiento')
        plt.plot(history['val_loss'], label='Validación')
        plt.title('Pérdida durante el entrenamiento')
        plt.xlabel('Época')
        plt.ylabel('Pérdida')
        plt.legend()

        plt.subplot(1, 2, 2)
        plt.plot(history['train_f1'], label='Entrenamiento')
        plt.plot(history['val_f1'], label='Validación')
        plt.title('F1-Score durante el entrenamiento')
        plt.xlabel('Época')
        plt.ylabel('F1-Score')
        plt.legend()

        plt.tight_layout()
        plt.show()

        # Cargar mejor modelo
        print("\nCargando el mejor modelo guardado...")
        checkpoint = torch.load('best_bilstm_model.pt')
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Mejor modelo de la época {checkpoint['epoch']+1} con Val Loss: {checkpoint['val_loss']:.4f}, Val F1: {checkpoint['val_f1']:.4f}")

        # Evaluar en conjunto de prueba
        print("\nEvaluando en conjunto de prueba...")
        label_classes = label_matrices['level1']['classes'].tolist()
        test_metrics = evaluate_bilstm(model, test_loader, criterion, device, label_classes)

    except Exception as e:
        print(f"Error al entrenar el modelo BiLSTM: {e}")
else:
    print("⚠️ No se encontraron datos para entrenar el modelo BiLSTM.")

### 5.3 Clasificador Jerárquico Multi-nivel

A continuación implementaremos un modelo de clasificación jerárquica que aprovecha la estructura natural de los códigos CIE-10. Este enfoque tiene varias ventajas:

1. **Reducción de complejidad**: Al dividir el problema en múltiples niveles jerárquicos, cada clasificador se enfrenta a un subconjunto más pequeño de etiquetas
2. **Uso de información estructurada**: Explota relaciones padre-hijo entre códigos médicos
3. **Mejora la precisión**: Los errores en niveles superiores se pueden corregir en niveles inferiores

La arquitectura consta de:
1. Un clasificador para categorías principales (nivel 1)
2. Varios clasificadores específicos para subcategorías (nivel 2), uno por cada categoría principal
3. Clasificadores de tercer nivel para códigos completos

Este enfoque en cascada reduce el espacio de búsqueda en cada nivel y mejora la precisión general.

In [ ]:
# Implementación de clasificador jerárquico multi-nivel

import numpy as np
import pandas as pd
from sklearn.multioutput import MultiOutputClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.metrics import precision_score, recall_score, hamming_loss
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import pickle
from tqdm import tqdm
from collections import defaultdict

class HierarchicalClassifier:
    """Clasificador jerárquico para etiquetas CIE-10."""

    def __init__(self, parent_classifier=None, child_classifier=None, vectorizer_method='tfidf'):
        """
        Inicializa el clasificador jerárquico.

        Args:
            parent_classifier: Clasificador para el nivel padre
            child_classifier: Función para crear clasificadores de nivel hijo
            vectorizer_method: Método de vectorización ('tfidf', 'bow')
        """
        # Clasificador de nivel superior (categorías principales)
        self.parent_classifier = parent_classifier or OneVsRestClassifier(
            LogisticRegression(C=1.0, solver='liblinear')
        )

        # Función para crear clasificadores de nivel inferior
        self.child_classifier_factory = child_classifier or (
            lambda: OneVsRestClassifier(LogisticRegression(C=1.0, solver='liblinear'))
        )

        # Diccionario de clasificadores de nivel hijo (uno por cada categoría padre)
        self.child_classifiers = {}

        # Método de vectorización
        self.vectorizer_method = vectorizer_method
        self.parent_vectorizer = None
        self.child_vectorizers = {}

        # Binarizadores para convertir entre etiquetas y matrices binarias
        self.parent_mlb = None
        self.child_mlbs = {}

        # Mapeos entre códigos padre e hijo
        self.parent_to_children = {}

    def _extract_parent_child_mappings(self, y_parent, y_child):
        """
        Extrae mapeos entre etiquetas de nivel padre e hijo.

        Args:
            y_parent: Matriz binaria de etiquetas padre
            y_child: Matriz binaria de etiquetas hijo
        """
        # Para cada documento, establecer la relación entre padres e hijos
        parent_child_map = defaultdict(set)

        # Iterar sobre cada documento
        for i in range(y_parent.shape[0]):
            # Obtener índices de etiquetas padre activas
            parent_indices = np.where(y_parent[i] == 1)[0]

            # Obtener índices de etiquetas hijo activas
            child_indices = np.where(y_child[i] == 1)[0]

            # Mapear cada etiqueta padre a sus hijos
            for parent_idx in parent_indices:
                parent_label = self.parent_mlb.classes_[parent_idx]

                for child_idx in child_indices:
                    child_label = self.child_mlb.classes_[child_idx]

                    # Solo asociar hijos que pertenezcan al padre
                    # (en CIE-10 el hijo empieza con el padre)
                    if child_label.startswith(parent_label):
                        parent_child_map[parent_label].add(child_label)

        # Convertir sets a listas
        for parent, children in parent_child_map.items():
            self.parent_to_children[parent] = list(children)

        return self.parent_to_children

    def fit(self, X_texts, y_parent, y_child, verbose=True):
        """
        Entrena el clasificador jerárquico.

        Args:
            X_texts: Textos de entrenamiento
            y_parent: Matriz binaria de etiquetas padre
            y_child: Matriz binaria de etiquetas hijo
            verbose: Si True, muestra información durante el entrenamiento

        Returns:
            Self para encadenamiento de métodos
        """
        # Guardar binarizadores
        self.parent_mlb = MultiLabelBinarizer().fit(y_parent)
        self.child_mlb = MultiLabelBinarizer().fit(y_child)

        if verbose:
            print("Entrenando clasificador jerárquico...")
            print(f"- {len(self.parent_mlb.classes_)} categorías padre")
            print(f"- {len(self.child_mlb.classes_)} categorías hijo")

        # Vectorizar textos para el clasificador padre
        if verbose:
            print("- Vectorizando textos para clasificador padre...")
        X_parent, self.parent_vectorizer = create_text_representations(
            X_texts, method=self.vectorizer_method
        )

        # Entrenar clasificador padre
        if verbose:
            print("- Entrenando clasificador padre...")
        start_time = time.time()
        self.parent_classifier.fit(X_parent, y_parent)
        parent_train_time = time.time() - start_time

        if verbose:
            print(f"  Completado en {parent_train_time:.2f} segundos")

        # Extraer mapeos entre padres e hijos
        self._extract_parent_child_mappings(y_parent, y_child)

        # Entrenar clasificador hijo para cada categoría padre
        if verbose:
            print("- Entrenando clasificadores hijo...")

        child_train_times = {}

        # Iterar sobre cada categoría padre
        for i, parent_label in enumerate(self.parent_mlb.classes_):
            if verbose:
                print(f"  Entrenando clasificador para {parent_label}...")

            # Obtener índices de documentos que tengan esta etiqueta padre
            parent_docs = np.where(y_parent[:, i] == 1)[0]

            if len(parent_docs) == 0:
                if verbose:
                    print(f"  Ningún documento con etiqueta {parent_label}, saltando...")
                continue

            # Filtrar documentos y etiquetas para esta categoría padre
            X_child_texts = [X_texts[idx] for idx in parent_docs]

            # Vectorizar textos para este clasificador hijo
            X_child, child_vectorizer = create_text_representations(
                X_child_texts, method=self.vectorizer_method
            )

            # Guardar vectorizador
            self.child_vectorizers[parent_label] = child_vectorizer

            # Filtrar etiquetas hijo que pertenecen a esta categoría padre
            children_of_parent = self.parent_to_children.get(parent_label, [])

            if not children_of_parent:
                if verbose:
                    print(f"  No se encontraron hijos para {parent_label}, saltando...")
                continue

            # Crear matriz binaria para las etiquetas hijo relevantes
            child_indices = [list(self.child_mlb.classes_).index(child)
                             for child in children_of_parent
                             if child in self.child_mlb.classes_]

            if not child_indices:
                if verbose:
                    print(f"  No se encontraron índices de hijo para {parent_label}, saltando...")
                continue

            # Filtrar sólo las columnas relevantes para los hijos de este padre
            y_child_filtered = y_child[parent_docs][:, child_indices]

            # Crear y guardar binarizador específico para este conjunto de hijos
            child_mlb = MultiLabelBinarizer()
            child_mlb.classes_ = np.array(children_of_parent)
            self.child_mlbs[parent_label] = child_mlb

            # Crear y entrenar clasificador hijo
            child_classifier = self.child_classifier_factory()

            start_time = time.time()
            child_classifier.fit(X_child, y_child_filtered)
            child_train_time = time.time() - start_time

            # Guardar clasificador hijo
            self.child_classifiers[parent_label] = child_classifier
            child_train_times[parent_label] = child_train_time

            if verbose:
                print(f"  Completado en {child_train_time:.2f} segundos")

        if verbose:
            print("✓ Entrenamiento del clasificador jerárquico completado")

        return self

    def predict(self, X_texts, parent_threshold=0.5, child_threshold=0.5):
        """
        Realiza predicciones jerárquicas sobre nuevos textos.

        Args:
            X_texts: Textos a clasificar
            parent_threshold: Umbral para clasificador padre
            child_threshold: Umbral para clasificadores hijo

        Returns:
            Predicciones para niveles padre e hijo
        """
        # Vectorizar textos para el clasificador padre
        X_parent, _ = create_text_representations(
            X_texts, method=self.vectorizer_method, vectorizer=self.parent_vectorizer
        )

        # Predecir etiquetas padre
        try:
            # Intentar con predict_proba si está disponible
            parent_pred_proba = self.parent_classifier.predict_proba(X_parent)
            parent_pred = (parent_pred_proba >= parent_threshold).astype(int)
        except:
            # Usar predict normal si predict_proba no está disponible
            parent_pred = self.parent_classifier.predict(X_parent)

        # Inicializar predicciones de hijo con ceros
        n_samples = len(X_texts)
        n_child_labels = len(self.child_mlb.classes_)
        child_pred = np.zeros((n_samples, n_child_labels))

        # Para cada documento
        for i in range(n_samples):
            # Obtener etiquetas padre predichas
            parent_indices = np.where(parent_pred[i] == 1)[0]

            # Para cada etiqueta padre predicha
            for parent_idx in parent_indices:
                parent_label = self.parent_mlb.classes_[parent_idx]

                # Verificar si tenemos un clasificador hijo para esta etiqueta padre
                if parent_label in self.child_classifiers:
                    # Obtener clasificador y vectorizador hijo
                    child_classifier = self.child_classifiers[parent_label]
                    child_vectorizer = self.child_vectorizers[parent_label]

                    # Vectorizar texto para este clasificador hijo
                    X_child, _ = create_text_representations(
                        [X_texts[i]], method=self.vectorizer_method, vectorizer=child_vectorizer
                    )

                    # Predecir etiquetas hijo
                    try:
                        # Intentar con predict_proba si está disponible
                        child_pred_proba = child_classifier.predict_proba(X_child)
                        local_child_pred = (child_pred_proba >= child_threshold).astype(int)[0]
                    except:
                        # Usar predict normal si predict_proba no está disponible
                        local_child_pred = child_classifier.predict(X_child)[0]

                    # Mapear predicciones locales a índices globales
                    if parent_label in self.child_mlbs:
                        child_mlb = self.child_mlbs[parent_label]

                        # Para cada etiqueta hijo predicha
                        for j, is_active in enumerate(local_child_pred):
                            if is_active:
                                child_label = child_mlb.classes_[j]

                                # Buscar índice en binarizador global
                                try:
                                    global_idx = list(self.child_mlb.classes_).index(child_label)
                                    child_pred[i, global_idx] = 1
                                except ValueError:
                                    # Etiqueta no presente en binarizador global
                                    continue

        return parent_pred, child_pred

    def evaluate(self, X_texts, y_parent_true, y_child_true,
                parent_threshold=0.5, child_threshold=0.5):
        """
        Evalúa el rendimiento del clasificador jerárquico.

        Args:
            X_texts: Textos a evaluar
            y_parent_true: Etiquetas padre verdaderas
            y_child_true: Etiquetas hijo verdaderas
            parent_threshold: Umbral para clasificador padre
            child_threshold: Umbral para clasificadores hijo

        Returns:
            Diccionario con métricas de evaluación
        """
        # Hacer predicciones
        parent_pred, child_pred = self.predict(X_texts, parent_threshold, child_threshold)

        # Calcular métricas para nivel padre
        parent_metrics = {
            'accuracy': accuracy_score(y_parent_true, parent_pred),
            'hamming_loss': hamming_loss(y_parent_true, parent_pred),
            'precision_micro': precision_score(y_parent_true, parent_pred, average='micro'),
            'recall_micro': recall_score(y_parent_true, parent_pred, average='micro'),
            'f1_micro': f1_score(y_parent_true, parent_pred, average='micro'),
            'precision_macro': precision_score(y_parent_true, parent_pred, average='macro'),
            'recall_macro': recall_score(y_parent_true, parent_pred, average='macro'),
            'f1_macro': f1_score(y_parent_true, parent_pred, average='macro'),
            'precision_weighted': precision_score(y_parent_true, parent_pred, average='weighted'),
            'recall_weighted': recall_score(y_parent_true, parent_pred, average='weighted'),
            'f1_weighted': f1_score(y_parent_true, parent_pred, average='weighted')
        }

        # Calcular métricas para nivel hijo
        child_metrics = {
            'accuracy': accuracy_score(y_child_true, child_pred),
            'hamming_loss': hamming_loss(y_child_true, child_pred),
            'precision_micro': precision_score(y_child_true, child_pred, average='micro'),
            'recall_micro': recall_score(y_child_true, child_pred, average='micro'),
            'f1_micro': f1_score(y_child_true, child_pred, average='micro'),
            'precision_macro': precision_score(y_child_true, child_pred, average='macro', zero_division=0),
            'recall_macro': recall_score(y_child_true, child_pred, average='macro', zero_division=0),
            'f1_macro': f1_score(y_child_true, child_pred, average='macro', zero_division=0),
            'precision_weighted': precision_score(y_child_true, child_pred, average='weighted', zero_division=0),
            'recall_weighted': recall_score(y_child_true, child_pred, average='weighted', zero_division=0),
            'f1_weighted': f1_score(y_child_true, child_pred, average='weighted', zero_division=0)
        }

        # Combinar métricas
        metrics = {
            'parent': parent_metrics,
            'child': child_metrics
        }

        return metrics

    def save(self, directory="models"):
        """
        Guarda el modelo entrenado en disco.

        Args:
            directory: Directorio donde guardar los archivos
        """
        # Crear directorio si no existe
        os.makedirs(directory, exist_ok=True)

        # Guardar clasificador padre
        parent_model_path = os.path.join(directory, "hierarchical_parent_classifier.pkl")
        with open(parent_model_path, 'wb') as f:
            pickle.dump(self.parent_classifier, f)

        # Guardar vectorizador padre
        parent_vectorizer_path = os.path.join(directory, "hierarchical_parent_vectorizer.pkl")
        with open(parent_vectorizer_path, 'wb') as f:
            pickle.dump(self.parent_vectorizer, f)

        # Guardar binarizadores
        binarizers_path = os.path.join(directory, "hierarchical_binarizers.pkl")
        binarizers = {
            'parent_mlb': self.parent_mlb,
            'child_mlb': self.child_mlb,
            'child_mlbs': self.child_mlbs
        }
        with open(binarizers_path, 'wb') as f:
            pickle.dump(binarizers, f)

        # Guardar mapeos padre-hijo
        mappings_path = os.path.join(directory, "hierarchical_mappings.pkl")
        with open(mappings_path, 'wb') as f:
            pickle.dump(self.parent_to_children, f)

        # Guardar clasificadores hijo
        for parent_label, child_classifier in self.child_classifiers.items():
            safe_name = parent_label.replace(".", "_").replace(" ", "_")
            child_model_path = os.path.join(directory, f"hierarchical_child_classifier_{safe_name}.pkl")
            with open(child_model_path, 'wb') as f:
                pickle.dump(child_classifier, f)

        # Guardar vectorizadores hijo
        for parent_label, child_vectorizer in self.child_vectorizers.items():
            safe_name = parent_label.replace(".", "_").replace(" ", "_")
            child_vectorizer_path = os.path.join(directory, f"hierarchical_child_vectorizer_{safe_name}.pkl")
            with open(child_vectorizer_path, 'wb') as f:
                pickle.dump(child_vectorizer, f)

        print(f"Modelo jerárquico guardado en {directory}")

    def load(self, directory="models"):
        """
        Carga el modelo entrenado desde disco.

        Args:
            directory: Directorio donde se encuentran los archivos
        """
        # Cargar clasificador padre
        parent_model_path = os.path.join(directory, "hierarchical_parent_classifier.pkl")
        with open(parent_model_path, 'rb') as f:
            self.parent_classifier = pickle.load(f)

        # Cargar vectorizador padre
        parent_vectorizer_path = os.path.join(directory, "hierarchical_parent_vectorizer.pkl")
        with open(parent_vectorizer_path, 'rb') as f:
            self.parent_vectorizer = pickle.load(f)

        # Cargar binarizadores
        binarizers_path = os.path.join(directory, "hierarchical_binarizers.pkl")
        with open(binarizers_path, 'rb') as f:
            binarizers = pickle.load(f)
            self.parent_mlb = binarizers['parent_mlb']
            self.child_mlb = binarizers['child_mlb']
            self.child_mlbs = binarizers['child_mlbs']

        # Cargar mapeos padre-hijo
        mappings_path = os.path.join(directory, "hierarchical_mappings.pkl")
        with open(mappings_path, 'rb') as f:
            self.parent_to_children = pickle.load(f)

        # Cargar clasificadores y vectorizadores hijo
        self.child_classifiers = {}
        self.child_vectorizers = {}

        for parent_label in self.parent_to_children.keys():
            safe_name = parent_label.replace(".", "_").replace(" ", "_")

            # Cargar clasificador hijo
            child_model_path = os.path.join(directory, f"hierarchical_child_classifier_{safe_name}.pkl")
            if os.path.exists(child_model_path):
                with open(child_model_path, 'rb') as f:
                    self.child_classifiers[parent_label] = pickle.load(f)

            # Cargar vectorizador hijo
            child_vectorizer_path = os.path.join(directory, f"hierarchical_child_vectorizer_{safe_name}.pkl")
            if os.path.exists(child_vectorizer_path):
                with open(child_vectorizer_path, 'rb') as f:
                    self.child_vectorizers[parent_label] = pickle.load(f)

        print(f"Modelo jerárquico cargado desde {directory}")

# Código para entrenar y evaluar el clasificador jerárquico si los datos están disponibles
if 'train_df' in locals() and 'label_matrices' in locals():
    print("🧩 Entrenamiento de clasificador jerárquico multi-nivel")

    try:
        # Preprocesar textos de entrenamiento
        processed_texts = train_df['text'].apply(preprocess_medical_text).tolist()

        # Dividir datos en entrenamiento y prueba
        X_train, X_test, y_train_level1, y_test_level1, y_train_level2, y_test_level2 = train_test_split(
            processed_texts,
            label_matrices['level1']['matrix'],
            label_matrices['level2']['matrix'],
            test_size=0.2,
            random_state=42
        )

        print(f"Datos de entrenamiento: {len(X_train)} documentos")
        print(f"Datos de prueba: {len(X_test)} documentos")
        print(f"Dimensión etiquetas nivel 1: {y_train_level1.shape}")
        print(f"Dimensión etiquetas nivel 2: {y_train_level2.shape}")

        # Crear y entrenar clasificador jerárquico
        print("\nCreando clasificador jerárquico...")

        hierarchical_classifier = HierarchicalClassifier(
            parent_classifier=OneVsRestClassifier(LogisticRegression(C=1.0, solver='liblinear')),
            child_classifier=lambda: OneVsRestClassifier(LogisticRegression(C=1.0, solver='liblinear')),
            vectorizer_method='tfidf'
        )

        print("\nEntrenando clasificador jerárquico...")
        hierarchical_classifier.fit(X_train, y_train_level1, y_train_level2)

        # Evaluar el modelo
        print("\nEvaluando clasificador jerárquico...")
        metrics = hierarchical_classifier.evaluate(X_test, y_test_level1, y_test_level2)

        # Mostrar métricas
        print("\nMétricas para nivel 1 (categorías principales):")
        for metric, value in metrics['parent'].items():
            print(f"- {metric}: {value:.4f}")

        print("\nMétricas para nivel 2 (subcategorías):")
        for metric, value in metrics['child'].items():
            print(f"- {metric}: {value:.4f}")

        # Visualizar comparación de rendimiento entre niveles
        metrics_to_plot = ['precision_micro', 'recall_micro', 'f1_micro',
                          'precision_macro', 'recall_macro', 'f1_macro']

        parent_values = [metrics['parent'][m] for m in metrics_to_plot]
        child_values = [metrics['child'][m] for m in metrics_to_plot]

        plt.figure(figsize=(12, 6))
        x = range(len(metrics_to_plot))
        width = 0.35

        plt.bar([i - width/2 for i in x], parent_values, width, label='Nivel 1')
        plt.bar([i + width/2 for i in x], child_values, width, label='Nivel 2')

        plt.ylabel('Puntuación')
        plt.title('Comparación de métricas entre niveles jerárquicos')
        plt.xticks(x, metrics_to_plot)
        plt.legend()
        plt.tight_layout()
        plt.show()

        # Guardar modelo
        print("\nGuardando clasificador jerárquico...")
        try:
            hierarchical_classifier.save("models/hierarchical")
            print("Modelo guardado correctamente")
        except Exception as e:
            print(f"Error al guardar el modelo: {e}")

    except Exception as e:
        print(f"Error al entrenar el clasificador jerárquico: {e}")
else:
    print("⚠️ No se encontraron datos para entrenar el clasificador jerárquico.")

## 6. Evaluación y Comparación de Modelos

En esta sección realizaremos una evaluación y comparación exhaustiva de los diferentes modelos implementados. Esto nos permitirá entender las fortalezas y debilidades de cada enfoque para la clasificación multi-etiqueta jerárquica de códigos CIE-10.

Evaluaremos:

1. **Métricas de rendimiento**:
   - Precisión, Recall, F1-Score (micro, macro y weighted)
   - Hamming Loss
   - Exact Match Ratio
   - Hierarchical Precision/Recall

2. **Comparación entre modelos**:
   - Modelos tradicionales vs. redes neuronales
   - Clasificación plana vs. jerárquica
   - Análisis de errores por nivel de jerarquía

3. **Análisis específicos**:
   - Rendimiento por categoría principal
   - Rendimiento en casos con múltiples etiquetas
   - Ejemplos de predicciones correctas e incorrectas

Finalmente, visualizaremos los resultados para facilitar la interpretación y extracción de conclusiones.

In [ ]:
# Evaluación y comparación de modelos

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, hamming_loss, f1_score
from sklearn.metrics import precision_score, recall_score, classification_report
from sklearn.metrics import multilabel_confusion_matrix
from collections import defaultdict
import os
import pickle
import json
import torch
from tqdm import tqdm

def exact_match_ratio(y_true, y_pred):
    """
    Calcula la proporción de muestras donde todas las etiquetas coinciden exactamente.

    Args:
        y_true: Etiquetas verdaderas
        y_pred: Etiquetas predichas

    Returns:
        Ratio de coincidencia exacta
    """
    exact_match = np.all(y_true == y_pred, axis=1)
    return np.mean(exact_match)

def hierarchical_precision_recall(y_true_parent, y_pred_parent, y_true_child, y_pred_child):
    """
    Calcula métricas de precisión y recall jerárquicas.

    Args:
        y_true_parent: Etiquetas verdaderas de nivel padre
        y_pred_parent: Etiquetas predichas de nivel padre
        y_true_child: Etiquetas verdaderas de nivel hijo
        y_pred_child: Etiquetas predichas de nivel hijo

    Returns:
        Diccionario con métricas jerárquicas
    """
    # Inicializar contadores
    h_true_pos = 0
    h_false_pos = 0
    h_false_neg = 0

    # Para cada muestra
    for i in range(y_true_parent.shape[0]):
        # Obtener índices de etiquetas activas
        true_parent_indices = np.where(y_true_parent[i] == 1)[0]
        pred_parent_indices = np.where(y_pred_parent[i] == 1)[0]
        true_child_indices = np.where(y_true_child[i] == 1)[0]
        pred_child_indices = np.where(y_pred_child[i] == 1)[0]

        # Calcular verdaderos positivos jerárquicos
        # Un verdadero positivo es un padre correctamente predicho junto con al menos uno de sus hijos
        for parent_idx in np.intersect1d(true_parent_indices, pred_parent_indices):
            # Contar como verdadero positivo a nivel padre
            h_true_pos += 1

            # Verificar hijos
            # TODO: Esta implementación asume que ya tenemos el mapeo de padres a hijos
            # En una implementación real, necesitaríamos extraer este mapeo de los datos

            # Por ahora, simplemente contamos coincidencias en los hijos
            child_match = False
            for child_idx in true_child_indices:
                if child_idx in pred_child_indices:
                    child_match = True
                    h_true_pos += 1
                    break

            if not child_match and len(true_child_indices) > 0:
                h_false_neg += 1

        # Falsos positivos: padres predichos incorrectamente
        for parent_idx in np.setdiff1d(pred_parent_indices, true_parent_indices):
            h_false_pos += 1

        # Falsos negativos: padres no predichos
        for parent_idx in np.setdiff1d(true_parent_indices, pred_parent_indices):
            h_false_neg += 1

    # Calcular métricas
    h_precision = h_true_pos / (h_true_pos + h_false_pos) if (h_true_pos + h_false_pos) > 0 else 0
    h_recall = h_true_pos / (h_true_pos + h_false_neg) if (h_true_pos + h_false_neg) > 0 else 0
    h_f1 = 2 * h_precision * h_recall / (h_precision + h_recall) if (h_precision + h_recall) > 0 else 0

    return {
        'h_precision': h_precision,
        'h_recall': h_recall,
        'h_f1': h_f1
    }

def analyze_errors_by_category(y_true, y_pred, class_names):
    """
    Analiza errores por categoría.

    Args:
        y_true: Etiquetas verdaderas
        y_pred: Etiquetas predichas
        class_names: Nombres de las clases

    Returns:
        DataFrame con análisis de errores por categoría
    """
    # Calcular matriz de confusión multilabel
    conf_matrices = multilabel_confusion_matrix(y_true, y_pred)

    # Inicializar listas para resultados
    categories = []
    true_positives = []
    false_positives = []
    false_negatives = []
    precisions = []
    recalls = []
    f1_scores = []
    support = []

    # Analizar cada categoría
    for i, conf_matrix in enumerate(conf_matrices):
        tn, fp, fn, tp = conf_matrix.ravel()

        categories.append(class_names[i])
        true_positives.append(tp)
        false_positives.append(fp)
        false_negatives.append(fn)

        # Calcular métricas
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

        precisions.append(prec)
        recalls.append(rec)
        f1_scores.append(f1)
        support.append(tp + fn)

    # Crear DataFrame
    error_analysis = pd.DataFrame({
        'Categoría': categories,
        'Verdaderos Positivos': true_positives,
        'Falsos Positivos': false_positives,
        'Falsos Negativos': false_negatives,
        'Precisión': precisions,
        'Recall': recalls,
        'F1-Score': f1_scores,
        'Support': support
    })

    # Ordenar por soporte (frecuencia)
    return error_analysis.sort_values('Support', ascending=False)

def analyze_multilabel_performance(y_true, y_pred):
    """
    Analiza rendimiento según el número de etiquetas por documento.

    Args:
        y_true: Etiquetas verdaderas
        y_pred: Etiquetas predichas

    Returns:
        DataFrame con rendimiento por número de etiquetas
    """
    # Contar número de etiquetas por documento
    n_labels_true = np.sum(y_true, axis=1)

    # Inicializar resultados
    label_counts = []
    accuracies = []
    f1_micros = []
    f1_macros = []
    exact_matches = []
    counts = []

    # Agrupar por número de etiquetas
    max_labels = max(n_labels_true)

    for n in range(1, max_labels + 1):
        # Seleccionar documentos con n etiquetas
        mask = n_labels_true == n

        if np.sum(mask) > 0:
            y_true_filtered = y_true[mask]
            y_pred_filtered = y_pred[mask]

            # Calcular métricas
            acc = accuracy_score(y_true_filtered, y_pred_filtered)
            f1_micro = f1_score(y_true_filtered, y_pred_filtered, average='micro')
            f1_macro = f1_score(y_true_filtered, y_pred_filtered, average='macro')
            emr = exact_match_ratio(y_true_filtered, y_pred_filtered)

            # Guardar resultados
            label_counts.append(n)
            accuracies.append(acc)
            f1_micros.append(f1_micro)
            f1_macros.append(f1_macro)
            exact_matches.append(emr)
            counts.append(np.sum(mask))

    # Crear DataFrame
    performance_by_labels = pd.DataFrame({
        'Número de Etiquetas': label_counts,
        'Accuracy': accuracies,
        'F1-Score (micro)': f1_micros,
        'F1-Score (macro)': f1_macros,
        'Exact Match Ratio': exact_matches,
        'Número de Documentos': counts
    })

    return performance_by_labels

def plot_model_comparison(model_results, metric='f1_micro'):
    """
    Visualiza comparación entre modelos para una métrica específica.

    Args:
        model_results: Diccionario con resultados de modelos
        metric: Métrica a comparar
    """
    # Extraer nombres de modelos y valores de métrica
    models = list(model_results.keys())
    values = [model_results[model]['metrics'][metric] for model in models]

    # Ordenar por valor de métrica
    sorted_indices = np.argsort(values)
    models = [models[i] for i in sorted_indices]
    values = [values[i] for i in sorted_indices]

    # Crear gráfico
    plt.figure(figsize=(12, 6))
    bars = plt.barh(models, values, color=plt.cm.viridis(np.linspace(0, 1, len(models))))

    # Añadir etiquetas con valores
    for i, v in enumerate(values):
        plt.text(v + 0.01, i, f"{v:.3f}", va='center')

    # Personalizar gráfico
    plt.xlabel(f'Puntuación ({metric})')
    plt.title(f'Comparación de modelos por {metric}')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

def plot_confusion_matrix(y_true, y_pred, class_names, top_n=10):
    """
    Visualiza matriz de confusión para las top_n clases más frecuentes.

    Args:
        y_true: Etiquetas verdaderas
        y_pred: Etiquetas predichas
        class_names: Nombres de las clases
        top_n: Número de clases a visualizar
    """
    # Calcular soporte para cada clase
    support = np.sum(y_true, axis=0)

    # Obtener índices de las top_n clases más frecuentes
    top_indices = np.argsort(-support)[:top_n]

    # Filtrar etiquetas
    y_true_filtered = y_true[:, top_indices]
    y_pred_filtered = y_pred[:, top_indices]
    class_names_filtered = [class_names[i] for i in top_indices]

    # Calcular matriz de confusión para cada clase
    conf_matrices = multilabel_confusion_matrix(y_true_filtered, y_pred_filtered)

    # Visualizar matrices de confusión
    n_cols = 2
    n_rows = (top_n + 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 5))
    axes = axes.flatten()

    for i, (conf_matrix, class_name) in enumerate(zip(conf_matrices, class_names_filtered)):
        # Normalizar matriz
        conf_matrix_norm = conf_matrix.astype('float') / conf_matrix.sum(axis=1)[:, np.newaxis]
        conf_matrix_norm = np.nan_to_num(conf_matrix_norm)

        # Visualizar
        sns.heatmap(conf_matrix_norm, annot=conf_matrix, fmt='d', cmap='Blues', ax=axes[i],
                   xticklabels=['Negativo', 'Positivo'], yticklabels=['Negativo', 'Positivo'])

        axes[i].set_title(f'Confusión para {class_name}')
        axes[i].set_xlabel('Predicción')
        axes[i].set_ylabel('Verdadero')

    # Ocultar ejes vacíos
    for i in range(top_n, len(axes)):
        axes[i].set_visible(False)

    plt.tight_layout()
    plt.show()

def show_example_predictions(X_texts, y_true, y_pred, class_names, n_examples=5, show_correct=True):
    """
    Muestra ejemplos de predicciones correctas o incorrectas.

    Args:
        X_texts: Textos de los ejemplos
        y_true: Etiquetas verdaderas
        y_pred: Etiquetas predichas
        class_names: Nombres de las clases
        n_examples: Número de ejemplos a mostrar
        show_correct: Si True, muestra ejemplos correctos; si False, incorrectos
    """
    # Calcular coincidencia exacta por documento
    exact_match = np.all(y_true == y_pred, axis=1)

    # Seleccionar índices según si queremos ejemplos correctos o incorrectos
    if show_correct:
        indices = np.where(exact_match)[0]
        example_type = "correctamente"
    else:
        indices = np.where(~exact_match)[0]
        example_type = "incorrectamente"

    # Mostrar ejemplos
    if len(indices) == 0:
        print(f"No se encontraron ejemplos clasificados {example_type}.")
        return

    # Seleccionar aleatoriamente n_examples
    np.random.shuffle(indices)
    selected_indices = indices[:min(n_examples, len(indices))]

    for i, idx in enumerate(selected_indices):
        print(f"Ejemplo {i+1} clasificado {example_type}:")
        print(f"Texto: {X_texts[idx][:200]}...")

        true_labels = np.where(y_true[idx] == 1)[0]
        pred_labels = np.where(y_pred[idx] == 1)[0]

        print("Etiquetas verdaderas:")
        for label_idx in true_labels:
            print(f"- {class_names[label_idx]}")

        print("Etiquetas predichas:")
        for label_idx in pred_labels:
            print(f"- {class_names[label_idx]}")

        print("-" * 80)

def load_and_compare_models(models_to_compare=None):
    """
    Carga y compara modelos entrenados.

    Args:
        models_to_compare: Lista con los nombres de modelos a comparar

    Returns:
        Diccionario con métricas de comparación
    """
    if models_to_compare is None:
        # Por defecto comparar todos los modelos disponibles
        models_to_compare = [
            "logistic_regression",
            "linear_svc",
            "random_forest",
            "bilstm_attention",
            "hierarchical"
        ]

    # Buscar archivos de métricas guardados
    metrics_files = {}

    for model_name in models_to_compare:
        file_path = f"models/{model_name}_metrics.json"
        if os.path.exists(file_path):
            try:
                with open(file_path, 'r') as f:
                    metrics_files[model_name] = json.load(f)
            except Exception as e:
                print(f"Error al cargar métricas para {model_name}: {e}")

    # Si no hay métricas guardadas, retornar vacío
    if not metrics_files:
        print("No se encontraron métricas guardadas para los modelos.")
        return {}

    # Recopilar métricas por modelo
    comparison = {}

    for model_name, metrics in metrics_files.items():
        comparison[model_name] = {
            'metrics': metrics,
            'source': 'loaded'
        }

    return comparison

# Comparar todos los modelos entrenados
print("📊 Evaluación y comparación de modelos")

try:
    # Cargar resultados de modelos
    model_results = load_and_compare_models()

    if model_results:
        print(f"Comparando {len(model_results)} modelos...")

        # Comparar métricas principales
        metrics_to_compare = ['accuracy', 'f1_micro', 'f1_macro', 'f1_weighted', 'hamming_loss']

        for metric in metrics_to_compare:
            print(f"\nComparación por {metric}:")

            # Ordenar modelos por métrica
            sorted_models = sorted(
                model_results.items(),
                key=lambda x: x[1]['metrics'].get(metric, 0),
                reverse=True if metric != 'hamming_loss' else False
            )

            for model_name, data in sorted_models:
                value = data['metrics'].get(metric, None)
                if value is not None:
                    print(f"- {model_name}: {value:.4f}")

        # Visualizar comparaciones
        for metric in ['f1_micro', 'f1_macro', 'f1_weighted']:
            plot_model_comparison(model_results, metric)

    else:
        print("No se encontraron modelos para comparar.")

        # Si tenemos datos de prueba, podemos evaluar modelos en tiempo real
        if 'X_test' in locals() and 'y_test_level1' in locals():
            print("\nEvaluando modelos con datos de prueba...")

            # Comparar modelos tradicionales
            for model_name in ['logistic_regression', 'linear_svc', 'random_forest']:
                if f"{model_name}_model" in locals():
                    model = locals()[f"{model_name}_model"]

                    # Evaluar modelo
                    print(f"\nEvaluando modelo {model_name}...")
                    metrics = model.evaluate(X_test, y_test_level1)

                    # Mostrar métricas principales
                    print(f"F1 Score (micro): {metrics['f1_micro']:.4f}")
                    print(f"F1 Score (macro): {metrics['f1_macro']:.4f}")
                    print(f"Hamming Loss: {metrics['hamming_loss']:.4f}")

            # Evaluar BiLSTM si está disponible
            if 'model' in locals() and isinstance(model, BiLSTMAttention) and 'test_loader' in locals():
                print("\nEvaluando modelo BiLSTM...")
                device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
                criterion = torch.nn.BCEWithLogitsLoss()

                metrics = evaluate_bilstm(model, test_loader, criterion, device)

            # Evaluar modelo jerárquico si está disponible
            if 'hierarchical_classifier' in locals():
                print("\nEvaluando modelo jerárquico...")
                metrics = hierarchical_classifier.evaluate(X_test, y_test_level1, y_test_level2)

                print("\nMétricas para nivel 1 (categorías principales):")
                for metric, value in metrics['parent'].items():
                    print(f"- {metric}: {value:.4f}")

                print("\nMétricas para nivel 2 (subcategorías):")
                for metric, value in metrics['child'].items():
                    print(f"- {metric}: {value:.4f}")

except Exception as e:
    print(f"Error al comparar modelos: {e}")

## 7. Conclusiones y Trabajo Futuro

### Conclusiones principales

Tras implementar y evaluar diversos modelos para la clasificación multi-etiqueta jerárquica de textos médicos con códigos CIE-10, podemos extraer varias conclusiones importantes:

1. **Ventaja del enfoque jerárquico**: Los modelos que aprovechan la estructura jerárquica de los códigos CIE-10 generalmente superan a los enfoques planos, especialmente para categorías específicas.

2. **Relevancia del preprocesamiento médico**: Las técnicas especializadas como expansión de abreviaturas y detección de negaciones mejoran significativamente la precisión de los modelos.

3. **Equilibrio precisión-complejidad**: Los modelos tradicionales (regresión logística, SVM) ofrecen un buen balance entre rendimiento y eficiencia computacional para niveles superiores de la jerarquía.

4. **Potencial de modelos neuronales**: Las arquitecturas como BiLSTM con atención y transformers capturan mejor las relaciones semánticas complejas en textos médicos largos.

### Limitaciones

1. **Escasez de datos etiquetados**: La clasificación efectiva requiere más ejemplos etiquetados, especialmente para códigos raros.

2. **Desbalance de clases**: Las categorías médicas presentan una distribución muy desigual, afectando el rendimiento en clases minoritarias.

3. **Dependencia del contexto**: Algunos diagnósticos requieren información adicional (historial del paciente, resultados de laboratorio) que puede no estar presente en el texto.

### Trabajo futuro

1. **Modelos pre-entrenados específicos**: Utilizar o desarrollar modelos transformers pre-entrenados en textos médicos en español.

2. **Técnicas de data augmentation**: Implementar estrategias para aumentar ejemplos de clases minoritarias.

3. **Atención multimodal**: Integrar datos estructurados con texto libre para mejorar la precisión diagnóstica.

4. **Interpretabilidad y explicabilidad**: Desarrollar métodos para justificar las asignaciones de códigos con segmentos relevantes del texto.

5. **Validación clínica**: Evaluar los modelos en entornos reales con profesionales médicos para comprobar su utilidad práctica.

En resumen, la clasificación automática de diagnósticos con códigos CIE-10 muestra un gran potencial para mejorar la eficiencia del registro médico y la calidad de los datos, pero requiere equilibrar técnicas avanzadas de NLP con las restricciones prácticas de los entornos clínicos.